## MOL_LAB DIDACTIC – Moleküle erforschen

**So startest du:** In Google Colab **Laufzeit → Alle ausführen** wählen. Anschließend ein Lernmodul auswählen.

- **Isomerie:** ein kuratiertes Molekülpaar auswählen und seine Strukturen, Eigenschaften und Reaktivität gemeinsam vergleichen.
- **Andere Module:** ein Molekül auswählen, seine 2D-/3D-Struktur untersuchen und bei Bedarf Vergleiche oder Konformere berechnen.
- **Arbeitsweise:** Beobachten → Vermuten → Antwort prüfen → Erklärung anzeigen.

**Wichtig:** Berechnete Strukturdeskriptoren sind Modellwerte. Experimentelle Stoffdaten werden gesondert gekennzeichnet. Die eigene chemische Begründung bleibt entscheidend.

**V0.9d – Test:** Polarität und Löslichkeit → eines der vier Basismoleküle → Molekül analysieren → Bindungsdipole untersuchen. Nach dem letzten Schritt der Vektoraddition kann der experimentelle Bezug freiwillig aufgeklappt werden.


## 1. Installation

In Google Colab: diese Zelle einmal ausführen. Die Installation dauert normalerweise unter einer Minute.


In [ ]:
# @title 🔧 Installation ausführen { display-mode: "form" }
!pip -q install rdkit py3Dmol ipywidgets pandas


## 2. Datenbank und Funktionen laden

Diese Zelle startet die eigentliche Oberfläche. In Colab ist sie als **Form-Zelle** vorbereitet; bei Bedarf kann über **Code anzeigen/ausblenden** zwischen Bedienoberfläche und technischem Code umgeschaltet werden.


In [ ]:
# @title 🧪 MOL_LAB DIDACTIC starten { display-mode: "form" }
import json, math, textwrap, html, base64, uuid, gc, random
from io import BytesIO
import pandas as pd
from IPython.display import display, HTML, clear_output, Javascript
import ipywidgets as widgets
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors, rdMolDescriptors, Crippen, rdMolTransforms, rdMolAlign
from rdkit.Chem.Draw import IPythonConsole
import py3Dmol


DATA_URL = "https://raw.githubusercontent.com/ekerzendorfer/MOL_LAB_DIDACTIC/main/molecules_v0_4_2.json"
LOCAL_DATA_FILE = "molecules_v0_4_2.json"

# Fallback-Daten: werden verwendet, wenn die externe JSON-Datei in Colab noch nicht erreichbar ist.
FALLBACK_DATA = {'meta': {'schema_version': '0.4.2', 'title': 'MOL_LAB DIDACTIC Unterrichtsdaten v0.4.2', 'language': 'de', 'description': 'Strukturierte Moleküldaten mit didaktischen Karten, Stoffdaten, Reaktivitäts-Hinweisen und kuratierten Vergleichspaaren.', 'notes': ['Die App berechnet Summenformel, molare Masse, logP, TPSA und weitere Deskriptoren weiterhin aus dem SMILES.', 'Stoffdaten sind kuratierte Unterrichtswerte und werden nur dort hinterlegt, wo sie didaktisch nützlich sind.', 'Reaktivitäts-Hinweise sind qualitative Orientierung anhand funktioneller Gruppen und keine vollständige Reaktionsvorhersage.', 'Alternative JSON-Datenpakete können später dieselbe Struktur nutzen.', 'v0.4.0 macht Reaktivitäts-Hinweise in der Oberfläche sichtbar, wo sie didaktisch sinnvoll sind.', 'SchülerInnen-Protokolle werden aus Leitfragen, Modellwerten, Mini-Aufgaben und optionalen Stoffdaten erzeugt.', 'Stoffdaten werden vor allem in Molekül- und Isomerenvergleichen angezeigt, damit sie als Struktur-Eigenschaft-Bezug verstanden werden.', 'v0.4.2 glättet Multiple-Choice-Fragen und Reaktivitäts-Hinweise, ergänzt n-Pentan und 2-Butanol im Konformer-Modul und bereitet einklappbare Bereiche für spätere Versionen vor.'], 'future_ui': {'collapsible_sections': ['Modellgrenzen', 'Reaktivitäts-Hinweise', 'Experimenteller Bezug', 'SchülerInnen-Protokolltext'], 'goal': 'Oberfläche übersichtlich halten, besonders auf kleinen Bildschirmen.'}}, 'modules': {'Polarität und Löslichkeit': [{'name': 'Wasser', 'smiles': 'O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Wasser ein stark polares Lösungsmittel?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage erklärt die besonderen Stoffeigenschaften von Wasser am besten?', 'antworten': ['Die gewinkelte Struktur und O-H-Bindungen ermöglichen ein Netzwerk aus Wasserstoffbrücken.', 'Wasser ist nur deshalb flüssig, weil seine Moleküle besonders schwer sind.', 'Wasser ist unpolar, aber die Moleküle ziehen sich durch Ionenbindungen an.', 'Die Summenformel H2O reicht allein aus, um Siedepunkt und Löslichkeit zu erklären.'], 'richtig': 0, 'feedback': 'Richtig: Die Kombination aus polarer O-H-Bindung, gewinkelter Struktur und Wasserstoffbrücken ist entscheidend.', 'erklaerung': 'Wasser besitzt polare O-H-Bindungen und eine gewinkelte Struktur; dadurch heben sich die Dipole nicht auf.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'wasser', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Wasser ein stark polares Lösungsmittel?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage erklärt die besonderen Stoffeigenschaften von Wasser am besten?', 'mc_antworten': ['Die gewinkelte Struktur und O-H-Bindungen ermöglichen ein Netzwerk aus Wasserstoffbrücken.', 'Wasser ist nur deshalb flüssig, weil seine Moleküle besonders schwer sind.', 'Wasser ist unpolar, aber die Moleküle ziehen sich durch Ionenbindungen an.', 'Die Summenformel H2O reicht allein aus, um Siedepunkt und Löslichkeit zu erklären.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Kombination aus polarer O-H-Bindung, gewinkelter Struktur und Wasserstoffbrücken ist entscheidend.', 'erklaerung': 'Wasser besitzt polare O-H-Bindungen und eine gewinkelte Struktur; dadurch heben sich die Dipole nicht auf.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '0', 'siedepunkt_C': '100', 'dichte_g_cm3': '0.998', 'wasserloeslichkeit': 'mit Wasser vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten; Werte je nach Quelle/Temperatur leicht abweichend', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv als Lösungsmittel und Reaktionspartner', 'warum': 'Wasser kann Protonen übertragen und an Hydrolysen beteiligt sein.', 'reaktionsidee': 'Säure-Base-Reaktionen, Hydratation, Hydrolyse.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Methan', 'smiles': 'C', 'niveau': 'Basis', 'leitfrage': 'Warum ist Methan unpolar und schlecht wasserlöslich?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage beschreibt Methan im Vergleich zu Wasser am besten?', 'antworten': ['Methan ist nahezu unpolar und besitzt keine geeigneten Gruppen für Wasserstoffbrücken.', 'Methan ist wegen seiner tetraedrischen Form stark polar.', 'Methan ist wasserlöslich, weil es sehr klein ist.', 'Methan enthält zwar keine Heteroatome, kann aber starke H-Brücken bilden.'], 'richtig': 0, 'feedback': 'Richtig: Methan ist nahezu unpolar; es wechselwirkt vor allem über London-Kräfte.', 'erklaerung': 'Methan ist sehr symmetrisch und besitzt keine stark polare funktionelle Gruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'methan', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Methan unpolar und schlecht wasserlöslich?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage beschreibt Methan im Vergleich zu Wasser am besten?', 'mc_antworten': ['Methan ist nahezu unpolar und besitzt keine geeigneten Gruppen für Wasserstoffbrücken.', 'Methan ist wegen seiner tetraedrischen Form stark polar.', 'Methan ist wasserlöslich, weil es sehr klein ist.', 'Methan enthält zwar keine Heteroatome, kann aber starke H-Brücken bilden.'], 'mc_richtig': 0, 'feedback': 'Richtig: Methan ist nahezu unpolar; es wechselwirkt vor allem über London-Kräfte.', 'erklaerung': 'Methan ist sehr symmetrisch und besitzt keine stark polare funktionelle Gruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−182.5', 'siedepunkt_C': '−161.5', 'dichte_g_cm3': None, 'wasserloeslichkeit': 'kaum wasserlöslich', 'aggregatzustand_raumtemperatur': 'gasförmig', 'bedingungen': '1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'eher reaktionsträge', 'warum': 'Methan besitzt nur C-H-Bindungen und keine stark polare funktionelle Gruppe.', 'reaktionsidee': 'Verbrennung nach Aktivierung; radikalische Halogenierung unter geeigneten Bedingungen.', 'modellgrenze': 'Qualitative Orientierung; mit Sauerstoff und Aktivierungsenergie kann auch ein „träges“ Molekül heftig reagieren.'}}, {'name': 'Ammoniak', 'smiles': 'N', 'niveau': 'Basis', 'leitfrage': 'Warum ist Ammoniak polar und gut wasserlöslich?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten zu Ammoniak?', 'antworten': ['Ammoniak ist polar und kann über N-H-Bindungen Wasserstoffbrücken bilden.', 'Ammoniak ist linear und daher unpolar wie CO2.', 'Ammoniak ist nur wegen seiner geringen molaren Masse gut wasserlöslich.', 'Ammoniak enthält Stickstoff und ist deshalb automatisch ein Salz.'], 'richtig': 0, 'feedback': 'Richtig: Ammoniak ist trigonal-pyramidal, polar und kann H-Brücken bilden.', 'erklaerung': 'Ammoniak besitzt polare N-H-Bindungen und ein freies Elektronenpaar am Stickstoff.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ammoniak', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Ammoniak polar und gut wasserlöslich?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten zu Ammoniak?', 'mc_antworten': ['Ammoniak ist polar und kann über N-H-Bindungen Wasserstoffbrücken bilden.', 'Ammoniak ist linear und daher unpolar wie CO2.', 'Ammoniak ist nur wegen seiner geringen molaren Masse gut wasserlöslich.', 'Ammoniak enthält Stickstoff und ist deshalb automatisch ein Salz.'], 'mc_richtig': 0, 'feedback': 'Richtig: Ammoniak ist trigonal-pyramidal, polar und kann H-Brücken bilden.', 'erklaerung': 'Ammoniak besitzt polare N-H-Bindungen und ein freies Elektronenpaar am Stickstoff.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−77.7', 'siedepunkt_C': '−33.3', 'dichte_g_cm3': None, 'wasserloeslichkeit': 'sehr gut löslich; bildet basische Lösungen', 'aggregatzustand_raumtemperatur': 'gasförmig', 'bedingungen': '1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv als Base und Nukleophil', 'warum': 'Das freie Elektronenpaar am Stickstoff kann Protonen aufnehmen oder Bindungen bilden.', 'reaktionsidee': 'Bildung von Ammoniumsalzen; Reaktionen als Nukleophil.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Kohlenstoffdioxid', 'smiles': 'O=C=O', 'niveau': 'Basis', 'leitfrage': 'Warum ist CO₂ trotz polarer C=O-Bindungen insgesamt kaum polar?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist CO2 trotz polarer C=O-Bindungen insgesamt kaum polar?', 'antworten': ['Die lineare, symmetrische Anordnung lässt die Bindungsdipole einander aufheben.', 'C=O-Bindungen sind grundsätzlich unpolar.', 'CO2 enthält Sauerstoff und muss daher stark polar sein.', 'CO2 ist unpolar, weil es keine Doppelbindungen enthält.'], 'richtig': 0, 'feedback': 'Richtig: Die Molekülgeometrie entscheidet mit; bei CO2 heben sich die Bindungsdipole auf.', 'erklaerung': 'CO₂ besitzt polare Bindungen, aber die lineare Anordnung führt zur Aufhebung der Dipole.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'kohlenstoffdioxid', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist CO₂ trotz polarer C=O-Bindungen insgesamt kaum polar?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist CO2 trotz polarer C=O-Bindungen insgesamt kaum polar?', 'mc_antworten': ['Die lineare, symmetrische Anordnung lässt die Bindungsdipole einander aufheben.', 'C=O-Bindungen sind grundsätzlich unpolar.', 'CO2 enthält Sauerstoff und muss daher stark polar sein.', 'CO2 ist unpolar, weil es keine Doppelbindungen enthält.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Molekülgeometrie entscheidet mit; bei CO2 heben sich die Bindungsdipole auf.', 'erklaerung': 'CO₂ besitzt polare Bindungen, aber die lineare Anordnung führt zur Aufhebung der Dipole.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': 'sublimiert bei −78.5', 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': 'begrenzt löslich; reagiert teilweise zu Kohlensäure/Hydrogencarbonat', 'aggregatzustand_raumtemperatur': 'gasförmig', 'bedingungen': '1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv gegenüber Basen/Nukleophilen', 'warum': 'CO2 ist linear und insgesamt unpolar, das C-Atom kann aber von starken Nukleophilen angegriffen werden.', 'reaktionsidee': 'Bildung von Carbonaten/Hydrogencarbonaten; Reaktion mit Wasser/Basen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Methanol', 'smiles': 'CO', 'niveau': 'Basis', 'leitfrage': 'Warum ist Methanol sehr gut mit Wasser mischbar?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Methanol sehr gut mit Wasser mischbar?', 'antworten': ['Die OH-Gruppe bildet H-Brücken und der unpolare Rest ist sehr klein.', 'Methanol ist gut löslich, weil alle organischen Moleküle mit Sauerstoff vollständig wasserlöslich sind.', 'Methanol enthält eine Carbonylgruppe, die stark sauer reagiert.', 'Die C-H-Bindungen sind der Hauptgrund für die Wasserlöslichkeit.'], 'richtig': 0, 'feedback': 'Richtig: Die OH-Gruppe dominiert hier die Eigenschaften, weil der unpolare Teil klein ist.', 'erklaerung': 'Methanol besitzt eine polare OH-Gruppe und nur eine kleine Methylgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'methanol', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Methanol sehr gut mit Wasser mischbar?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Methanol sehr gut mit Wasser mischbar?', 'mc_antworten': ['Die OH-Gruppe bildet H-Brücken und der unpolare Rest ist sehr klein.', 'Methanol ist gut löslich, weil alle organischen Moleküle mit Sauerstoff vollständig wasserlöslich sind.', 'Methanol enthält eine Carbonylgruppe, die stark sauer reagiert.', 'Die C-H-Bindungen sind der Hauptgrund für die Wasserlöslichkeit.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die OH-Gruppe dominiert hier die Eigenschaften, weil der unpolare Teil klein ist.', 'erklaerung': 'Methanol besitzt eine polare OH-Gruppe und nur eine kleine Methylgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−97.6', 'siedepunkt_C': '64.7', 'dichte_g_cm3': '0.79', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die OH-Gruppe ermöglicht Oxidation, Veresterung und Säure-Base-Wechselwirkungen.', 'reaktionsidee': 'Oxidation zu Methanal; Bildung von Estern.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Ethanol', 'smiles': 'CCO', 'niveau': 'Basis', 'leitfrage': 'Warum ist Ethanol mit Wasser mischbar, obwohl es auch einen unpolaren Teil besitzt?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Ethanol gut mit Wasser mischbar, obwohl es einen Kohlenwasserstoffrest besitzt?', 'antworten': ['Die OH-Gruppe kann H-Brücken bilden; der unpolare Rest ist noch relativ klein.', 'Der Ethylrest bildet starke H-Brücken mit Wasser.', 'Ethanol ist gut wasserlöslich, weil es keine unpolaren Bereiche besitzt.', 'Ethanol ist ionisch und löst sich deshalb wie Kochsalz.'], 'richtig': 0, 'feedback': 'Richtig: Ethanol vereint eine polare OH-Gruppe mit einem noch kleinen unpolaren Rest.', 'erklaerung': 'Ethanol besitzt eine polare OH-Gruppe und einen noch kleinen unpolaren Ethylrest.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ethanol', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Ethanol mit Wasser mischbar, obwohl es auch einen unpolaren Teil besitzt?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Ethanol gut mit Wasser mischbar, obwohl es einen Kohlenwasserstoffrest besitzt?', 'mc_antworten': ['Die OH-Gruppe kann H-Brücken bilden; der unpolare Rest ist noch relativ klein.', 'Der Ethylrest bildet starke H-Brücken mit Wasser.', 'Ethanol ist gut wasserlöslich, weil es keine unpolaren Bereiche besitzt.', 'Ethanol ist ionisch und löst sich deshalb wie Kochsalz.'], 'mc_richtig': 0, 'feedback': 'Richtig: Ethanol vereint eine polare OH-Gruppe mit einem noch kleinen unpolaren Rest.', 'erklaerung': 'Ethanol besitzt eine polare OH-Gruppe und einen noch kleinen unpolaren Ethylrest.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−114.1', 'siedepunkt_C': '78.4', 'dichte_g_cm3': '0.789', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die OH-Gruppe ermöglicht Oxidation, Veresterung und Wasserstoffbrücken.', 'reaktionsidee': 'Oxidation zu Ethanal/Essigsäure; Esterbildung.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'n-Butanol', 'smiles': 'CCCCO', 'niveau': 'Basis', 'leitfrage': 'Warum ist n-Butanol schlechter wasserlöslich als Methanol oder Ethanol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist n-Butanol weniger wasserfreundlich als Methanol oder Ethanol?', 'antworten': ['Die OH-Gruppe bleibt polar, aber der größere unpolare Rest wird stärker wirksam.', 'n-Butanol besitzt keine OH-Gruppe mehr.', 'Die längere Kohlenstoffkette macht die O-H-Bindung ionisch.', 'Alle Alkohole sind unabhängig von der Kettenlänge gleich gut wasserlöslich.'], 'richtig': 0, 'feedback': 'Richtig: Mit wachsender Alkylkette nimmt der unpolare Anteil zu.', 'erklaerung': 'Die OH-Gruppe bleibt polar, aber mit wachsender Alkylkette nimmt der unpolare Anteil zu.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'n_butanol', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist n-Butanol schlechter wasserlöslich als Methanol oder Ethanol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist n-Butanol weniger wasserfreundlich als Methanol oder Ethanol?', 'mc_antworten': ['Die OH-Gruppe bleibt polar, aber der größere unpolare Rest wird stärker wirksam.', 'n-Butanol besitzt keine OH-Gruppe mehr.', 'Die längere Kohlenstoffkette macht die O-H-Bindung ionisch.', 'Alle Alkohole sind unabhängig von der Kettenlänge gleich gut wasserlöslich.'], 'mc_richtig': 0, 'feedback': 'Richtig: Mit wachsender Alkylkette nimmt der unpolare Anteil zu.', 'erklaerung': 'Die OH-Gruppe bleibt polar, aber mit wachsender Alkylkette nimmt der unpolare Anteil zu.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−89.8', 'siedepunkt_C': '117.7', 'dichte_g_cm3': '0.81', 'wasserloeslichkeit': 'begrenzt löslich', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die OH-Gruppe reagiert wie bei Alkoholen, der unpolare Rest beeinflusst vor allem Stoffeigenschaften.', 'reaktionsidee': 'Oxidation und Esterbildung.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Methanal', 'smiles': 'C=O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Methanal polar, obwohl es keine OH-Gruppe besitzt?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Was unterscheidet Methanal als Carbonylverbindung von Methanol?', 'antworten': ['Methanal kann H-Brücken aufnehmen, aber keine O-H-H-Brücken spenden.', 'Methanal besitzt wie Methanol eine Alkoholgruppe.', 'Methanal ist völlig unpolar, weil es nur ein O-Atom enthält.', 'Methanal und Methanol haben dieselbe funktionelle Gruppe.'], 'richtig': 0, 'feedback': 'Richtig: Methanal enthält eine Carbonylgruppe, keine OH-Gruppe.', 'erklaerung': 'Die C=O-Doppelbindung ist polar; der Sauerstoff kann als Akzeptor wirken.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'methanal', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Methanal polar, obwohl es keine OH-Gruppe besitzt?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Was unterscheidet Methanal als Carbonylverbindung von Methanol?', 'mc_antworten': ['Methanal kann H-Brücken aufnehmen, aber keine O-H-H-Brücken spenden.', 'Methanal besitzt wie Methanol eine Alkoholgruppe.', 'Methanal ist völlig unpolar, weil es nur ein O-Atom enthält.', 'Methanal und Methanol haben dieselbe funktionelle Gruppe.'], 'mc_richtig': 0, 'feedback': 'Richtig: Methanal enthält eine Carbonylgruppe, keine OH-Gruppe.', 'erklaerung': 'Die C=O-Doppelbindung ist polar; der Sauerstoff kann als Akzeptor wirken.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−92', 'siedepunkt_C': '−19', 'dichte_g_cm3': None, 'wasserloeslichkeit': 'gut wasserlöslich; in Wasser chemisch nicht unverändert (Hydratbildung/Polymerisation möglich)', 'aggregatzustand_raumtemperatur': 'gasförmig', 'bedingungen': '1 atm; Formaldehyd wird im Labor oft als wässrige Lösung betrachtet', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv', 'warum': 'Die Carbonylgruppe eines Aldehyds ist elektrophil; Methanal ist besonders reaktiv.', 'reaktionsidee': 'Nukleophile Addition; Oxidation zu Ameisensäure; Hydratbildung.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Aceton', 'smiles': 'CC(=O)C', 'niveau': 'Basis', 'leitfrage': 'Warum ist Aceton gut mit Wasser mischbar, obwohl es keine OH-Gruppe besitzt?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage beschreibt Aceton als Lösungsmittel am besten?', 'antworten': ['Aceton ist polar aprotisch: es kann H-Brücken aufnehmen, aber keine spenden.', 'Aceton ist ein Alkohol und kann deshalb H-Brücken spenden.', 'Aceton ist unpolar, weil es drei Kohlenstoffatome enthält.', 'Aceton besitzt eine Carboxylgruppe und reagiert sauer.'], 'richtig': 0, 'feedback': 'Richtig: Die C=O-Gruppe macht Aceton polar, aber ohne O-H/N-H kann es keine H-Brücken spenden.', 'erklaerung': 'Aceton besitzt eine polare Carbonylgruppe und zwei kleine unpolare Methylgruppen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'aceton', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Aceton gut mit Wasser mischbar, obwohl es keine OH-Gruppe besitzt?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage beschreibt Aceton als Lösungsmittel am besten?', 'mc_antworten': ['Aceton ist polar aprotisch: es kann H-Brücken aufnehmen, aber keine spenden.', 'Aceton ist ein Alkohol und kann deshalb H-Brücken spenden.', 'Aceton ist unpolar, weil es drei Kohlenstoffatome enthält.', 'Aceton besitzt eine Carboxylgruppe und reagiert sauer.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die C=O-Gruppe macht Aceton polar, aber ohne O-H/N-H kann es keine H-Brücken spenden.', 'erklaerung': 'Aceton besitzt eine polare Carbonylgruppe und zwei kleine unpolare Methylgruppen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−95', 'siedepunkt_C': '56.1', 'dichte_g_cm3': '0.79', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv als Keton', 'warum': 'Das Carbonyl-C ist elektrophil; Ketone sind aber gegenüber Oxidation stabiler als Aldehyde.', 'reaktionsidee': 'Nukleophile Addition an die C=O-Gruppe.', 'modellgrenze': 'Qualitative Orientierung anhand der Carbonylgruppe; Reaktionspartner und Bedingungen sind entscheidend.'}}, {'name': 'Diethylether', 'smiles': 'CCOCC', 'niveau': 'Basis', 'leitfrage': 'Warum unterscheidet sich Diethylether deutlich von Alkoholen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Diethylether deutlich anders als Ethanol?', 'antworten': ['Er kann H-Brücken aufnehmen, aber mangels O-H-Gruppe nicht spenden.', 'Er enthält keine polaren Bindungen.', 'Er ist wegen des Sauerstoffatoms automatisch vollständig mit Wasser mischbar.', 'Er besitzt eine Carboxylgruppe und reagiert sauer.'], 'richtig': 0, 'feedback': 'Richtig: Ether-Sauerstoff kann H-Brücken aufnehmen, aber Diethylether kann keine spenden.', 'erklaerung': 'Diethylether enthält ein O-Atom, aber keine O-H-Bindung.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'diethylether', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum unterscheidet sich Diethylether deutlich von Alkoholen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Diethylether deutlich anders als Ethanol?', 'mc_antworten': ['Er kann H-Brücken aufnehmen, aber mangels O-H-Gruppe nicht spenden.', 'Er enthält keine polaren Bindungen.', 'Er ist wegen des Sauerstoffatoms automatisch vollständig mit Wasser mischbar.', 'Er besitzt eine Carboxylgruppe und reagiert sauer.'], 'mc_richtig': 0, 'feedback': 'Richtig: Ether-Sauerstoff kann H-Brücken aufnehmen, aber Diethylether kann keine spenden.', 'erklaerung': 'Diethylether enthält ein O-Atom, aber keine O-H-Bindung.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−116', 'siedepunkt_C': '34.6', 'dichte_g_cm3': '0.71', 'wasserloeslichkeit': 'begrenzt löslich', 'aggregatzustand_raumtemperatur': 'flüssig, sehr flüchtig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'relativ reaktionsträge unter milden Bedingungen', 'warum': 'Ether besitzen keine O-H-Bindung und reagieren meist weniger leicht als Alkohole.', 'reaktionsidee': 'Spaltung unter starken sauren Bedingungen; Peroxidbildung als Sicherheitsaspekt.', 'modellgrenze': 'Qualitative Orientierung; Sicherheitsaspekte hängen stark von Lagerung und Bedingungen ab.'}}, {'name': 'Essigsäure', 'smiles': 'CC(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Essigsäure stark polar und gut wasserlöslich?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Essigsäure besonders gut für Struktur-Eigenschafts-Vergleiche geeignet?', 'antworten': ['Die Carboxylgruppe verbindet Polarität, H-Brücken und Säure-Base-Verhalten.', 'Essigsäure ist ein völlig unpolares Molekül.', 'Essigsäure enthält nur eine Ethergruppe.', 'Die Methylgruppe allein erklärt die hohe Wasserlöslichkeit.'], 'richtig': 0, 'feedback': 'Richtig: Die Carboxylgruppe ist der zentrale Strukturbaustein für die Eigenschaften.', 'erklaerung': 'Essigsäure besitzt eine Carboxylgruppe mit C=O und O-H am selben Kohlenstoffatom.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'essigsaeure', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Essigsäure stark polar und gut wasserlöslich?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Essigsäure besonders gut für Struktur-Eigenschafts-Vergleiche geeignet?', 'mc_antworten': ['Die Carboxylgruppe verbindet Polarität, H-Brücken und Säure-Base-Verhalten.', 'Essigsäure ist ein völlig unpolares Molekül.', 'Essigsäure enthält nur eine Ethergruppe.', 'Die Methylgruppe allein erklärt die hohe Wasserlöslichkeit.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Carboxylgruppe ist der zentrale Strukturbaustein für die Eigenschaften.', 'erklaerung': 'Essigsäure besitzt eine Carboxylgruppe mit C=O und O-H am selben Kohlenstoffatom.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '16.6', 'siedepunkt_C': '118.1', 'dichte_g_cm3': '1.05', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv in Säure-Base-Reaktionen', 'warum': 'Die Carboxylgruppe kann ein Proton abgeben und mit Alkoholen Ester bilden.', 'reaktionsidee': 'Neutralisation mit Basen; Esterbildung mit Alkoholen.', 'modellgrenze': 'Qualitative Orientierung; Dissoziation hängt von pH und Medium ab.'}}, {'name': 'Schwefelwasserstoff', 'smiles': 'S', 'niveau': 'Vergleich', 'leitfrage': 'Warum ist Schwefelwasserstoff bei Raumtemperatur gasförmig, Wasser aber flüssig?', 'beobachten': ['Vergleiche die Molekülform mit Wasser.', 'Achte darauf, ob starke Wasserstoffbrücken möglich sind.', 'Nutze die Modellwerte nur als grobe Orientierung.'], 'frage': 'Warum verhält sich H2S nicht einfach wie „schweres Wasser“?', 'antworten': ['S-H-Bindungen führen nicht zu einem wasserähnlich starken H-Brücken-Netzwerk.', 'H2S ist linear und besitzt keine freien Elektronenpaare.', 'H2S ist ein Ionengitter und daher fest.', 'H2S ist mit Wasser identisch, weil beide dreiatomig sind.'], 'richtig': 0, 'feedback': 'Richtig: H2S ist zwar gewinkelt, bildet aber keine ähnlich starken H-Brücken wie Wasser.', 'erklaerung': 'Wasser und Schwefelwasserstoff wirken auf den ersten Blick ähnlich, unterscheiden sich aber stark in ihren zwischenmolekularen Kräften. Wasser bildet ein starkes Wasserstoffbrücken-Netzwerk; bei H₂S sind diese Wechselwirkungen viel schwächer.', 'modellgrenze': 'Für sehr kleine anorganische Moleküle sind RDKit-Deskriptoren wie logP/TPSA nur eingeschränkt aussagekräftig. Hier ist der experimentelle Bezug besonders wichtig.', 'id': 'schwefelwasserstoff', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Schwefelwasserstoff bei Raumtemperatur gasförmig, Wasser aber flüssig?', 'beobachten': ['Vergleiche die Molekülform mit Wasser.', 'Achte darauf, ob starke Wasserstoffbrücken möglich sind.', 'Nutze die Modellwerte nur als grobe Orientierung.'], 'mc_frage': 'Warum verhält sich H2S nicht einfach wie „schweres Wasser“?', 'mc_antworten': ['S-H-Bindungen führen nicht zu einem wasserähnlich starken H-Brücken-Netzwerk.', 'H2S ist linear und besitzt keine freien Elektronenpaare.', 'H2S ist ein Ionengitter und daher fest.', 'H2S ist mit Wasser identisch, weil beide dreiatomig sind.'], 'mc_richtig': 0, 'feedback': 'Richtig: H2S ist zwar gewinkelt, bildet aber keine ähnlich starken H-Brücken wie Wasser.', 'erklaerung': 'Wasser und Schwefelwasserstoff wirken auf den ersten Blick ähnlich, unterscheiden sich aber stark in ihren zwischenmolekularen Kräften. Wasser bildet ein starkes Wasserstoffbrücken-Netzwerk; bei H₂S sind diese Wechselwirkungen viel schwächer.', 'modellgrenze': 'Für sehr kleine anorganische Moleküle sind RDKit-Deskriptoren wie logP/TPSA nur eingeschränkt aussagekräftig. Hier ist der experimentelle Bezug besonders wichtig.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−85.5', 'siedepunkt_C': '−60.3', 'dichte_g_cm3': None, 'wasserloeslichkeit': 'mäßig löslich; bildet keine wasserähnlichen H-Brücken-Netzwerke', 'aggregatzustand_raumtemperatur': 'gasförmig', 'bedingungen': '1 atm', 'quelle': 'kuratierte Unterrichtsdaten; Sicherheitsaspekt: giftiges Gas', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv in Redox- und Säure-Base-Zusammenhängen', 'warum': 'H2S ist eine schwache Säure und kann als Reduktionsmittel wirken.', 'reaktionsidee': 'Bildung von Sulfiden; Oxidation zu Schwefel/Schwefelverbindungen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Hexan', 'smiles': 'CCCCCC', 'niveau': 'Vergleich', 'leitfrage': 'Warum ist Hexan flüssig, Methan aber gasförmig, obwohl beide unpolar sind?', 'beobachten': ['Vergleiche die Molekülgröße.', 'Betrachte die unpolare Oberfläche.', 'Überlege, warum größere unpolare Moleküle stärkere London-Kräfte besitzen.'], 'frage': 'Warum ist Hexan flüssig, Methan aber gasförmig?', 'antworten': ['Hexan ist größer; seine Elektronenhülle ist leichter polarisierbar, London-Kräfte sind stärker.', 'Hexan ist polarer, weil es mehr C-H-Bindungen besitzt.', 'Methan hat stärkere H-Brücken als Hexan.', 'Hexan ist ionisch, Methan nicht.'], 'richtig': 0, 'feedback': 'Richtig: Bei unpolaren Molekülen werden London-Kräfte mit Molekülgröße/Oberfläche stärker.', 'erklaerung': 'Unpolar bedeutet nicht wechselwirkungsfrei. Hexan hat eine deutlich größere Oberfläche als Methan; dadurch wirken stärkere London-Kräfte zwischen den Molekülen.', 'modellgrenze': 'Die App zeigt London-Kräfte nicht direkt, sondern nur indirekt über Molekülgröße und Oberfläche.', 'id': 'hexan', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Hexan flüssig, Methan aber gasförmig, obwohl beide unpolar sind?', 'beobachten': ['Vergleiche die Molekülgröße.', 'Betrachte die unpolare Oberfläche.', 'Überlege, warum größere unpolare Moleküle stärkere London-Kräfte besitzen.'], 'mc_frage': 'Warum ist Hexan flüssig, Methan aber gasförmig?', 'mc_antworten': ['Hexan ist größer; seine Elektronenhülle ist leichter polarisierbar, London-Kräfte sind stärker.', 'Hexan ist polarer, weil es mehr C-H-Bindungen besitzt.', 'Methan hat stärkere H-Brücken als Hexan.', 'Hexan ist ionisch, Methan nicht.'], 'mc_richtig': 0, 'feedback': 'Richtig: Bei unpolaren Molekülen werden London-Kräfte mit Molekülgröße/Oberfläche stärker.', 'erklaerung': 'Unpolar bedeutet nicht wechselwirkungsfrei. Hexan hat eine deutlich größere Oberfläche als Methan; dadurch wirken stärkere London-Kräfte zwischen den Molekülen.', 'modellgrenze': 'Die App zeigt London-Kräfte nicht direkt, sondern nur indirekt über Molekülgröße und Oberfläche.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−95', 'siedepunkt_C': '68.7', 'dichte_g_cm3': '0.66', 'wasserloeslichkeit': 'kaum wasserlöslich', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'eher reaktionsträge', 'warum': 'Hexan enthält nur C-C- und C-H-Bindungen; es besitzt keine funktionelle Gruppe für einfache polare Reaktionen.', 'reaktionsidee': 'Verbrennung; radikalische Substitution unter speziellen Bedingungen.', 'modellgrenze': 'Qualitative Orientierung; Brand- und Lösemitteleigenschaften sind keine „Unreaktivität“ im Alltag.'}}, {'name': '1-Hexanol', 'smiles': 'CCCCCCO', 'niveau': 'Vergleich', 'leitfrage': 'Warum ist 1-Hexanol viel schlechter wasserlöslich als Ethanol?', 'beobachten': ['Suche die OH-Gruppe.', 'Vergleiche den unpolaren Kohlenwasserstoffrest mit Ethanol.', 'Achte auf logP, TPSA und die Wasserlöslichkeits-Tendenz.'], 'frage': 'Was zeigt 1-Hexanol im Vergleich zu Ethanol besonders gut?', 'antworten': ['Eine OH-Gruppe reicht nicht immer aus, um einen großen unpolaren Rest zu überwiegen.', 'Alkohole verlieren mit längerer Kette ihre OH-Gruppe.', '1-Hexanol ist wegen der OH-Gruppe vollständig mit Wasser mischbar.', 'Die Kettenlänge hat keinen Einfluss auf Stoffeigenschaften.'], 'richtig': 0, 'feedback': 'Richtig: Das Verhältnis von polarer Gruppe zu unpolarem Rest ist entscheidend.', 'erklaerung': 'Bei kurzen Alkoholen dominiert die OH-Gruppe die Wechselwirkung mit Wasser. Bei 1-Hexanol ist der unpolare Kohlenwasserstoffrest so groß, dass die Wasserlöslichkeit deutlich sinkt.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': '1_hexanol', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist 1-Hexanol viel schlechter wasserlöslich als Ethanol?', 'beobachten': ['Suche die OH-Gruppe.', 'Vergleiche den unpolaren Kohlenwasserstoffrest mit Ethanol.', 'Achte auf logP, TPSA und die Wasserlöslichkeits-Tendenz.'], 'mc_frage': 'Was zeigt 1-Hexanol im Vergleich zu Ethanol besonders gut?', 'mc_antworten': ['Eine OH-Gruppe reicht nicht immer aus, um einen großen unpolaren Rest zu überwiegen.', 'Alkohole verlieren mit längerer Kette ihre OH-Gruppe.', '1-Hexanol ist wegen der OH-Gruppe vollständig mit Wasser mischbar.', 'Die Kettenlänge hat keinen Einfluss auf Stoffeigenschaften.'], 'mc_richtig': 0, 'feedback': 'Richtig: Das Verhältnis von polarer Gruppe zu unpolarem Rest ist entscheidend.', 'erklaerung': 'Bei kurzen Alkoholen dominiert die OH-Gruppe die Wechselwirkung mit Wasser. Bei 1-Hexanol ist der unpolare Kohlenwasserstoffrest so groß, dass die Wasserlöslichkeit deutlich sinkt.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−45', 'siedepunkt_C': '157', 'dichte_g_cm3': '0.81', 'wasserloeslichkeit': 'schlecht bis begrenzt löslich', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die OH-Gruppe bleibt reaktionsfähig, auch wenn der große Alkylrest die Wasserlöslichkeit verringert.', 'reaktionsidee': 'Oxidation und Esterbildung.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Palmitinsäure', 'smiles': 'CCCCCCCCCCCCCCCC(=O)O', 'niveau': 'Vertiefung', 'leitfrage': 'Warum ist Palmitinsäure trotz Carboxylgruppe kaum wasserlöslich?', 'beobachten': ['Suche die Carboxylgruppe.', 'Vergleiche die Länge des unpolaren Restes mit Essigsäure.', 'Überlege, welcher Molekülteil die Stoffeigenschaft stärker prägt.'], 'frage': 'Warum ist Palmitinsäure trotz Carboxylgruppe kaum wasserlöslich?', 'antworten': ['Der sehr lange unpolare Kohlenwasserstoffrest dominiert die Stoffeigenschaft.', 'Palmitinsäure enthält keine polare funktionelle Gruppe.', 'Alle Carbonsäuren sind grundsätzlich unpolar.', 'Die Carboxylgruppe wird bei längeren Molekülen chemisch entfernt.'], 'richtig': 0, 'feedback': 'Richtig: Die Carboxylgruppe ist polar, aber der lange Alkylrest überwiegt stark.', 'erklaerung': 'Palmitinsäure besitzt zwar eine Carboxylgruppe, aber auch eine sehr lange unpolare Kohlenwasserstoffkette. Dadurch ist sie in Wasser kaum löslich.', 'modellgrenze': 'Das Modell berücksichtigt keine Kristallpackung, pH-Effekte oder die Bildung von Aggregaten/Mizellen.', 'id': 'palmitinsaeure', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Palmitinsäure trotz Carboxylgruppe kaum wasserlöslich?', 'beobachten': ['Suche die Carboxylgruppe.', 'Vergleiche die Länge des unpolaren Restes mit Essigsäure.', 'Überlege, welcher Molekülteil die Stoffeigenschaft stärker prägt.'], 'mc_frage': 'Warum ist Palmitinsäure trotz Carboxylgruppe kaum wasserlöslich?', 'mc_antworten': ['Der sehr lange unpolare Kohlenwasserstoffrest dominiert die Stoffeigenschaft.', 'Palmitinsäure enthält keine polare funktionelle Gruppe.', 'Alle Carbonsäuren sind grundsätzlich unpolar.', 'Die Carboxylgruppe wird bei längeren Molekülen chemisch entfernt.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Carboxylgruppe ist polar, aber der lange Alkylrest überwiegt stark.', 'erklaerung': 'Palmitinsäure besitzt zwar eine Carboxylgruppe, aber auch eine sehr lange unpolare Kohlenwasserstoffkette. Dadurch ist sie in Wasser kaum löslich.', 'modellgrenze': 'Das Modell berücksichtigt keine Kristallpackung, pH-Effekte oder die Bildung von Aggregaten/Mizellen.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '63', 'siedepunkt_C': '>300 (Zersetzung/hoher Siedebereich)', 'dichte_g_cm3': None, 'wasserloeslichkeit': 'praktisch unlöslich', 'aggregatzustand_raumtemperatur': 'fest', 'bedingungen': 'ca. 20–25 °C', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv an der Carboxylgruppe, aber stofflich stark lipophil', 'warum': 'Die Carboxylgruppe bleibt chemisch reaktionsfähig; der lange unpolare Rest prägt Löslichkeit und Schmelzverhalten.', 'reaktionsidee': 'Salzbildung; Esterbildung.', 'modellgrenze': 'Qualitative Orientierung; Aggregation, Kristallstruktur und pH-Effekte werden nicht vollständig abgebildet.'}}, {'name': 'Diethylenglycol', 'smiles': 'OCCOCCO', 'niveau': 'Vergleich', 'leitfrage': 'Warum ist Diethylenglycol deutlich wasserfreundlicher als Diethylether?', 'beobachten': ['Zähle die Sauerstoffatome.', 'Suche OH-Gruppen.', 'Vergleiche H-Brücken-Donoren und -Akzeptoren.'], 'frage': 'Warum ist Diethylenglycol viel wasserfreundlicher als Diethylether?', 'antworten': ['Es besitzt mehrere O-Atome und zwei OH-Gruppen, kann also H-Brücken spenden und aufnehmen.', 'Es ist wasserfreundlicher, weil es keine Heteroatome besitzt.', 'Diethylether kann mehr H-Brücken spenden als Diethylenglycol.', 'Die zusätzliche OH-Gruppe macht das Molekül vollständig unpolar.'], 'richtig': 0, 'feedback': 'Richtig: Mehrere polare Zentren und OH-Gruppen erhöhen die Wasserwechselwirkung stark.', 'erklaerung': 'Diethylether kann H-Brücken nur aufnehmen. Diethylenglycol kann durch seine OH-Gruppen auch H-Brücken spenden und besitzt eine größere polare Oberfläche.', 'modellgrenze': 'Die Toxizität von Diethylenglycol ist hier nicht Thema; betrachtet wird nur der Struktur–Eigenschaft-Zusammenhang.', 'id': 'diethylenglycol', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Diethylenglycol deutlich wasserfreundlicher als Diethylether?', 'beobachten': ['Zähle die Sauerstoffatome.', 'Suche OH-Gruppen.', 'Vergleiche H-Brücken-Donoren und -Akzeptoren.'], 'mc_frage': 'Warum ist Diethylenglycol viel wasserfreundlicher als Diethylether?', 'mc_antworten': ['Es besitzt mehrere O-Atome und zwei OH-Gruppen, kann also H-Brücken spenden und aufnehmen.', 'Es ist wasserfreundlicher, weil es keine Heteroatome besitzt.', 'Diethylether kann mehr H-Brücken spenden als Diethylenglycol.', 'Die zusätzliche OH-Gruppe macht das Molekül vollständig unpolar.'], 'mc_richtig': 0, 'feedback': 'Richtig: Mehrere polare Zentren und OH-Gruppen erhöhen die Wasserwechselwirkung stark.', 'erklaerung': 'Diethylether kann H-Brücken nur aufnehmen. Diethylenglycol kann durch seine OH-Gruppen auch H-Brücken spenden und besitzt eine größere polare Oberfläche.', 'modellgrenze': 'Die Toxizität von Diethylenglycol ist hier nicht Thema; betrachtet wird nur der Struktur–Eigenschaft-Zusammenhang.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−10', 'siedepunkt_C': '245', 'dichte_g_cm3': '1.12', 'wasserloeslichkeit': 'sehr gut mischbar/wasserfreundlich', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten; toxikologisch relevant', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die OH-Gruppen ermöglichen typische Alkoholreaktionen.', 'reaktionsidee': 'Esterbildung; Oxidation unter geeigneten Bedingungen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}], 'Isomerie': [{'name': 'Ethanol', 'smiles': 'CCO', 'niveau': 'Basis', 'leitfrage': 'Warum hat Ethanol andere Eigenschaften als Dimethylether, obwohl beide C₂H₆O besitzen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum hat Ethanol andere Stoffeigenschaften als Dimethylether?', 'antworten': ['Ethanol besitzt eine O-H-Gruppe und kann H-Brücken spenden; Dimethylether nicht.', 'Beide müssen gleiche Eigenschaften haben, weil ihre Summenformel gleich ist.', 'Dimethylether ist polarer, weil der Sauerstoff zwischen zwei C-Atomen sitzt.', 'Ethanol und Dimethylether unterscheiden sich nur in der Molmasse.'], 'richtig': 0, 'feedback': 'Richtig: Die Verknüpfung der Atome erzeugt unterschiedliche funktionelle Gruppen.', 'erklaerung': 'Gleiche Summenformel, aber andere Verknüpfung der Atome führt zu anderen funktionellen Gruppen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ethanol', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum hat Ethanol andere Eigenschaften als Dimethylether, obwohl beide C₂H₆O besitzen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum hat Ethanol andere Stoffeigenschaften als Dimethylether?', 'mc_antworten': ['Ethanol besitzt eine O-H-Gruppe und kann H-Brücken spenden; Dimethylether nicht.', 'Beide müssen gleiche Eigenschaften haben, weil ihre Summenformel gleich ist.', 'Dimethylether ist polarer, weil der Sauerstoff zwischen zwei C-Atomen sitzt.', 'Ethanol und Dimethylether unterscheiden sich nur in der Molmasse.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Verknüpfung der Atome erzeugt unterschiedliche funktionelle Gruppen.', 'erklaerung': 'Gleiche Summenformel, aber andere Verknüpfung der Atome führt zu anderen funktionellen Gruppen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−114.1', 'siedepunkt_C': '78.4', 'dichte_g_cm3': '0.789', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die OH-Gruppe ermöglicht Oxidation, Veresterung und Wasserstoffbrücken.', 'reaktionsidee': 'Oxidation zu Ethanal/Essigsäure; Esterbildung.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Dimethylether', 'smiles': 'COC', 'niveau': 'Basis', 'leitfrage': 'Warum siedet Dimethylether viel niedriger als Ethanol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum siedet Dimethylether viel niedriger als Ethanol?', 'antworten': ['Dimethylether kann keine H-Brücken zwischen eigenen Molekülen spenden.', 'Dimethylether besitzt keine polaren Bindungen.', 'Dimethylether hat eine viel kleinere molare Masse als Ethanol.', 'Dimethylether ist ein Ionengitter.'], 'richtig': 0, 'feedback': 'Richtig: Der fehlende H-Brücken-Donor ist ein zentraler Unterschied.', 'erklaerung': 'Er besitzt eine Ethergruppe statt einer Alkoholgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'dimethylether', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum siedet Dimethylether viel niedriger als Ethanol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum siedet Dimethylether viel niedriger als Ethanol?', 'mc_antworten': ['Dimethylether kann keine H-Brücken zwischen eigenen Molekülen spenden.', 'Dimethylether besitzt keine polaren Bindungen.', 'Dimethylether hat eine viel kleinere molare Masse als Ethanol.', 'Dimethylether ist ein Ionengitter.'], 'mc_richtig': 0, 'feedback': 'Richtig: Der fehlende H-Brücken-Donor ist ein zentraler Unterschied.', 'erklaerung': 'Er besitzt eine Ethergruppe statt einer Alkoholgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {"schmelzpunkt_C":"−141,5","siedepunkt_C":"−24,8","dichte_g_cm3":"0,665 (verflüssigt bei 25 °C; unter Druck)","wasserloeslichkeit":"begrenzt (ca. 2,4 g/100 mL; temperaturabhängig)","aggregatzustand_raumtemperatur":"gasförmig","bedingungen":"ca. 20–25 °C, 1 atm; Dichte als Flüssigkeit unter Druck","quelle":"PubChem CID 8254; NIST Chemistry WebBook; FSANZ A1062 (Dichte der Flüssigkeit)","hinweis":"Dichtewert gilt für verflüssigten Dimethylether, nicht für das Gas bei Normaldruck."}, 'reaktivitaet': {"hinweis":"Leicht entzündliches Gas; typische Ethergruppe","warum":"Die C–O–C-Gruppe besitzt kein O–H. Anders als Ethanol kann Dimethylether zwischen eigenen Molekülen keine Wasserstoffbrücken spenden; bei Oxidation und Spaltung unterscheidet sich sein Verhalten von einem Alkohol.","reaktionsidee":"Vollständige Verbrennung: CH₃–O–CH₃ + 3 O₂ → 2 CO₂ + 3 H₂O.","modellgrenze":"Qualitativer Hinweis anhand funktioneller Gruppen, keine vollständige Reaktions- oder Sicherheitsvorhersage."}}, {'name': 'Propanal', 'smiles': 'CCC=O', 'niveau': 'Basis', 'leitfrage': 'Wie unterscheidet sich Propanal von Aceton?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Was unterscheidet Propanal von Aceton?', 'antworten': ['Propanal ist ein Aldehyd; das Carbonyl-C trägt ein H-Atom.', 'Propanal und Aceton besitzen unterschiedliche Summenformeln.', 'Propanal ist ein Alkohol, Aceton ein Ether.', 'Bei Carbonylverbindungen spielt die Position der C=O-Gruppe keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Aldehyd und Keton unterscheiden sich in der Umgebung der Carbonylgruppe.', 'erklaerung': 'Die Carbonylgruppe liegt am Ende der Kette.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'propanal', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Wie unterscheidet sich Propanal von Aceton?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Was unterscheidet Propanal von Aceton?', 'mc_antworten': ['Propanal ist ein Aldehyd; das Carbonyl-C trägt ein H-Atom.', 'Propanal und Aceton besitzen unterschiedliche Summenformeln.', 'Propanal ist ein Alkohol, Aceton ein Ether.', 'Bei Carbonylverbindungen spielt die Position der C=O-Gruppe keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Aldehyd und Keton unterscheiden sich in der Umgebung der Carbonylgruppe.', 'erklaerung': 'Die Carbonylgruppe liegt am Ende der Kette.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {"schmelzpunkt_C": "−81", "siedepunkt_C": "48–49", "dichte_g_cm3": "ca. 0,807 (flüssig, 20 °C)", "wasserloeslichkeit": "gut, aber nicht vollständig mit Wasser mischbar (temperatur- und quellenabhängig)", "aggregatzustand_raumtemperatur": "flüssig, sehr flüchtig", "bedingungen": "ca. 20 °C, 1 atm", "quelle": "NIST Chemistry WebBook; PubChem CID 527; EPA Propionaldehyde AEGL", "hinweis": "Im Vergleich zu Aceton: beide C₃H₆O, aber Aldehyd statt Keton; Messwerte können je nach Quelle leicht abweichen."}, 'reaktivitaet': {"hinweis": "reaktiv als Aldehyd", "warum": "Das elektrophile Carbonyl-C ermöglicht Additionsreaktionen. Aldehyde lassen sich unter passenden Bedingungen leichter zu Carbonsäuren oxidieren als Ketone.", "reaktionsidee": "Oxidation zu Propansäure; Reduktion zu 1-Propanol; nukleophile Addition an C=O.", "modellgrenze": "Qualitative Orientierung; tatsächlicher Reaktionsverlauf hängt von Reagenzien und Bedingungen ab."}}, {'name': 'Aceton', 'smiles': 'CC(=O)C', 'niveau': 'Basis', 'leitfrage': 'Wie unterscheidet sich Aceton von Propanal?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage beschreibt Aceton als Lösungsmittel am besten?', 'antworten': ['Aceton ist polar aprotisch: es kann H-Brücken aufnehmen, aber keine spenden.', 'Aceton ist ein Alkohol und kann deshalb H-Brücken spenden.', 'Aceton ist unpolar, weil es drei Kohlenstoffatome enthält.', 'Aceton besitzt eine Carboxylgruppe und reagiert sauer.'], 'richtig': 0, 'feedback': 'Richtig: Die C=O-Gruppe macht Aceton polar, aber ohne O-H/N-H kann es keine H-Brücken spenden.', 'erklaerung': 'Die Carbonylgruppe liegt zwischen zwei Kohlenstoffatomen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'aceton', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Wie unterscheidet sich Aceton von Propanal?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage beschreibt Aceton als Lösungsmittel am besten?', 'mc_antworten': ['Aceton ist polar aprotisch: es kann H-Brücken aufnehmen, aber keine spenden.', 'Aceton ist ein Alkohol und kann deshalb H-Brücken spenden.', 'Aceton ist unpolar, weil es drei Kohlenstoffatome enthält.', 'Aceton besitzt eine Carboxylgruppe und reagiert sauer.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die C=O-Gruppe macht Aceton polar, aber ohne O-H/N-H kann es keine H-Brücken spenden.', 'erklaerung': 'Die Carbonylgruppe liegt zwischen zwei Kohlenstoffatomen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−95', 'siedepunkt_C': '56.1', 'dichte_g_cm3': '0.79', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv als Keton', 'warum': 'Das Carbonyl-C ist elektrophil; Ketone sind aber gegenüber Oxidation stabiler als Aldehyde.', 'reaktionsidee': 'Nukleophile Addition an die C=O-Gruppe.', 'modellgrenze': 'Qualitative Orientierung anhand der Carbonylgruppe; Reaktionspartner und Bedingungen sind entscheidend.'}}, {'name': 'Butan', 'smiles': 'CCCC', 'niveau': 'Basis', 'leitfrage': 'Was unterscheidet Butan von Isobutan?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Was unterscheidet Butan von Isobutan?', 'antworten': ['Die C-Atome sind anders verknüpft: Butan ist unverzweigt, Isobutan verzweigt.', 'Butan und Isobutan unterscheiden sich in der Summenformel.', 'Isobutan enthält eine OH-Gruppe.', 'Verzweigung verändert keine physikalischen Eigenschaften.'], 'richtig': 0, 'feedback': 'Richtig: Es handelt sich um Konstitutionsisomerie mit anderer C-Gerüstform.', 'erklaerung': 'Beide haben dieselbe Summenformel, aber unterschiedliche Kohlenstoffverknüpfung.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'butan', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Was unterscheidet Butan von Isobutan?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Was unterscheidet Butan von Isobutan?', 'mc_antworten': ['Die C-Atome sind anders verknüpft: Butan ist unverzweigt, Isobutan verzweigt.', 'Butan und Isobutan unterscheiden sich in der Summenformel.', 'Isobutan enthält eine OH-Gruppe.', 'Verzweigung verändert keine physikalischen Eigenschaften.'], 'mc_richtig': 0, 'feedback': 'Richtig: Es handelt sich um Konstitutionsisomerie mit anderer C-Gerüstform.', 'erklaerung': 'Beide haben dieselbe Summenformel, aber unterschiedliche Kohlenstoffverknüpfung.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Isobutan', 'smiles': 'CC(C)C', 'niveau': 'Basis', 'leitfrage': 'Warum ist Isobutan kompakter als Butan?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum kann Verzweigung Stoffeigenschaften verändern?', 'antworten': ['Die Molekülform und Kontaktfläche zwischen Molekülen ändern sich.', 'Verzweigung erzeugt automatisch Ionen.', 'Verzweigung ändert immer die Summenformel.', 'Verzweigte Moleküle können keine London-Kräfte bilden.'], 'richtig': 0, 'feedback': 'Richtig: Verzweigung beeinflusst Form, Oberfläche und zwischenmolekulare Wechselwirkungen.', 'erklaerung': 'Verzweigung verändert Molekülform und damit Stoffeigenschaften.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'isobutan', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Isobutan kompakter als Butan?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum kann Verzweigung Stoffeigenschaften verändern?', 'mc_antworten': ['Die Molekülform und Kontaktfläche zwischen Molekülen ändern sich.', 'Verzweigung erzeugt automatisch Ionen.', 'Verzweigung ändert immer die Summenformel.', 'Verzweigte Moleküle können keine London-Kräfte bilden.'], 'mc_richtig': 0, 'feedback': 'Richtig: Verzweigung beeinflusst Form, Oberfläche und zwischenmolekulare Wechselwirkungen.', 'erklaerung': 'Verzweigung verändert Molekülform und damit Stoffeigenschaften.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Propansäure', 'smiles': 'CCC(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum unterscheidet sich Propansäure stark von Essigsäuremethylester?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Propansäure nicht mit Essigsäuremethylester gleichzusetzen?', 'antworten': ['Propansäure besitzt eine Carboxylgruppe mit saurem O-H; der Ester nicht.', 'Beide haben dieselbe funktionelle Gruppe.', 'Der Ester ist eine Carbonsäure, weil er zwei Sauerstoffatome enthält.', 'Nur die Summenformel entscheidet über Säureeigenschaften.'], 'richtig': 0, 'feedback': 'Richtig: Carbonsäure und Ester haben unterschiedliche funktionelle Gruppen.', 'erklaerung': 'Funktionelle Isomerie: Carbonsäure vs. Ester.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'propansaeure', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum unterscheidet sich Propansäure stark von Essigsäuremethylester?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Propansäure nicht mit Essigsäuremethylester gleichzusetzen?', 'mc_antworten': ['Propansäure besitzt eine Carboxylgruppe mit saurem O-H; der Ester nicht.', 'Beide haben dieselbe funktionelle Gruppe.', 'Der Ester ist eine Carbonsäure, weil er zwei Sauerstoffatome enthält.', 'Nur die Summenformel entscheidet über Säureeigenschaften.'], 'mc_richtig': 0, 'feedback': 'Richtig: Carbonsäure und Ester haben unterschiedliche funktionelle Gruppen.', 'erklaerung': 'Funktionelle Isomerie: Carbonsäure vs. Ester.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Essigsäuremethylester', 'smiles': 'CC(=O)OC', 'niveau': 'Basis', 'leitfrage': 'Warum ist Essigsäuremethylester kein Säure-Isomer im Eigenschaftssinn?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum zeigt ein Ester andere Eigenschaften als eine Carbonsäure?', 'antworten': ['Beim Ester fehlt die saure O-H-Gruppe der Carboxylgruppe.', 'Ester enthalten keine Sauerstoffatome.', 'Ester sind immer ionisch.', 'Carbonsäure und Ester müssen gleich reagieren, wenn die Summenformel passt.'], 'richtig': 0, 'feedback': 'Richtig: Der Austausch der O-H-Gruppe verändert Polarität und Reaktivität stark.', 'erklaerung': 'Gleiche Summenformel kann durch andere funktionelle Gruppe ganz andere Eigenschaften ergeben.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'essigsaeuremethylester', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Essigsäuremethylester kein Säure-Isomer im Eigenschaftssinn?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum zeigt ein Ester andere Eigenschaften als eine Carbonsäure?', 'mc_antworten': ['Beim Ester fehlt die saure O-H-Gruppe der Carboxylgruppe.', 'Ester enthalten keine Sauerstoffatome.', 'Ester sind immer ionisch.', 'Carbonsäure und Ester müssen gleich reagieren, wenn die Summenformel passt.'], 'mc_richtig': 0, 'feedback': 'Richtig: Der Austausch der O-H-Gruppe verändert Polarität und Reaktivität stark.', 'erklaerung': 'Gleiche Summenformel kann durch andere funktionelle Gruppe ganz andere Eigenschaften ergeben.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'D-Alanin', 'smiles': 'C[C@@H](N)C(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum sind D- und L-Alanin trotz gleicher Verknüpfung nicht dasselbe Molekül?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Was zeigt der Vergleich von D- und L-Alanin?', 'antworten': ['Gleiche Verknüpfung kann durch räumliche Spiegelbildlichkeit zu unterschiedlichen biologischen Wirkungen führen.', 'D- und L-Alanin unterscheiden sich in der Summenformel.', 'D-Alanin enthält eine andere funktionelle Gruppe als L-Alanin.', 'Spiegelbildlichkeit spielt bei Biomolekülen keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Enantiomere sind spiegelbildlich und in chiralen Umgebungen oft verschieden wirksam.', 'erklaerung': 'Chiralität zeigt, dass auch die räumliche Anordnung entscheidend ist.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'd_alanin', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum sind D- und L-Alanin trotz gleicher Verknüpfung nicht dasselbe Molekül?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Was zeigt der Vergleich von D- und L-Alanin?', 'mc_antworten': ['Gleiche Verknüpfung kann durch räumliche Spiegelbildlichkeit zu unterschiedlichen biologischen Wirkungen führen.', 'D- und L-Alanin unterscheiden sich in der Summenformel.', 'D-Alanin enthält eine andere funktionelle Gruppe als L-Alanin.', 'Spiegelbildlichkeit spielt bei Biomolekülen keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Enantiomere sind spiegelbildlich und in chiralen Umgebungen oft verschieden wirksam.', 'erklaerung': 'Chiralität zeigt, dass auch die räumliche Anordnung entscheidend ist.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'L-Alanin', 'smiles': 'C[C@H](N)C(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Chiralität bei Aminosäuren biologisch wichtig?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['L-Alanin ist eine spiegelbildliche Form von D-Alanin.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: L-Alanin ist eine spiegelbildliche Form von D-Alanin.', 'erklaerung': 'Biologische Systeme unterscheiden Spiegelbilder häufig sehr genau.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'l_alanin', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Chiralität bei Aminosäuren biologisch wichtig?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['L-Alanin ist eine spiegelbildliche Form von D-Alanin.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: L-Alanin ist eine spiegelbildliche Form von D-Alanin.', 'erklaerung': 'Biologische Systeme unterscheiden Spiegelbilder häufig sehr genau.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Glucose offen', 'smiles': 'O=CC(O)C(O)C(O)C(O)CO', 'niveau': 'Basis', 'leitfrage': 'Warum ist Glucose ein komplexeres Isomeriebeispiel?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum sind Glucose und Fructose trotz gleicher Summenformel verschieden?', 'antworten': ['Glucose ist in der offenkettigen Form eine Aldose, Fructose eine Ketose.', 'Beide besitzen dieselbe funktionelle Gruppe an derselben Stelle.', 'Die Lage der C=O-Gruppe beeinflusst Eigenschaften nicht.', 'Zucker unterscheiden sich nur durch die Anzahl der C-Atome.'], 'richtig': 0, 'feedback': 'Richtig: Die Position und Art der Carbonylgruppe ist entscheidend.', 'erklaerung': 'Glucose und Fructose haben dieselbe Summenformel, unterscheiden sich aber in der Carbonylgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'glucose_offen', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Glucose ein komplexeres Isomeriebeispiel?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum sind Glucose und Fructose trotz gleicher Summenformel verschieden?', 'mc_antworten': ['Glucose ist in der offenkettigen Form eine Aldose, Fructose eine Ketose.', 'Beide besitzen dieselbe funktionelle Gruppe an derselben Stelle.', 'Die Lage der C=O-Gruppe beeinflusst Eigenschaften nicht.', 'Zucker unterscheiden sich nur durch die Anzahl der C-Atome.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Position und Art der Carbonylgruppe ist entscheidend.', 'erklaerung': 'Glucose und Fructose haben dieselbe Summenformel, unterscheiden sich aber in der Carbonylgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Fructose offen', 'smiles': 'OCC(O)C(O)C(O)C(=O)CO', 'niveau': 'Basis', 'leitfrage': 'Warum ist Fructose trotz gleicher Summenformel anders als Glucose?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Fructose ist eine Ketose.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Fructose ist eine Ketose.', 'erklaerung': 'In Lösung sind zusätzlich Ringformen wichtig; hier wird bewusst vereinfacht.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'fructose_offen', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Fructose trotz gleicher Summenformel anders als Glucose?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Fructose ist eine Ketose.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Fructose ist eine Ketose.', 'erklaerung': 'In Lösung sind zusätzlich Ringformen wichtig; hier wird bewusst vereinfacht.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Maleinsäure', 'smiles': 'O=C(O)/C=C\\C(=O)O', 'niveau': 'Vertiefung', 'leitfrage': 'Warum kann die cis/trans-Anordnung die Stoffeigenschaften stark verändern?', 'beobachten': ['Vergleiche die Lage der beiden Carboxylgruppen.', 'Achte auf die räumliche Nähe der Gruppen.', 'Vergleiche mit Fumarsäure im Isomerenvergleich.'], 'frage': 'Warum kann cis/trans-Isomerie die Eigenschaften stark verändern?', 'antworten': ['Die räumliche Lage der Carboxylgruppen verändert Wechselwirkungen und Stoffeigenschaften.', 'cis/trans-Isomere haben immer verschiedene Summenformeln.', 'Doppelbindungen sind frei drehbar wie Einfachbindungen.', 'Maleinsäure und Fumarsäure sind nur verschiedene Namen für dasselbe Molekül.'], 'richtig': 0, 'feedback': 'Richtig: Die starre C=C-Doppelbindung fixiert unterschiedliche räumliche Anordnungen.', 'erklaerung': 'Maleinsäure ist das cis-Isomer; die Carboxylgruppen liegen räumlich nahe beieinander. Dadurch unterscheiden sich Eigenschaften wie Löslichkeit, Schmelzpunkt und mögliche intramolekulare Wechselwirkungen von Fumarsäure.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'maleinsaeure', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum kann die cis/trans-Anordnung die Stoffeigenschaften stark verändern?', 'beobachten': ['Vergleiche die Lage der beiden Carboxylgruppen.', 'Achte auf die räumliche Nähe der Gruppen.', 'Vergleiche mit Fumarsäure im Isomerenvergleich.'], 'mc_frage': 'Warum kann cis/trans-Isomerie die Eigenschaften stark verändern?', 'mc_antworten': ['Die räumliche Lage der Carboxylgruppen verändert Wechselwirkungen und Stoffeigenschaften.', 'cis/trans-Isomere haben immer verschiedene Summenformeln.', 'Doppelbindungen sind frei drehbar wie Einfachbindungen.', 'Maleinsäure und Fumarsäure sind nur verschiedene Namen für dasselbe Molekül.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die starre C=C-Doppelbindung fixiert unterschiedliche räumliche Anordnungen.', 'erklaerung': 'Maleinsäure ist das cis-Isomer; die Carboxylgruppen liegen räumlich nahe beieinander. Dadurch unterscheiden sich Eigenschaften wie Löslichkeit, Schmelzpunkt und mögliche intramolekulare Wechselwirkungen von Fumarsäure.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': 'ca. 130–135', 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': 'gut löslich', 'aggregatzustand_raumtemperatur': 'fest', 'bedingungen': 'ca. 20–25 °C', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv', 'warum': 'Zwei Carboxylgruppen und die C=C-Doppelbindung bieten mehrere Reaktionsmöglichkeiten.', 'reaktionsidee': 'Säure-Base-Reaktionen; Additionen an C=C unter geeigneten Bedingungen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Fumarsäure', 'smiles': 'O=C(O)/C=C/C(=O)O', 'niveau': 'Vertiefung', 'leitfrage': 'Warum ist Fumarsäure trotz gleicher Summenformel anders als Maleinsäure?', 'beobachten': ['Vergleiche die Lage der Carboxylgruppen.', 'Achte auf die trans-Anordnung.', 'Überlege, warum das die Wechselwirkungen verändert.'], 'frage': 'Warum ist Fumarsäure nicht einfach identisch mit Maleinsäure?', 'antworten': ['Fumarsäure ist das trans-Isomer; die Carboxylgruppen liegen auf gegenüberliegenden Seiten.', 'Fumarsäure hat keine Doppelbindung.', 'Fumarsäure enthält keine Carboxylgruppen.', 'trans-Isomere sind immer unpolar.'], 'richtig': 0, 'feedback': 'Richtig: Die trans-Anordnung verändert Wechselwirkungen und Stoffeigenschaften.', 'erklaerung': 'Fumarsäure besitzt dieselbe Summenformel wie Maleinsäure, aber eine andere räumliche Anordnung. Diese scheinbar kleine Änderung kann deutliche Stoffeigenschaftsunterschiede bewirken.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'fumarsaeure', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Fumarsäure trotz gleicher Summenformel anders als Maleinsäure?', 'beobachten': ['Vergleiche die Lage der Carboxylgruppen.', 'Achte auf die trans-Anordnung.', 'Überlege, warum das die Wechselwirkungen verändert.'], 'mc_frage': 'Warum ist Fumarsäure nicht einfach identisch mit Maleinsäure?', 'mc_antworten': ['Fumarsäure ist das trans-Isomer; die Carboxylgruppen liegen auf gegenüberliegenden Seiten.', 'Fumarsäure hat keine Doppelbindung.', 'Fumarsäure enthält keine Carboxylgruppen.', 'trans-Isomere sind immer unpolar.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die trans-Anordnung verändert Wechselwirkungen und Stoffeigenschaften.', 'erklaerung': 'Fumarsäure besitzt dieselbe Summenformel wie Maleinsäure, aber eine andere räumliche Anordnung. Diese scheinbar kleine Änderung kann deutliche Stoffeigenschaftsunterschiede bewirken.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '>280 (Zersetzung/Sublimation)', 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': 'schlecht löslich', 'aggregatzustand_raumtemperatur': 'fest', 'bedingungen': 'ca. 20–25 °C', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv', 'warum': 'Zwei Carboxylgruppen und die C=C-Doppelbindung bieten mehrere Reaktionsmöglichkeiten.', 'reaktionsidee': 'Säure-Base-Reaktionen; Additionen an C=C unter geeigneten Bedingungen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}], 'Funktionelle Gruppen': [{'name': 'Methanol', 'smiles': 'CO', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man einen Alkohol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Methanol sehr gut mit Wasser mischbar?', 'antworten': ['Die OH-Gruppe bildet H-Brücken und der unpolare Rest ist sehr klein.', 'Methanol ist gut löslich, weil alle organischen Moleküle mit Sauerstoff vollständig wasserlöslich sind.', 'Methanol enthält eine Carbonylgruppe, die stark sauer reagiert.', 'Die C-H-Bindungen sind der Hauptgrund für die Wasserlöslichkeit.'], 'richtig': 0, 'feedback': 'Richtig: Die OH-Gruppe dominiert hier die Eigenschaften, weil der unpolare Teil klein ist.', 'erklaerung': 'Eine OH-Gruppe an einem gesättigten Kohlenstoff prägt Polarität und H-Brücken.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'methanol', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man einen Alkohol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Methanol sehr gut mit Wasser mischbar?', 'mc_antworten': ['Die OH-Gruppe bildet H-Brücken und der unpolare Rest ist sehr klein.', 'Methanol ist gut löslich, weil alle organischen Moleküle mit Sauerstoff vollständig wasserlöslich sind.', 'Methanol enthält eine Carbonylgruppe, die stark sauer reagiert.', 'Die C-H-Bindungen sind der Hauptgrund für die Wasserlöslichkeit.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die OH-Gruppe dominiert hier die Eigenschaften, weil der unpolare Teil klein ist.', 'erklaerung': 'Eine OH-Gruppe an einem gesättigten Kohlenstoff prägt Polarität und H-Brücken.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−97.6', 'siedepunkt_C': '64.7', 'dichte_g_cm3': '0.79', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die OH-Gruppe ermöglicht Oxidation, Veresterung und Säure-Base-Wechselwirkungen.', 'reaktionsidee': 'Oxidation zu Methanal; Bildung von Estern.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Ethanal', 'smiles': 'CC=O', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man einen Aldehyd?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Ethanal reaktiver als ein vergleichbarer Alkohol oder Alkan?', 'antworten': ['Die Aldehydgruppe enthält ein elektrophiles Carbonyl-C und kann leicht weiter reagieren.', 'Ethanal ist besonders reaktionsträge, weil es nur C-C- und C-H-Bindungen besitzt.', 'Ethanal ist ein Ether und kann deshalb keine Carbonylreaktionen eingehen.', 'Die funktionelle Gruppe spielt keine Rolle; nur die Summenformel entscheidet.'], 'richtig': 0, 'feedback': 'Richtig: Die Aldehydgruppe macht das Carbonyl-C für nucleophile Angriffe zugänglich und Aldehyde sind leicht oxidierbar.', 'erklaerung': 'Die Carbonylgruppe liegt am Kettenende.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ethanal', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man einen Aldehyd?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Ethanal reaktiver als ein vergleichbarer Alkohol oder Alkan?', 'mc_antworten': ['Die Aldehydgruppe enthält ein elektrophiles Carbonyl-C und kann leicht weiter reagieren.', 'Ethanal ist besonders reaktionsträge, weil es nur C-C- und C-H-Bindungen besitzt.', 'Ethanal ist ein Ether und kann deshalb keine Carbonylreaktionen eingehen.', 'Die funktionelle Gruppe spielt keine Rolle; nur die Summenformel entscheidet.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Aldehydgruppe macht das Carbonyl-C für nucleophile Angriffe zugänglich und Aldehyde sind leicht oxidierbar.', 'erklaerung': 'Die Carbonylgruppe liegt am Kettenende.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv als Aldehyd', 'warum': 'Das Carbonyl-C ist elektrophil; Aldehyde sind außerdem leichter oxidierbar als Ketone.', 'reaktionsidee': 'Nukleophile Addition; Oxidation zu Carbonsäuren.', 'modellgrenze': 'Qualitative Orientierung anhand der Aldehydgruppe.'}}, {'name': 'Aceton', 'smiles': 'CC(=O)C', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man ein Keton?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage beschreibt Aceton als Lösungsmittel am besten?', 'antworten': ['Aceton ist polar aprotisch: es kann H-Brücken aufnehmen, aber keine spenden.', 'Aceton ist ein Alkohol und kann deshalb H-Brücken spenden.', 'Aceton ist unpolar, weil es drei Kohlenstoffatome enthält.', 'Aceton besitzt eine Carboxylgruppe und reagiert sauer.'], 'richtig': 0, 'feedback': 'Richtig: Die C=O-Gruppe macht Aceton polar, aber ohne O-H/N-H kann es keine H-Brücken spenden.', 'erklaerung': 'Die Carbonylgruppe liegt zwischen zwei Kohlenstoffresten.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'aceton', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man ein Keton?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage beschreibt Aceton als Lösungsmittel am besten?', 'mc_antworten': ['Aceton ist polar aprotisch: es kann H-Brücken aufnehmen, aber keine spenden.', 'Aceton ist ein Alkohol und kann deshalb H-Brücken spenden.', 'Aceton ist unpolar, weil es drei Kohlenstoffatome enthält.', 'Aceton besitzt eine Carboxylgruppe und reagiert sauer.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die C=O-Gruppe macht Aceton polar, aber ohne O-H/N-H kann es keine H-Brücken spenden.', 'erklaerung': 'Die Carbonylgruppe liegt zwischen zwei Kohlenstoffresten.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−95', 'siedepunkt_C': '56.1', 'dichte_g_cm3': '0.79', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv als Keton', 'warum': 'Das Carbonyl-C ist elektrophil; Ketone sind aber gegenüber Oxidation stabiler als Aldehyde.', 'reaktionsidee': 'Nukleophile Addition an die C=O-Gruppe.', 'modellgrenze': 'Qualitative Orientierung anhand der Carbonylgruppe; Reaktionspartner und Bedingungen sind entscheidend.'}}, {'name': 'Essigsäure', 'smiles': 'CC(=O)O', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man eine Carbonsäure?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Essigsäure besonders gut für Struktur-Eigenschafts-Vergleiche geeignet?', 'antworten': ['Die Carboxylgruppe verbindet Polarität, H-Brücken und Säure-Base-Verhalten.', 'Essigsäure ist ein völlig unpolares Molekül.', 'Essigsäure enthält nur eine Ethergruppe.', 'Die Methylgruppe allein erklärt die hohe Wasserlöslichkeit.'], 'richtig': 0, 'feedback': 'Richtig: Die Carboxylgruppe ist der zentrale Strukturbaustein für die Eigenschaften.', 'erklaerung': 'C=O und OH am selben Kohlenstoffatom bilden die Carboxylgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'essigsaeure', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man eine Carbonsäure?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Essigsäure besonders gut für Struktur-Eigenschafts-Vergleiche geeignet?', 'mc_antworten': ['Die Carboxylgruppe verbindet Polarität, H-Brücken und Säure-Base-Verhalten.', 'Essigsäure ist ein völlig unpolares Molekül.', 'Essigsäure enthält nur eine Ethergruppe.', 'Die Methylgruppe allein erklärt die hohe Wasserlöslichkeit.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Carboxylgruppe ist der zentrale Strukturbaustein für die Eigenschaften.', 'erklaerung': 'C=O und OH am selben Kohlenstoffatom bilden die Carboxylgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '16.6', 'siedepunkt_C': '118.1', 'dichte_g_cm3': '1.05', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv in Säure-Base-Reaktionen', 'warum': 'Die Carboxylgruppe kann ein Proton abgeben und mit Alkoholen Ester bilden.', 'reaktionsidee': 'Neutralisation mit Basen; Esterbildung mit Alkoholen.', 'modellgrenze': 'Qualitative Orientierung; Dissoziation hängt von pH und Medium ab.'}}, {'name': 'Ethylacetat', 'smiles': 'CC(=O)OCC', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man einen Ester?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Was unterscheidet einen Ester wie Ethylacetat von einer Carbonsäure?', 'antworten': ['Der Ester besitzt keine saure O-H-Gruppe; dadurch ändern sich H-Brücken und Reaktivität.', 'Ester und Carbonsäuren reagieren gleich, wenn beide zwei O-Atome enthalten.', 'Ethylacetat ist ein Ionengitter und deshalb salzartig wasserlöslich.', 'Ein Ester enthält grundsätzlich keine polaren Bindungen.'], 'richtig': 0, 'feedback': 'Richtig: Beim Ester fehlt die saure O-H-Gruppe; Ester können hydrolysiert werden, sind aber keine Carbonsäuren.', 'erklaerung': 'Das Strukturmotiv ist C(=O)-O-C.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ethylacetat', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man einen Ester?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Was unterscheidet einen Ester wie Ethylacetat von einer Carbonsäure?', 'mc_antworten': ['Der Ester besitzt keine saure O-H-Gruppe; dadurch ändern sich H-Brücken und Reaktivität.', 'Ester und Carbonsäuren reagieren gleich, wenn beide zwei O-Atome enthalten.', 'Ethylacetat ist ein Ionengitter und deshalb salzartig wasserlöslich.', 'Ein Ester enthält grundsätzlich keine polaren Bindungen.'], 'mc_richtig': 0, 'feedback': 'Richtig: Beim Ester fehlt die saure O-H-Gruppe; Ester können hydrolysiert werden, sind aber keine Carbonsäuren.', 'erklaerung': 'Das Strukturmotiv ist C(=O)-O-C.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Ester können hydrolysiert werden, sind aber weniger reaktiv als Säureanhydride.', 'reaktionsidee': 'Saure oder basische Esterhydrolyse.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Diethylether', 'smiles': 'CCOCC', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man einen Ether?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Diethylether deutlich anders als Ethanol?', 'antworten': ['Er kann H-Brücken aufnehmen, aber mangels O-H-Gruppe nicht spenden.', 'Er enthält keine polaren Bindungen.', 'Er ist wegen des Sauerstoffatoms automatisch vollständig mit Wasser mischbar.', 'Er besitzt eine Carboxylgruppe und reagiert sauer.'], 'richtig': 0, 'feedback': 'Richtig: Ether-Sauerstoff kann H-Brücken aufnehmen, aber Diethylether kann keine spenden.', 'erklaerung': 'Ein Sauerstoffatom ist mit zwei Kohlenstoffresten verbunden.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'diethylether', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man einen Ether?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Diethylether deutlich anders als Ethanol?', 'mc_antworten': ['Er kann H-Brücken aufnehmen, aber mangels O-H-Gruppe nicht spenden.', 'Er enthält keine polaren Bindungen.', 'Er ist wegen des Sauerstoffatoms automatisch vollständig mit Wasser mischbar.', 'Er besitzt eine Carboxylgruppe und reagiert sauer.'], 'mc_richtig': 0, 'feedback': 'Richtig: Ether-Sauerstoff kann H-Brücken aufnehmen, aber Diethylether kann keine spenden.', 'erklaerung': 'Ein Sauerstoffatom ist mit zwei Kohlenstoffresten verbunden.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−116', 'siedepunkt_C': '34.6', 'dichte_g_cm3': '0.71', 'wasserloeslichkeit': 'begrenzt löslich', 'aggregatzustand_raumtemperatur': 'flüssig, sehr flüchtig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'relativ reaktionsträge unter milden Bedingungen', 'warum': 'Ether besitzen keine O-H-Bindung und reagieren meist weniger leicht als Alkohole.', 'reaktionsidee': 'Spaltung unter starken sauren Bedingungen; Peroxidbildung als Sicherheitsaspekt.', 'modellgrenze': 'Qualitative Orientierung; Sicherheitsaspekte hängen stark von Lagerung und Bedingungen ab.'}}, {'name': 'Acetanhydrid', 'smiles': 'CC(=O)OC(=O)C', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man ein Carbonsäureanhydrid?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum gilt Acetanhydrid als deutlich reaktiver als ein einfacher Ester?', 'antworten': ['Säureanhydride reagieren leicht mit Wasser, Alkoholen oder Aminen zu Carbonsäure-Derivaten.', 'Acetanhydrid ist unreaktiv, weil es keine Heteroatome besitzt.', 'Acetanhydrid ist ein Alkan und reagiert nur durch Verbrennung.', 'Der Name „Anhydrid“ bedeutet, dass das Molekül in Wasser gar nicht reagieren kann.'], 'richtig': 0, 'feedback': 'Richtig: Säureanhydride sind aktivierte Carbonsäure-Derivate und reagieren leicht mit Nukleophilen.', 'erklaerung': 'Zwei Acylgruppen sind über ein Sauerstoffatom verbunden.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'acetanhydrid', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man ein Carbonsäureanhydrid?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum gilt Acetanhydrid als deutlich reaktiver als ein einfacher Ester?', 'mc_antworten': ['Säureanhydride reagieren leicht mit Wasser, Alkoholen oder Aminen zu Carbonsäure-Derivaten.', 'Acetanhydrid ist unreaktiv, weil es keine Heteroatome besitzt.', 'Acetanhydrid ist ein Alkan und reagiert nur durch Verbrennung.', 'Der Name „Anhydrid“ bedeutet, dass das Molekül in Wasser gar nicht reagieren kann.'], 'mc_richtig': 0, 'feedback': 'Richtig: Säureanhydride sind aktivierte Carbonsäure-Derivate und reagieren leicht mit Nukleophilen.', 'erklaerung': 'Zwei Acylgruppen sind über ein Sauerstoffatom verbunden.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv gegenüber Nukleophilen', 'warum': 'Säureanhydride sind aktivierte Carbonsäure-Derivate und reagieren leicht mit Wasser, Alkoholen und Aminen.', 'reaktionsidee': 'Hydrolyse; Acylierung; Bildung von Estern oder Amiden.', 'modellgrenze': 'Qualitative Orientierung; genaue Reaktivität hängt von Reaktionspartner und Bedingungen ab.'}}, {'name': 'Ethylamin', 'smiles': 'CCN', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man ein Amin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum reagiert Ethylamin als Base?', 'antworten': ['Das freie Elektronenpaar am Stickstoff kann ein Proton aufnehmen.', 'Die C-H-Bindungen machen Ethylamin stark sauer.', 'Ethylamin ist ein Ester und wird deshalb hydrolysiert.', 'Aminogruppen sind immer völlig unpolar und reagieren nicht.'], 'richtig': 0, 'feedback': 'Richtig: Das freie Elektronenpaar am Stickstoff ist der zentrale Reaktivitätsgrund.', 'erklaerung': 'Amine enthalten Stickstoff und können basische Eigenschaften zeigen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ethylamin', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man ein Amin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum reagiert Ethylamin als Base?', 'mc_antworten': ['Das freie Elektronenpaar am Stickstoff kann ein Proton aufnehmen.', 'Die C-H-Bindungen machen Ethylamin stark sauer.', 'Ethylamin ist ein Ester und wird deshalb hydrolysiert.', 'Aminogruppen sind immer völlig unpolar und reagieren nicht.'], 'mc_richtig': 0, 'feedback': 'Richtig: Das freie Elektronenpaar am Stickstoff ist der zentrale Reaktivitätsgrund.', 'erklaerung': 'Amine enthalten Stickstoff und können basische Eigenschaften zeigen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv als Base/Nukleophil', 'warum': 'Das freie Elektronenpaar am Stickstoff kann Protonen aufnehmen und Bindungen bilden.', 'reaktionsidee': 'Bildung von Ammoniumsalzen; nukleophile Reaktionen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Acetamid', 'smiles': 'CC(=O)N', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man ein Amid?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Acetamid weniger reaktiv als viele andere Carbonylverbindungen?', 'antworten': ['Die Amidbindung ist durch Mesomerie stabilisiert; das Carbonyl-C ist weniger leicht angreifbar.', 'Acetamid enthält keine C=O-Gruppe.', 'Acetamid ist ein reines Alkan.', 'Amide sind immer stärker reaktiv als Säureanhydride.'], 'richtig': 0, 'feedback': 'Richtig: Mesomere Stabilisierung macht Amide vergleichsweise stabil.', 'erklaerung': 'C(=O)-N ist das zentrale Motiv; die Peptidbindung ist ebenfalls ein Amid.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'acetamid', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man ein Amid?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Acetamid weniger reaktiv als viele andere Carbonylverbindungen?', 'mc_antworten': ['Die Amidbindung ist durch Mesomerie stabilisiert; das Carbonyl-C ist weniger leicht angreifbar.', 'Acetamid enthält keine C=O-Gruppe.', 'Acetamid ist ein reines Alkan.', 'Amide sind immer stärker reaktiv als Säureanhydride.'], 'mc_richtig': 0, 'feedback': 'Richtig: Mesomere Stabilisierung macht Amide vergleichsweise stabil.', 'erklaerung': 'C(=O)-N ist das zentrale Motiv; die Peptidbindung ist ebenfalls ein Amid.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'gering bis mäßig reaktiv', 'warum': 'Die Amidbindung ist durch Mesomerie stabilisiert.', 'reaktionsidee': 'Hydrolyse unter kräftigen sauren oder basischen Bedingungen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Phenol', 'smiles': 'c1ccccc1O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Phenol nicht einfach ein normaler Alkohol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Phenol kein „normaler Alkohol“ wie Ethanol?', 'antworten': ['Die OH-Gruppe ist direkt an einen aromatischen Ring gebunden; dadurch ändern sich Acidität und Reaktivität.', 'Phenol besitzt keine OH-Gruppe.', 'Phenol ist ein Alkanol mit frei drehbarer C-C-Kette.', 'Der aromatische Ring hat keinen Einfluss auf die OH-Gruppe.'], 'richtig': 0, 'feedback': 'Richtig: Der aromatische Ring beeinflusst die O-H-Bindung und die Reaktivität deutlich.', 'erklaerung': 'Phenole unterscheiden sich in Eigenschaften und Reaktivität von einfachen Alkoholen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'phenol', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Phenol nicht einfach ein normaler Alkohol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Phenol kein „normaler Alkohol“ wie Ethanol?', 'mc_antworten': ['Die OH-Gruppe ist direkt an einen aromatischen Ring gebunden; dadurch ändern sich Acidität und Reaktivität.', 'Phenol besitzt keine OH-Gruppe.', 'Phenol ist ein Alkanol mit frei drehbarer C-C-Kette.', 'Der aromatische Ring hat keinen Einfluss auf die OH-Gruppe.'], 'mc_richtig': 0, 'feedback': 'Richtig: Der aromatische Ring beeinflusst die O-H-Bindung und die Reaktivität deutlich.', 'erklaerung': 'Phenole unterscheiden sich in Eigenschaften und Reaktivität von einfachen Alkoholen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv; schwach sauer', 'warum': 'Der aromatische Ring stabilisiert das Phenolat-Ion und beeinflusst Reaktionen am Ring.', 'reaktionsidee': 'Bildung von Phenolaten; elektrophile aromatische Substitution.', 'modellgrenze': 'Qualitative Orientierung; Phenol ist nicht einfach ein normaler Alkohol.'}}, {'name': 'Anilin', 'smiles': 'c1ccccc1N', 'niveau': 'Basis', 'leitfrage': 'Warum ist Anilin ein aromatisches Amin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum verhält sich Anilin anders als ein einfaches Alkylamin?', 'antworten': ['Die Aminogruppe ist mit einem aromatischen Ring verknüpft; dadurch werden Basizität und Ringreaktionen beeinflusst.', 'Anilin enthält keine Aminogruppe.', 'Der aromatische Ring macht Anilin zu einer Carbonsäure.', 'Anilin unterscheidet sich nur durch eine andere Summenformel, nicht durch Struktur.'], 'richtig': 0, 'feedback': 'Richtig: Der aromatische Ring verändert die Elektronenverteilung der Aminogruppe und des Rings.', 'erklaerung': 'Aromat und Aminogruppe beeinflussen sich gegenseitig.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'anilin', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Anilin ein aromatisches Amin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum verhält sich Anilin anders als ein einfaches Alkylamin?', 'mc_antworten': ['Die Aminogruppe ist mit einem aromatischen Ring verknüpft; dadurch werden Basizität und Ringreaktionen beeinflusst.', 'Anilin enthält keine Aminogruppe.', 'Der aromatische Ring macht Anilin zu einer Carbonsäure.', 'Anilin unterscheidet sich nur durch eine andere Summenformel, nicht durch Struktur.'], 'mc_richtig': 0, 'feedback': 'Richtig: Der aromatische Ring verändert die Elektronenverteilung der Aminogruppe und des Rings.', 'erklaerung': 'Aromat und Aminogruppe beeinflussen sich gegenseitig.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die Aminogruppe ist basisch/nukleophil und aktiviert den aromatischen Ring.', 'reaktionsidee': 'Salzbildung; elektrophile aromatische Substitution.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Salicylsäure', 'smiles': 'O=C(O)c1ccccc1O', 'niveau': 'Vertiefung', 'leitfrage': 'Welche funktionellen Gruppen erkennt man in Salicylsäure?', 'beobachten': ['Suche die Carboxylgruppe.', 'Suche die phenolische OH-Gruppe.', 'Achte auf den aromatischen Ring.'], 'frage': 'Warum ist Salicylsäure für funktionelle Gruppen besonders interessant?', 'antworten': ['Carboxylgruppe, Phenol-OH und Aromat liegen in einem kleinen Molekül zusammen.', 'Salicylsäure enthält nur Alkaneigenschaften.', 'Salicylsäure ist ein Ether ohne OH-Gruppe.', 'Der aromatische Ring verhindert jede Polarität.'], 'richtig': 0, 'feedback': 'Richtig: Mehrere funktionelle Gruppen beeinflussen Polarität und Reaktivität gleichzeitig.', 'erklaerung': 'Salicylsäure ist ein gutes Beispiel dafür, dass ein Molekül mehrere funktionelle Gruppen tragen kann. Diese Gruppen bestimmen gemeinsam Polarität, Säureeigenschaften und Wechselwirkungen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'salicylsaeure', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Welche funktionellen Gruppen erkennt man in Salicylsäure?', 'beobachten': ['Suche die Carboxylgruppe.', 'Suche die phenolische OH-Gruppe.', 'Achte auf den aromatischen Ring.'], 'mc_frage': 'Warum ist Salicylsäure für funktionelle Gruppen besonders interessant?', 'mc_antworten': ['Carboxylgruppe, Phenol-OH und Aromat liegen in einem kleinen Molekül zusammen.', 'Salicylsäure enthält nur Alkaneigenschaften.', 'Salicylsäure ist ein Ether ohne OH-Gruppe.', 'Der aromatische Ring verhindert jede Polarität.'], 'mc_richtig': 0, 'feedback': 'Richtig: Mehrere funktionelle Gruppen beeinflussen Polarität und Reaktivität gleichzeitig.', 'erklaerung': 'Salicylsäure ist ein gutes Beispiel dafür, dass ein Molekül mehrere funktionelle Gruppen tragen kann. Diese Gruppen bestimmen gemeinsam Polarität, Säureeigenschaften und Wechselwirkungen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv in Säure-Base- und Esterbildungsreaktionen', 'warum': 'Carboxylgruppe und Phenolgruppe ermöglichen mehrere Reaktionswege.', 'reaktionsidee': 'Salzbildung; Esterbildung; intramolekulare Wechselwirkungen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Vanillin', 'smiles': 'COc1cc(C=O)ccc1O', 'niveau': 'Vertiefung', 'leitfrage': 'Warum ist Vanillin ein gutes Beispiel für mehrere funktionelle Gruppen in einem Alltagsmolekül?', 'beobachten': ['Suche die Aldehydgruppe.', 'Suche die Methoxy-/Ethergruppe.', 'Suche die phenolische OH-Gruppe und den aromatischen Ring.'], 'frage': 'Welche Aussage erklärt Vanillin als gutes Beispiel für funktionelle Gruppen?', 'antworten': ['Ein bekanntes Molekül kann mehrere funktionelle Gruppen gleichzeitig besitzen.', 'Vanillin ist ein einfacher Kohlenwasserstoff ohne Heteroatome.', 'Vanillin enthält nur eine einzige C-C-Einfachbindung.', 'Die funktionellen Gruppen spielen für Geruch und Reaktivität keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Aldehyd, Phenol, Methoxygruppe und Aromat prägen gemeinsam die Eigenschaften.', 'erklaerung': 'Vanillin zeigt, wie mehrere funktionelle Gruppen zusammen ein charakteristisches Molekül ergeben. Die Struktur erklärt, warum es polarere Bereiche besitzt, aber auch einen aromatischen, eher unpolaren Bereich.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'vanillin', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Vanillin ein gutes Beispiel für mehrere funktionelle Gruppen in einem Alltagsmolekül?', 'beobachten': ['Suche die Aldehydgruppe.', 'Suche die Methoxy-/Ethergruppe.', 'Suche die phenolische OH-Gruppe und den aromatischen Ring.'], 'mc_frage': 'Welche Aussage erklärt Vanillin als gutes Beispiel für funktionelle Gruppen?', 'mc_antworten': ['Ein bekanntes Molekül kann mehrere funktionelle Gruppen gleichzeitig besitzen.', 'Vanillin ist ein einfacher Kohlenwasserstoff ohne Heteroatome.', 'Vanillin enthält nur eine einzige C-C-Einfachbindung.', 'Die funktionellen Gruppen spielen für Geruch und Reaktivität keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Aldehyd, Phenol, Methoxygruppe und Aromat prägen gemeinsam die Eigenschaften.', 'erklaerung': 'Vanillin zeigt, wie mehrere funktionelle Gruppen zusammen ein charakteristisches Molekül ergeben. Die Struktur erklärt, warum es polarere Bereiche besitzt, aber auch einen aromatischen, eher unpolaren Bereich.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv mit mehreren Reaktionszentren', 'warum': 'Aldehydgruppe, Phenol-OH und Methoxygruppe beeinflussen Elektronenverteilung und Reaktionen.', 'reaktionsidee': 'Reaktionen der Aldehydgruppe; Reaktionen am aromatischen Ring.', 'modellgrenze': 'Qualitative Orientierung; das Beispiel zeigt mehrere funktionelle Gruppen in einem Alltagsmolekül.'}}], 'Konformere': [{'name': 'Butan', 'smiles': 'CCCC', 'niveau': 'Basis', 'leitfrage': 'Warum gibt es bei Butan mehrere räumliche Formen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Worin unterscheiden sich anti- und gauche-Konformationen von n-Butan?', 'antworten': ['Sie entstehen durch Rotation um die zentrale C–C-Einfachbindung und unterscheiden sich in ihrer relativen Energie.', 'Die beiden Formen besitzen unterschiedliche Summenformeln.', 'Beim Übergang von anti zu gauche wird eine C–C-Bindung gebrochen.', 'Es handelt sich um Konstitutionsisomere wie Butan und Isobutan.'], 'richtig': 0, 'feedback': 'Anti und gauche sind Konformere desselben Moleküls: Der Diederwinkel zwischen den Methylgruppen beträgt ungefähr 180° bzw. 60°.', 'erklaerung': 'Rotation um die zentrale C–C-Einfachbindung verändert die räumliche Anordnung ohne Bindungsbruch; anti ist im einfachen Kraftfeldmodell energetisch günstiger als gauche.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'butan', 'typ': 'molecule', 'modul': 'Konformere', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum gibt es bei Butan mehrere räumliche Formen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Worin unterscheiden sich anti- und gauche-Konformationen von n-Butan?', 'mc_antworten': ['Sie entstehen durch Rotation um die zentrale C–C-Einfachbindung und unterscheiden sich in ihrer relativen Energie.', 'Die beiden Formen besitzen unterschiedliche Summenformeln.', 'Beim Übergang von anti zu gauche wird eine C–C-Bindung gebrochen.', 'Es handelt sich um Konstitutionsisomere wie Butan und Isobutan.'], 'mc_richtig': 0, 'feedback': 'Anti und gauche sind Konformere desselben Moleküls: Der Diederwinkel zwischen den Methylgruppen beträgt ungefähr 180° bzw. 60°.', 'erklaerung': 'Rotation um die zentrale C–C-Einfachbindung verändert die räumliche Anordnung ohne Bindungsbruch; anti ist im einfachen Kraftfeldmodell energetisch günstiger als gauche.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': '1,2-Dichlorethan', 'smiles': 'ClCCCl', 'niveau': 'Basis', 'leitfrage': 'Wie beeinflusst Rotation die Orientierung der Chloratome?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum können bei 1,2-Dichlorethan verschiedene Konformere unterschiedliche Eigenschaften haben?', 'antworten': ['Die räumliche Stellung der C-Cl-Bindungen verändert Dipolanteile und Nähe der Cl-Atome.', 'C-C-Einfachbindungen sind völlig starr und können nicht rotieren.', 'Die Chloratome verschwinden bei der 3D-Optimierung.', 'Alle Konformere haben immer exakt dieselbe räumliche Anordnung.'], 'richtig': 0, 'feedback': 'Richtig: Rotation um die C-C-Einfachbindung verändert die räumliche Anordnung der polaren C-Cl-Bindungen.', 'erklaerung': 'Rotation verändert Abstände und Wechselwirkungen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': '1_2_dichlorethan', 'typ': 'molecule', 'modul': 'Konformere', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Wie beeinflusst Rotation die Orientierung der Chloratome?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum können bei 1,2-Dichlorethan verschiedene Konformere unterschiedliche Eigenschaften haben?', 'mc_antworten': ['Die räumliche Stellung der C-Cl-Bindungen verändert Dipolanteile und Nähe der Cl-Atome.', 'C-C-Einfachbindungen sind völlig starr und können nicht rotieren.', 'Die Chloratome verschwinden bei der 3D-Optimierung.', 'Alle Konformere haben immer exakt dieselbe räumliche Anordnung.'], 'mc_richtig': 0, 'feedback': 'Richtig: Rotation um die C-C-Einfachbindung verändert die räumliche Anordnung der polaren C-Cl-Bindungen.', 'erklaerung': 'Rotation verändert Abstände und Wechselwirkungen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Ethylenglykol', 'smiles': 'OCCO', 'niveau': 'Basis', 'leitfrage': 'Warum kann Ethylenglykol mehrere günstige Formen einnehmen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Ethylenglykol ein gutes Beispiel für Konformere?', 'antworten': ['Mehrere Einfachbindungen und zwei OH-Gruppen erlauben unterschiedliche räumliche Anordnungen und H-Brücken-Möglichkeiten.', 'Ethylenglykol besitzt nur Doppelbindungen und ist deshalb starr.', 'Die OH-Gruppen verhindern jede Rotation.', 'Konformere entstehen nur bei aromatischen Molekülen.'], 'richtig': 0, 'feedback': 'Richtig: Bewegliche Einfachbindungen und polare OH-Gruppen machen verschiedene Anordnungen plausibel.', 'erklaerung': 'Die beiden OH-Gruppen und flexible Bindungen ermöglichen mehrere Konformere.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ethylenglykol', 'typ': 'molecule', 'modul': 'Konformere', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum kann Ethylenglykol mehrere günstige Formen einnehmen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Ethylenglykol ein gutes Beispiel für Konformere?', 'mc_antworten': ['Mehrere Einfachbindungen und zwei OH-Gruppen erlauben unterschiedliche räumliche Anordnungen und H-Brücken-Möglichkeiten.', 'Ethylenglykol besitzt nur Doppelbindungen und ist deshalb starr.', 'Die OH-Gruppen verhindern jede Rotation.', 'Konformere entstehen nur bei aromatischen Molekülen.'], 'mc_richtig': 0, 'feedback': 'Richtig: Bewegliche Einfachbindungen und polare OH-Gruppen machen verschiedene Anordnungen plausibel.', 'erklaerung': 'Die beiden OH-Gruppen und flexible Bindungen ermöglichen mehrere Konformere.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Cyclohexan', 'smiles': 'C1CCCCC1', 'niveau': 'Basis', 'leitfrage': 'Warum sind Sessel, Twist-Boat, Wanne und Halbsessel von Cyclohexan verschieden energiereich, obwohl die Bindungsverknüpfung gleich bleibt?', 'beobachten': ['Vergleiche die vier gezielt dargestellten Ringformen Sessel, Twist-Boat, Wanne und Halbsessel.', 'Untersuche gestaffelte bzw. ekliptische Bindungsstellungen und die Flaggenmast-Wechselwirkungen in der Wanne.', 'Ordne die MMFF-Modellpotentiale und unterscheide Minima von instabilen und geometrisch fixierten Übergangsmodellen.', 'Erkläre, warum die freie MMFF-Minimierung die Wanne und besonders den Halbsessel nicht zuverlässig als stabile Struktur erhält.'], 'frage': 'Welche Aussage zu Sessel, Twist-Boat, Wanne und Halbsessel von Cyclohexan trifft zu?', 'antworten': ['Alle vier Ringformen sind gleichermaßen stabile Energieminima.', 'Der Halbsessel ist energetisch günstiger als der Sessel, weil vier C-Atome annähernd koplanar sind.', 'Sessel und Twist-Boat sind Minima, die Wanne ist eine instabile Sattelgeometrie; der hochenergetische Halbsessel wird gezielt als eingeschränktes Übergangsmodell dargestellt.', 'Die relativen MMFF-Potentialenergien sind unmittelbar gemessene Ringinversionsbarrieren.'], 'richtig': 2, 'feedback': 'Richtig: Sessel und Twist-Boat sind stabile bzw. metastabile Minima. Die Wanne liegt höher und ist kein Minimum; der Halbsessel wird als geometrisch eingeschränkte Annäherung an die hochenergetische Übergangslage gezeigt.', 'erklaerung': 'Der Sessel hat günstige Winkel und weitgehend gestaffelte Bindungen. Das Twist-Boat ist ein höheres lokales Minimum; die Wanne besitzt ekliptische und Flaggenmast-Wechselwirkungen und ist eine Sattelgeometrie. Der Halbsessel ist für die Ringinversion die höchste der vier charakteristischen Formen. In DIDACTIC wird er mit Positionsrestriktionen erzeugt und sein MMFF-Potential ausgewertet, nicht als exakter Übergangszustand berechnet.', 'modellgrenze': 'Vier kuratierte MMFF-Geometriemodelle, keine vier stabilen Konformere. Sessel und Twist-Boat sind Minima, Wanne ist eine instabile Sattelgeometrie. Der Halbsessel wird als geometrisch eingeschränkte Referenz mit vier annähernd koplanaren Ring-C-Atomen erzeugt; eine vollständige Übergangszustandslokalisierung mit Hesse-/Frequenzprüfung erfolgt nicht. Relative MMFF-Potentiale sind keine experimentellen freien Energien oder nachgewiesenen Ringinversionsbarrieren.', 'id': 'cyclohexan', 'typ': 'molecule', 'modul': 'Konformere', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum sind Sessel, Twist-Boat, Wanne und Halbsessel von Cyclohexan verschieden energiereich, obwohl die Bindungsverknüpfung gleich bleibt?', 'beobachten': ['Vergleiche die vier gezielt dargestellten Ringformen Sessel, Twist-Boat, Wanne und Halbsessel.', 'Untersuche gestaffelte bzw. ekliptische Bindungsstellungen und die Flaggenmast-Wechselwirkungen in der Wanne.', 'Ordne die MMFF-Modellpotentiale und unterscheide Minima von instabilen und geometrisch fixierten Übergangsmodellen.', 'Erkläre, warum die freie MMFF-Minimierung die Wanne und besonders den Halbsessel nicht zuverlässig als stabile Struktur erhält.'], 'mc_frage': 'Welche Aussage zu Sessel, Twist-Boat, Wanne und Halbsessel von Cyclohexan trifft zu?', 'mc_antworten': ['Alle vier Ringformen sind gleichermaßen stabile Energieminima.', 'Der Halbsessel ist energetisch günstiger als der Sessel, weil vier C-Atome annähernd koplanar sind.', 'Sessel und Twist-Boat sind Minima, die Wanne ist eine instabile Sattelgeometrie; der hochenergetische Halbsessel wird gezielt als eingeschränktes Übergangsmodell dargestellt.', 'Die relativen MMFF-Potentialenergien sind unmittelbar gemessene Ringinversionsbarrieren.'], 'mc_richtig': 2, 'feedback': 'Richtig: Sessel und Twist-Boat sind stabile bzw. metastabile Minima. Die Wanne liegt höher und ist kein Minimum; der Halbsessel wird als geometrisch eingeschränkte Annäherung an die hochenergetische Übergangslage gezeigt.', 'erklaerung': 'Der Sessel hat günstige Winkel und weitgehend gestaffelte Bindungen. Das Twist-Boat ist ein höheres lokales Minimum; die Wanne besitzt ekliptische und Flaggenmast-Wechselwirkungen und ist eine Sattelgeometrie. Der Halbsessel ist für die Ringinversion die höchste der vier charakteristischen Formen. In DIDACTIC wird er mit Positionsrestriktionen erzeugt und sein MMFF-Potential ausgewertet, nicht als exakter Übergangszustand berechnet.', 'modellgrenze': 'Vier kuratierte MMFF-Geometriemodelle, keine vier stabilen Konformere. Sessel und Twist-Boat sind Minima, Wanne ist eine instabile Sattelgeometrie. Der Halbsessel wird als geometrisch eingeschränkte Referenz mit vier annähernd koplanaren Ring-C-Atomen erzeugt; eine vollständige Übergangszustandslokalisierung mit Hesse-/Frequenzprüfung erfolgt nicht. Relative MMFF-Potentiale sind keine experimentellen freien Energien oder nachgewiesenen Ringinversionsbarrieren.', 'mini_aufgaben': ['Ordne die vier Formen nach ihren Modellpotentialen und begründe die Unterschiede.', 'Welche der vier Darstellungen zeigen Minima, welche Übergangs-/Sattelgeometrien?', 'Erläutere die Ringinversion und warum der Halbsessel mit einer normalen Minimierung verschwindet.'], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Milchsäure', 'smiles': 'C[C@H](O)C(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Milchsäure ein gutes Beispiel für Flexibilität und Chiralität?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Was zeigt Milchsäure im Konformer-Modul besonders gut?', 'antworten': ['Ein kleines Molekül kann zugleich chiral, polar und räumlich beweglich sein.', 'Milchsäure ist ein unpolares Alkan ohne funktionelle Gruppe.', 'Chiralität bedeutet, dass sich die Summenformel ändert.', 'OH- und COOH-Gruppen haben keinen Einfluss auf Wechselwirkungen.'], 'richtig': 0, 'feedback': 'Richtig: Milchsäure verbindet Chiralität, Polarität und bewegliche Einfachbindungen.', 'erklaerung': 'Milchsäure verbindet funktionelle Gruppen, Chiralität und Beweglichkeit.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'milchsaeure', 'typ': 'molecule', 'modul': 'Konformere', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Milchsäure ein gutes Beispiel für Flexibilität und Chiralität?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Was zeigt Milchsäure im Konformer-Modul besonders gut?', 'mc_antworten': ['Ein kleines Molekül kann zugleich chiral, polar und räumlich beweglich sein.', 'Milchsäure ist ein unpolares Alkan ohne funktionelle Gruppe.', 'Chiralität bedeutet, dass sich die Summenformel ändert.', 'OH- und COOH-Gruppen haben keinen Einfluss auf Wechselwirkungen.'], 'mc_richtig': 0, 'feedback': 'Richtig: Milchsäure verbindet Chiralität, Polarität und bewegliche Einfachbindungen.', 'erklaerung': 'Milchsäure verbindet funktionelle Gruppen, Chiralität und Beweglichkeit.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'n-Pentan', 'smiles': 'CCCCC', 'niveau': 'Basis', 'leitfrage': 'Warum nimmt die Zahl möglicher räumlicher Formen bei längeren Alkanketten zu?', 'beobachten': ['Vergleiche n-Pentan mit Butan.', 'Achte auf die zusätzlichen C-C-Einfachbindungen.', 'Betrachte die niedrigsten Konformere nebeneinander.'], 'frage': 'Warum liefert n-Pentan mehr mögliche Konformere als Butan?', 'antworten': ['Es besitzt mehr rotierbare C-C-Einfachbindungen und dadurch mehr mögliche Kettenanordnungen.', 'n-Pentan ist polarer als Butan und bildet deshalb H-Brücken.', 'n-Pentan enthält eine Doppelbindung, die frei rotiert.', 'Konformere entstehen nur bei Molekülen mit Sauerstoff.'], 'richtig': 0, 'feedback': 'Richtig: Mehr rotierbare Einfachbindungen bedeuten mehr mögliche räumliche Anordnungen.', 'erklaerung': 'n-Pentan zeigt, dass mit wachsender Kettenlänge die Zahl möglicher Konformere schnell zunimmt. Die Molekülform wird flexibler, obwohl nur C-C- und C-H-Bindungen vorkommen.', 'modellgrenze': 'Die Konformer-Suche zeigt stabile Minima nach Modelloptimierung. Übergangslagen werden dadurch meist nicht sichtbar.', 'id': 'n_pentan', 'typ': 'molecule', 'modul': 'Konformere', 'tags': ['Konformere'], 'didaktik': {'leitfrage': 'Warum nimmt die Zahl möglicher räumlicher Formen bei längeren Alkanketten zu?', 'beobachten': ['Vergleiche n-Pentan mit Butan.', 'Achte auf die zusätzlichen C-C-Einfachbindungen.', 'Betrachte die niedrigsten Konformere nebeneinander.'], 'mc_frage': 'Warum liefert n-Pentan mehr mögliche Konformere als Butan?', 'mc_antworten': ['Es besitzt mehr rotierbare C-C-Einfachbindungen und dadurch mehr mögliche Kettenanordnungen.', 'n-Pentan ist polarer als Butan und bildet deshalb H-Brücken.', 'n-Pentan enthält eine Doppelbindung, die frei rotiert.', 'Konformere entstehen nur bei Molekülen mit Sauerstoff.'], 'mc_richtig': 0, 'feedback': 'Richtig: Mehr rotierbare Einfachbindungen bedeuten mehr mögliche räumliche Anordnungen.', 'erklaerung': 'n-Pentan zeigt, dass mit wachsender Kettenlänge die Zahl möglicher Konformere schnell zunimmt. Die Molekülform wird flexibler, obwohl nur C-C- und C-H-Bindungen vorkommen.', 'modellgrenze': 'Die Konformer-Suche zeigt stabile Minima nach Modelloptimierung. Übergangslagen werden dadurch meist nicht sichtbar.'}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'eher reaktionsträge', 'warum': 'n-Pentan enthält nur C-C- und C-H-Bindungen ohne stark polare funktionelle Gruppe.', 'reaktionsidee': 'Verbrennung; radikalische Substitution unter geeigneten Bedingungen.', 'modellgrenze': 'Qualitative Orientierung; Reaktivität hängt stark von Aktivierungsenergie und Reaktionsbedingungen ab.'}}, {'name': '2-Butanol', 'smiles': 'CCC(O)C', 'niveau': 'Vertiefung', 'leitfrage': 'Wie beeinflussen OH-Gruppe, Chiralitätszentrum und Einfachbindungen die räumlichen Formen?', 'beobachten': ['Suche die OH-Gruppe.', 'Achte auf das C-Atom, das vier verschiedene Gruppen trägt.', 'Vergleiche die berechneten Konformere.'], 'frage': 'Was macht 2-Butanol als Konformer-Beispiel interessanter als ein einfaches Alkan?', 'antworten': ['Es besitzt eine OH-Gruppe, ein chirales Zentrum und mehrere rotierbare Einfachbindungen.', '2-Butanol ist völlig starr und kann keine Konformere bilden.', 'Die OH-Gruppe macht das Molekül automatisch ionisch.', 'Chiralität bedeutet, dass keine räumlichen Anordnungen möglich sind.'], 'richtig': 0, 'feedback': 'Richtig: 2-Butanol verbindet Beweglichkeit, Polarität und Chiralität in einem noch überschaubaren Molekül.', 'erklaerung': '2-Butanol zeigt, dass selbst kleine Moleküle mehrere räumliche Aspekte gleichzeitig besitzen können: rotierbare Bindungen, eine polare OH-Gruppe und ein chirales Zentrum.', 'modellgrenze': 'Die Konformer-Suche zeigt stabile Minima nach Modelloptimierung. Übergangslagen werden dadurch meist nicht sichtbar.', 'id': '2_butanol', 'typ': 'molecule', 'modul': 'Konformere', 'tags': ['Konformere'], 'didaktik': {'leitfrage': 'Wie beeinflussen OH-Gruppe, Chiralitätszentrum und Einfachbindungen die räumlichen Formen?', 'beobachten': ['Suche die OH-Gruppe.', 'Achte auf das C-Atom, das vier verschiedene Gruppen trägt.', 'Vergleiche die berechneten Konformere.'], 'mc_frage': 'Was macht 2-Butanol als Konformer-Beispiel interessanter als ein einfaches Alkan?', 'mc_antworten': ['Es besitzt eine OH-Gruppe, ein chirales Zentrum und mehrere rotierbare Einfachbindungen.', '2-Butanol ist völlig starr und kann keine Konformere bilden.', 'Die OH-Gruppe macht das Molekül automatisch ionisch.', 'Chiralität bedeutet, dass keine räumlichen Anordnungen möglich sind.'], 'mc_richtig': 0, 'feedback': 'Richtig: 2-Butanol verbindet Beweglichkeit, Polarität und Chiralität in einem noch überschaubaren Molekül.', 'erklaerung': '2-Butanol zeigt, dass selbst kleine Moleküle mehrere räumliche Aspekte gleichzeitig besitzen können: rotierbare Bindungen, eine polare OH-Gruppe und ein chirales Zentrum.', 'modellgrenze': 'Die Konformer-Suche zeigt stabile Minima nach Modelloptimierung. Übergangslagen werden dadurch meist nicht sichtbar.'}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv als sekundärer Alkohol', 'warum': 'Die OH-Gruppe ermöglicht typische Alkoholreaktionen; sekundäre Alkohole können zu Ketonen oxidiert werden.', 'reaktionsidee': 'Oxidation zu 2-Butanon; Esterbildung mit Carbonsäuren.', 'modellgrenze': 'Qualitative Orientierung anhand der Alkoholgruppe; Reaktionsbedingungen entscheiden über den tatsächlichen Ablauf.'}}], 'Wirkstoffe und Biomoleküle': [{'name': 'Coffein', 'smiles': 'Cn1cnc2n(C)c(=O)n(C)c(=O)c12', 'niveau': 'Basis', 'leitfrage': 'Warum kann Coffein mit Biomolekülen wechselwirken?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Coffein trotz mehrerer N- und O-Atome nicht einfach „sehr wasserliebend“?', 'antworten': ['Es besitzt mehrere H-Brücken-Akzeptoren, aber keine Donoren und auch unpolare Molekülbereiche.', 'Coffein besitzt keine Heteroatome.', 'Coffein ist ein Salz und daher ausschließlich ionisch.', 'Die Ringstruktur verhindert jede Wechselwirkung mit Wasser.'], 'richtig': 0, 'feedback': 'Richtig: Coffein kann H-Brücken aufnehmen, aber nicht spenden; die Gesamtstruktur bleibt entscheidend.', 'erklaerung': 'Das heteroatomreiche Ringsystem erlaubt mehrere schwache Wechselwirkungen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'coffein', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum kann Coffein mit Biomolekülen wechselwirken?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Coffein trotz mehrerer N- und O-Atome nicht einfach „sehr wasserliebend“?', 'mc_antworten': ['Es besitzt mehrere H-Brücken-Akzeptoren, aber keine Donoren und auch unpolare Molekülbereiche.', 'Coffein besitzt keine Heteroatome.', 'Coffein ist ein Salz und daher ausschließlich ionisch.', 'Die Ringstruktur verhindert jede Wechselwirkung mit Wasser.'], 'mc_richtig': 0, 'feedback': 'Richtig: Coffein kann H-Brücken aufnehmen, aber nicht spenden; die Gesamtstruktur bleibt entscheidend.', 'erklaerung': 'Das heteroatomreiche Ringsystem erlaubt mehrere schwache Wechselwirkungen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Aspirin', 'smiles': 'CC(=O)Oc1ccccc1C(=O)O', 'niveau': 'Basis', 'leitfrage': 'Welche funktionellen Gruppen prägen Aspirin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Aspirin strukturchemisch interessant?', 'antworten': ['Carboxylgruppe, Estergruppe und Aromat verbinden Polarität, Reaktivität und lipophile Bereiche.', 'Aspirin besteht nur aus C-C- und C-H-Bindungen.', 'Aspirin ist eine Aminosäure.', 'Die funktionellen Gruppen spielen für Wirkung und Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Mehrere funktionelle Gruppen erklären wichtige Eigenschaften und Reaktionsmöglichkeiten.', 'erklaerung': 'Polare und unpolare Bereiche kommen in einem Wirkstoffmolekül zusammen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'aspirin', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Welche funktionellen Gruppen prägen Aspirin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Aspirin strukturchemisch interessant?', 'mc_antworten': ['Carboxylgruppe, Estergruppe und Aromat verbinden Polarität, Reaktivität und lipophile Bereiche.', 'Aspirin besteht nur aus C-C- und C-H-Bindungen.', 'Aspirin ist eine Aminosäure.', 'Die funktionellen Gruppen spielen für Wirkung und Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Mehrere funktionelle Gruppen erklären wichtige Eigenschaften und Reaktionsmöglichkeiten.', 'erklaerung': 'Polare und unpolare Bereiche kommen in einem Wirkstoffmolekül zusammen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Paracetamol', 'smiles': 'CC(=O)Nc1ccc(O)cc1', 'niveau': 'Basis', 'leitfrage': 'Welche Strukturmerkmale machen Paracetamol polar?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Gruppen ermöglichen Paracetamol Wasserstoffbrücken, und welcher Bereich ist weniger polar?', 'antworten': ['Phenolische OH-Gruppe und Amid können Wasserstoffbrücken bilden; der aromatische Ring ist vergleichsweise unpolar.', 'Nur der aromatische Ring bildet starke Wasserstoffbrücken mit Wasser.', 'Paracetamol besteht aus einem Ester und einer Carbonsäure.', 'Aus der Summenformel allein folgt die Wasserlöslichkeit eindeutig.'], 'richtig': 0, 'feedback': 'Phenolische OH-Gruppe und Amid ermöglichen Donor-/Akzeptor-Wechselwirkungen; die aromatische Struktur beeinflusst das Gesamtverhalten.', 'erklaerung': 'Die phenolische OH-Gruppe und die Amidgruppe bieten verschiedene polare Wechselwirkungsmöglichkeiten. Daraus folgt weder unmittelbar die Löslichkeit noch die pharmakologische Wirkung.', 'modellgrenze': 'Die neutrale 2D-Struktur beschreibt keine Solvatation, Kristallstruktur, Bindestelle oder pharmakologische Wirkung. RDKit-Modellwerte sind keine experimentellen Arzneistoffdaten.', 'id': 'paracetamol', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Welche Strukturmerkmale machen Paracetamol polar?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Gruppen ermöglichen Paracetamol Wasserstoffbrücken, und welcher Bereich ist weniger polar?', 'mc_antworten': ['Phenolische OH-Gruppe und Amid können Wasserstoffbrücken bilden; der aromatische Ring ist vergleichsweise unpolar.', 'Nur der aromatische Ring bildet starke Wasserstoffbrücken mit Wasser.', 'Paracetamol besteht aus einem Ester und einer Carbonsäure.', 'Aus der Summenformel allein folgt die Wasserlöslichkeit eindeutig.'], 'mc_richtig': 0, 'feedback': 'Phenolische OH-Gruppe und Amid ermöglichen Donor-/Akzeptor-Wechselwirkungen; die aromatische Struktur beeinflusst das Gesamtverhalten.', 'erklaerung': 'Die phenolische OH-Gruppe und die Amidgruppe bieten verschiedene polare Wechselwirkungsmöglichkeiten. Daraus folgt weder unmittelbar die Löslichkeit noch die pharmakologische Wirkung.', 'modellgrenze': 'Die neutrale 2D-Struktur beschreibt keine Solvatation, Kristallstruktur, Bindestelle oder pharmakologische Wirkung. RDKit-Modellwerte sind keine experimentellen Arzneistoffdaten.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Ibuprofen', 'smiles': 'CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Ibuprofen relativ lipophil?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum kann die Wasserlöslichkeit von Ibuprofen vom pH-Wert abhängen?', 'antworten': ['Die Carboxylgruppe kann deprotonieren; daneben besitzt Ibuprofen einen großen lipophilen Aromat- und Alkylbereich.', 'Ibuprofen hat keine ionisierbare Gruppe und liegt bei jedem pH gleich vor.', 'Der aromatische Ring wird bei höherem pH immer zu einem positiv geladenen Ion.', 'Die zwei Sauerstoffatome machen jede Ibuprofenform vollständig mit Wasser mischbar.'], 'richtig': 0, 'feedback': 'Eine Carboxylgruppe kann pH-abhängig ionisiert sein; der große Kohlenwasserstoffanteil beeinflusst das Verhalten der neutralen Form.', 'erklaerung': 'Die saure Carboxylgruppe kann zu einem Carboxylat deprotonieren; die aromatische und verzweigte Seitenkette ist überwiegend lipophil. Entscheidend sind pH, Stoffform und Umgebung.', 'modellgrenze': 'Die gezeigte Struktur besitzt eine definierte Stereokonfiguration und zeigt die neutrale Säureform. RDKit-logP ist kein pH-abhängiger logD und keine gesicherte Wasserlöslichkeitsprognose.', 'id': 'ibuprofen', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Ibuprofen relativ lipophil?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum kann die Wasserlöslichkeit von Ibuprofen vom pH-Wert abhängen?', 'mc_antworten': ['Die Carboxylgruppe kann deprotonieren; daneben besitzt Ibuprofen einen großen lipophilen Aromat- und Alkylbereich.', 'Ibuprofen hat keine ionisierbare Gruppe und liegt bei jedem pH gleich vor.', 'Der aromatische Ring wird bei höherem pH immer zu einem positiv geladenen Ion.', 'Die zwei Sauerstoffatome machen jede Ibuprofenform vollständig mit Wasser mischbar.'], 'mc_richtig': 0, 'feedback': 'Eine Carboxylgruppe kann pH-abhängig ionisiert sein; der große Kohlenwasserstoffanteil beeinflusst das Verhalten der neutralen Form.', 'erklaerung': 'Die saure Carboxylgruppe kann zu einem Carboxylat deprotonieren; die aromatische und verzweigte Seitenkette ist überwiegend lipophil. Entscheidend sind pH, Stoffform und Umgebung.', 'modellgrenze': 'Die gezeigte Struktur besitzt eine definierte Stereokonfiguration und zeigt die neutrale Säureform. RDKit-logP ist kein pH-abhängiger logD und keine gesicherte Wasserlöslichkeitsprognose.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Acetazolamid', 'smiles': 'CC(=O)Nc1nnc(s1)S(N)(=O)=O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Acetazolamid ein Brückenbeispiel zum Proteinlabor?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Strukturteile von Acetazolamid sind für polare Wechselwirkungen und Ionisierbarkeit bedeutsam?', 'antworten': ['Sulfonamid, Amidgruppe und heteroatomreicher Ring bieten verschiedene polare Wechselwirkungen; der Ladungszustand hängt von der Umgebung ab.', 'Das Molekül enthält ausschließlich Kohlenstoff und Wasserstoff.', 'Acetazolamid besitzt weder Sauerstoff noch Stickstoff oder Schwefel.', 'Alle N-Atome sind unabhängig vom pH-Wert stets positiv geladen.'], 'richtig': 0, 'feedback': 'Sulfonamid- und Amidgruppen sowie der heteroatomreiche Ring tragen zu polaren Wechselwirkungen bei; die Sulfonamidgruppe kann ionisieren.', 'erklaerung': 'Acetazolamid besitzt eine Sulfonamidgruppe, eine Amidgruppe und ein schwefel-/stickstoffhaltiges Ringsystem. Diese Struktur erklärt mögliche Wechselwirkungen, nicht automatisch die Bindung an ein Protein.', 'modellgrenze': 'Die Darstellung enthält keine Protonierungszustandsrechnung und keine Bindepose an Carboanhydrase. Biologische Wirksamkeit lässt sich aus der Elementoberfläche oder RDKit-Deskriptoren nicht ableiten.', 'id': 'acetazolamid', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Acetazolamid ein Brückenbeispiel zum Proteinlabor?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Strukturteile von Acetazolamid sind für polare Wechselwirkungen und Ionisierbarkeit bedeutsam?', 'mc_antworten': ['Sulfonamid, Amidgruppe und heteroatomreicher Ring bieten verschiedene polare Wechselwirkungen; der Ladungszustand hängt von der Umgebung ab.', 'Das Molekül enthält ausschließlich Kohlenstoff und Wasserstoff.', 'Acetazolamid besitzt weder Sauerstoff noch Stickstoff oder Schwefel.', 'Alle N-Atome sind unabhängig vom pH-Wert stets positiv geladen.'], 'mc_richtig': 0, 'feedback': 'Sulfonamid- und Amidgruppen sowie der heteroatomreiche Ring tragen zu polaren Wechselwirkungen bei; die Sulfonamidgruppe kann ionisieren.', 'erklaerung': 'Acetazolamid besitzt eine Sulfonamidgruppe, eine Amidgruppe und ein schwefel-/stickstoffhaltiges Ringsystem. Diese Struktur erklärt mögliche Wechselwirkungen, nicht automatisch die Bindung an ein Protein.', 'modellgrenze': 'Die Darstellung enthält keine Protonierungszustandsrechnung und keine Bindepose an Carboanhydrase. Biologische Wirksamkeit lässt sich aus der Elementoberfläche oder RDKit-Deskriptoren nicht ableiten.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Glycin', 'smiles': 'NCC(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Glycin die einfachste Aminosäure?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Glycin trotz kleiner Größe stark polar?', 'antworten': ['Aminogruppe und Carboxylgruppe können stark mit Wasser und Protonen wechselwirken.', 'Glycin ist ein reiner Kohlenwasserstoff.', 'Die geringe Molekülgröße macht Glycin automatisch unpolar.', 'Aminosäuren besitzen grundsätzlich keine funktionellen Gruppen.'], 'richtig': 0, 'feedback': 'Richtig: Amino- und Carboxylgruppe prägen die Polarität und Säure-Base-Eigenschaften.', 'erklaerung': 'Die Seitenkette ist nur ein Wasserstoffatom.', 'modellgrenze': 'Glycin ist achiral; das neutrale SMILES ist eine vereinfachte Struktur. In Wasser ist je nach pH die Zwitterionform besonders wichtig; die Neutralform-Deskriptoren beschreiben nicht alle wässrigen Eigenschaften.', 'id': 'glycin', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Glycin die einfachste Aminosäure?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Glycin trotz kleiner Größe stark polar?', 'mc_antworten': ['Aminogruppe und Carboxylgruppe können stark mit Wasser und Protonen wechselwirken.', 'Glycin ist ein reiner Kohlenwasserstoff.', 'Die geringe Molekülgröße macht Glycin automatisch unpolar.', 'Aminosäuren besitzen grundsätzlich keine funktionellen Gruppen.'], 'mc_richtig': 0, 'feedback': 'Richtig: Amino- und Carboxylgruppe prägen die Polarität und Säure-Base-Eigenschaften.', 'erklaerung': 'Die Seitenkette ist nur ein Wasserstoffatom.', 'modellgrenze': 'Glycin ist achiral; das neutrale SMILES ist eine vereinfachte Struktur. In Wasser ist je nach pH die Zwitterionform besonders wichtig; die Neutralform-Deskriptoren beschreiben nicht alle wässrigen Eigenschaften.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Alanin', 'smiles': 'C[C@H](N)C(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Alanin chiral?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Alanin chiral und warum hängt seine Ladungsform vom pH-Wert ab?', 'antworten': ['Das α-C trägt vier verschiedene Substituenten; Amino- und Carboxylgruppe können Protonen aufnehmen bzw. abgeben.', 'Alanin besitzt kein stereogenes Zentrum, weil es eine Methylgruppe enthält.', 'Die Aminogruppe kann niemals protoniert werden.', 'Die neutrale Struktur ist bei allen pH-Werten die einzige Form in Wasser.'], 'richtig': 0, 'feedback': 'Alanin ist am α-C chiral; in wässriger Lösung bestimmen Säure-Base-Gleichgewichte den Protonierungszustand der Amino- und Carboxylgruppe.', 'erklaerung': 'Die Gruppen H, CH₃, NH₂ und COOH am α-C sind verschieden. In Wasser liegt Alanin häufig als Zwitterion vor; das gezeigte neutrale SMILES ist ein vereinfachtes Strukturmodell.', 'modellgrenze': 'Das chirale SMILES zeigt die neutrale Struktur; bei passenden wässrigen pH-Werten überwiegt die Zwitterionform. Berechnete Neutralform-Deskriptoren sind nicht automatisch Eigenschaften des wässrigen Alanins.', 'id': 'alanin', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Alanin chiral?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Alanin chiral und warum hängt seine Ladungsform vom pH-Wert ab?', 'mc_antworten': ['Das α-C trägt vier verschiedene Substituenten; Amino- und Carboxylgruppe können Protonen aufnehmen bzw. abgeben.', 'Alanin besitzt kein stereogenes Zentrum, weil es eine Methylgruppe enthält.', 'Die Aminogruppe kann niemals protoniert werden.', 'Die neutrale Struktur ist bei allen pH-Werten die einzige Form in Wasser.'], 'mc_richtig': 0, 'feedback': 'Alanin ist am α-C chiral; in wässriger Lösung bestimmen Säure-Base-Gleichgewichte den Protonierungszustand der Amino- und Carboxylgruppe.', 'erklaerung': 'Die Gruppen H, CH₃, NH₂ und COOH am α-C sind verschieden. In Wasser liegt Alanin häufig als Zwitterion vor; das gezeigte neutrale SMILES ist ein vereinfachtes Strukturmodell.', 'modellgrenze': 'Das chirale SMILES zeigt die neutrale Struktur; bei passenden wässrigen pH-Werten überwiegt die Zwitterionform. Berechnete Neutralform-Deskriptoren sind nicht automatisch Eigenschaften des wässrigen Alanins.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Phenylalanin', 'smiles': 'N[C@@H](Cc1ccccc1)C(=O)O', 'niveau': 'Basis', 'leitfrage': 'Was unterscheidet Phenylalanin von Glycin und Alanin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Wie beeinflussen die Seitenkette und ionisierbaren Gruppen die Eigenschaften von Phenylalanin?', 'antworten': ['Die aromatische Seitenkette ist überwiegend unpolar; Amino- und Carboxylgruppe können pH-abhängige Ladungen tragen.', 'Die aromatische Seitenkette ist eine stark saure Carboxylgruppe.', 'Phenylalanin enthält keine Aminogruppe und keine Carboxylgruppe.', 'Das Molekül besteht ausschließlich aus unpolaren Bindungen und kann keine Ionen bilden.'], 'richtig': 0, 'feedback': 'Die Phenylgruppe ermöglicht hydrophobe und aromatische Wechselwirkungen; Amino- und Carboxylgruppe prägen die Säure-Base-Eigenschaften.', 'erklaerung': 'Phenylalanin kombiniert eine hydrophobe aromatische Seitenkette mit den für Aminosäuren typischen ionisierbaren Gruppen. In Proteinen hängt die Rolle dieser Seitenkette von ihrer Umgebung ab.', 'modellgrenze': 'Das SMILES zeigt eine definierte Konfiguration, aber eine neutrale Aminosäureform. pH-abhängige Zwitterionen, Proteinstruktur und Bindungsenergie werden nicht berechnet.', 'id': 'phenylalanin', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Was unterscheidet Phenylalanin von Glycin und Alanin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Wie beeinflussen die Seitenkette und ionisierbaren Gruppen die Eigenschaften von Phenylalanin?', 'mc_antworten': ['Die aromatische Seitenkette ist überwiegend unpolar; Amino- und Carboxylgruppe können pH-abhängige Ladungen tragen.', 'Die aromatische Seitenkette ist eine stark saure Carboxylgruppe.', 'Phenylalanin enthält keine Aminogruppe und keine Carboxylgruppe.', 'Das Molekül besteht ausschließlich aus unpolaren Bindungen und kann keine Ionen bilden.'], 'mc_richtig': 0, 'feedback': 'Die Phenylgruppe ermöglicht hydrophobe und aromatische Wechselwirkungen; Amino- und Carboxylgruppe prägen die Säure-Base-Eigenschaften.', 'erklaerung': 'Phenylalanin kombiniert eine hydrophobe aromatische Seitenkette mit den für Aminosäuren typischen ionisierbaren Gruppen. In Proteinen hängt die Rolle dieser Seitenkette von ihrer Umgebung ab.', 'modellgrenze': 'Das SMILES zeigt eine definierte Konfiguration, aber eine neutrale Aminosäureform. pH-abhängige Zwitterionen, Proteinstruktur und Bindungsenergie werden nicht berechnet.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Adenin', 'smiles': 'Nc1ncnc2[nH]cnc12', 'niveau': 'Basis', 'leitfrage': 'Warum ist Adenin als Nukleobase interessant?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Adenin für Biomoleküle wichtig?', 'antworten': ['Es ist eine stickstoffhaltige Nukleobase und kann spezifische Wechselwirkungen eingehen.', 'Adenin ist ein einfacher Alkohol.', 'Adenin enthält nur C-H-Bindungen.', 'Nukleobasen spielen bei DNA/RNA keine strukturelle Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Adenin ist eine Nukleobase und kann über H-Brücken und Stapelwechselwirkungen an Strukturen beteiligt sein.', 'erklaerung': 'Das heteroaromatische Ringsystem kann an Basenpaarung beteiligt sein.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'adenin', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Adenin als Nukleobase interessant?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Adenin für Biomoleküle wichtig?', 'mc_antworten': ['Es ist eine stickstoffhaltige Nukleobase und kann spezifische Wechselwirkungen eingehen.', 'Adenin ist ein einfacher Alkohol.', 'Adenin enthält nur C-H-Bindungen.', 'Nukleobasen spielen bei DNA/RNA keine strukturelle Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Adenin ist eine Nukleobase und kann über H-Brücken und Stapelwechselwirkungen an Strukturen beteiligt sein.', 'erklaerung': 'Das heteroaromatische Ringsystem kann an Basenpaarung beteiligt sein.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Ribose', 'smiles': 'O=CC(O)C(O)C(O)CO', 'niveau': 'Basis', 'leitfrage': 'Warum ist Ribose ein wichtiger RNA-Baustein?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Ribose stark wasserfreundlich?', 'antworten': ['Mehrere OH-Gruppen ermöglichen viele Wasserstoffbrücken mit Wasser.', 'Ribose ist ein unpolarer Kohlenwasserstoff.', 'Zucker sind wasserunlöslich, weil sie nur C-H-Bindungen besitzen.', 'Die OH-Gruppen machen Ribose vollständig lipophil.'], 'richtig': 0, 'feedback': 'Richtig: Mehrere OH-Gruppen ermöglichen viele Wechselwirkungen mit Wasser.', 'erklaerung': 'In RNA ist Ribose Teil des Zucker-Phosphat-Rückgrats; hier wird die offenkettige Form vereinfacht gezeigt.', 'modellgrenze': 'Das SMILES zeigt eine offenkettige Ribose ohne Stereomarker. In Wasser liegen überwiegend cyclische Formen im Gleichgewicht vor. Die gezeigte Struktur darf nicht als vollständige 3D-/Stereochemie der biologischen D-Ribose interpretiert werden.', 'id': 'ribose', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Ribose ein wichtiger RNA-Baustein?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Ribose stark wasserfreundlich?', 'mc_antworten': ['Mehrere OH-Gruppen ermöglichen viele Wasserstoffbrücken mit Wasser.', 'Ribose ist ein unpolarer Kohlenwasserstoff.', 'Zucker sind wasserunlöslich, weil sie nur C-H-Bindungen besitzen.', 'Die OH-Gruppen machen Ribose vollständig lipophil.'], 'mc_richtig': 0, 'feedback': 'Richtig: Mehrere OH-Gruppen ermöglichen viele Wechselwirkungen mit Wasser.', 'erklaerung': 'In RNA ist Ribose Teil des Zucker-Phosphat-Rückgrats; hier wird die offenkettige Form vereinfacht gezeigt.', 'modellgrenze': 'Das SMILES zeigt eine offenkettige Ribose ohne Stereomarker. In Wasser liegen überwiegend cyclische Formen im Gleichgewicht vor. Die gezeigte Struktur darf nicht als vollständige 3D-/Stereochemie der biologischen D-Ribose interpretiert werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Citronensäure', 'smiles': 'O=C(O)CC(O)(CC(=O)O)C(=O)O', 'niveau': 'Vertiefung', 'leitfrage': 'Warum ist Citronensäure stark wasserfreundlich und als Stoffwechselprodukt interessant?', 'beobachten': ['Suche die drei Carboxylgruppen.', 'Suche die OH-Gruppe.', 'Vergleiche TPSA und H-Brücken-Möglichkeiten.'], 'frage': 'Warum ist Citronensäure stark wasserfreundlich und reaktiv in Säure-Base-Zusammenhängen?', 'antworten': ['Mehrere Carboxylgruppen und eine OH-Gruppe ermöglichen starke Wechselwirkungen mit Wasser.', 'Citronensäure besteht fast nur aus unpolaren C-H-Bindungen.', 'Citronensäure ist ein Protein und deshalb wasserlöslich.', 'Nur die Molmasse entscheidet über Wasserlöslichkeit.'], 'richtig': 0, 'feedback': 'Richtig: Mehrere polare Gruppen prägen die Wasserfreundlichkeit und Säureeigenschaften.', 'erklaerung': 'Citronensäure ist noch überschaubar, zeigt aber bereits sehr deutlich, wie mehrere polare funktionelle Gruppen ein Molekül stark wasserfreundlich machen. Gleichzeitig stellt sie eine Brücke zum Stoffwechsel dar.', 'modellgrenze': 'In biologischen Systemen liegen Carbonsäuren häufig teilweise oder vollständig deprotoniert vor; diese einfache Darstellung bildet pH- und Salzformen nicht vollständig ab.', 'id': 'citronensaeure', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Citronensäure stark wasserfreundlich und als Stoffwechselprodukt interessant?', 'beobachten': ['Suche die drei Carboxylgruppen.', 'Suche die OH-Gruppe.', 'Vergleiche TPSA und H-Brücken-Möglichkeiten.'], 'mc_frage': 'Warum ist Citronensäure stark wasserfreundlich und reaktiv in Säure-Base-Zusammenhängen?', 'mc_antworten': ['Mehrere Carboxylgruppen und eine OH-Gruppe ermöglichen starke Wechselwirkungen mit Wasser.', 'Citronensäure besteht fast nur aus unpolaren C-H-Bindungen.', 'Citronensäure ist ein Protein und deshalb wasserlöslich.', 'Nur die Molmasse entscheidet über Wasserlöslichkeit.'], 'mc_richtig': 0, 'feedback': 'Richtig: Mehrere polare Gruppen prägen die Wasserfreundlichkeit und Säureeigenschaften.', 'erklaerung': 'Citronensäure ist noch überschaubar, zeigt aber bereits sehr deutlich, wie mehrere polare funktionelle Gruppen ein Molekül stark wasserfreundlich machen. Gleichzeitig stellt sie eine Brücke zum Stoffwechsel dar.', 'modellgrenze': 'In biologischen Systemen liegen Carbonsäuren häufig teilweise oder vollständig deprotoniert vor; diese einfache Darstellung bildet pH- und Salzformen nicht vollständig ab.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv in Säure-Base- und Komplexbildungs-Zusammenhängen', 'warum': 'Mehrere Carboxylgruppen können Protonen abgeben; zusätzlich kann das Molekül Metallionen binden.', 'reaktionsidee': 'Neutralisation; Bildung von Citrat-Salzen und Metall-Komplexen.', 'modellgrenze': 'Qualitative Orientierung; pH-Wert und Ionisierungszustand sind entscheidend.'}}, {"id":"atorvastatin","name":"Atorvastatin","smiles":"CC(C)C1=C(C(=C(N1CC[C@H](C[C@H](CC(=O)O)O)O)C2=CC=C(C=C2)F)C3=CC=CC=C3)C(=O)NC4=CC=CC=C4","niveau":"Vertiefung","leitfrage":"Wie kann ein Wirkstoff mit 41 Schweratomen mehrere funktionelle Gruppen, aromatische Ringe, zwei Stereozentren und flexible Molekülbereiche vereinen?","beobachten":["Markiere die Carboxylgruppe, zwei Alkoholgruppen, die Amidgruppe, die aromatischen Ringe und die Aryl-F-Bindung.","Suche die beiden Stereozentren der Dihydroxy-Seitenkette: Konfiguration (3R,5R).","Betrachte das 3D-Modell und unterscheide starre Ringsysteme und bewegliche Kettenabschnitte.","Vergleiche 41 Schweratome und zwölf RDKit-rotierbare Bindungen. Warum ist eine einzige 3D-Geometrie nicht eindeutig?"],"frage":"Welche Aussage erklärt die molekulare Komplexität von Atorvastatin am besten?","antworten":["Atorvastatin besteht aus einem einzelnen starren aromatischen Ring ohne stereogene Zentren.","Mehrere aromatische Ringe, polare Gruppen, eine flexible Seitenkette und zwei definierte Stereozentren kommen in einem Molekül zusammen.","Eine einzelne Kraftfeld-Minimierung liefert die experimentell nachgewiesene Bindepose im Zielprotein.","Die Summenformel legt die pharmakologische Wirkung und die Wasserlöslichkeit eindeutig fest."],"richtig":1,"feedback":"Richtig: Vier Ringsysteme, Carboxyl-, Alkohol- und Amidfunktionen, fluoriertes Aromatensystem, flexible Kette und zwei stereogene Zentren erklären die hohe strukturelle Vielfalt.","erklaerung":"Atorvastatin (PubChem CID 60823, freie Säure) enthält drei Phenylringe, einen substituierten Pyrrolring, zwei alkoholische OH-Gruppen, eine Carboxylgruppe und eine Amidfunktion sowie eine Aryl-F-Bindung. RDKit erkennt 41 Schweratome, zwölf rotierbare Bindungen und zwei CIP-R-Stereozentren (3R,5R).","modellgrenze":"Dargestellt ist die freie neutrale Atorvastatinsäure, nicht das häufig verwendete Calciumsalz oder das pH-abhängige Carboxylat. Eine einzelne RDKit-MMFF-Geometrie ist weder die biologisch aktive Bindepose noch eine Vorhersage der Wasserlöslichkeit, Wirksamkeit oder Verteilung.","typ":"molecule","modul":"Wirkstoffe und Biomoleküle","tags":["Komplexität","Stereochemie"],"didaktik":{"fehlvorstellung":"Ein einziges 3D-Modell und die Summenformel erlauben eine eindeutige biologische Wirkungsprognose.","leitfrage":"Wie kann ein Wirkstoff mit 41 Schweratomen mehrere funktionelle Gruppen, aromatische Ringe, zwei Stereozentren und flexible Molekülbereiche vereinen?","beobachten":["Markiere die Carboxylgruppe, zwei Alkoholgruppen, die Amidgruppe, die aromatischen Ringe und die Aryl-F-Bindung.","Suche die beiden Stereozentren der Dihydroxy-Seitenkette: Konfiguration (3R,5R).","Betrachte das 3D-Modell und unterscheide starre Ringsysteme und bewegliche Kettenabschnitte.","Vergleiche 41 Schweratome und zwölf RDKit-rotierbare Bindungen. Warum ist eine einzige 3D-Geometrie nicht eindeutig?"],"mc_frage":"Welche Aussage erklärt die molekulare Komplexität von Atorvastatin am besten?","mc_antworten":["Atorvastatin besteht aus einem einzelnen starren aromatischen Ring ohne stereogene Zentren.","Mehrere aromatische Ringe, polare Gruppen, eine flexible Seitenkette und zwei definierte Stereozentren kommen in einem Molekül zusammen.","Eine einzelne Kraftfeld-Minimierung liefert die experimentell nachgewiesene Bindepose im Zielprotein.","Die Summenformel legt die pharmakologische Wirkung und die Wasserlöslichkeit eindeutig fest."],"mc_richtig":1,"feedback":"Richtig: Vier Ringsysteme, Carboxyl-, Alkohol- und Amidfunktionen, fluoriertes Aromatensystem, flexible Kette und zwei stereogene Zentren erklären die hohe strukturelle Vielfalt.","erklaerung":"Atorvastatin (PubChem CID 60823, freie Säure) enthält drei Phenylringe, einen substituierten Pyrrolring, zwei alkoholische OH-Gruppen, eine Carboxylgruppe und eine Amidfunktion sowie eine Aryl-F-Bindung. RDKit erkennt 41 Schweratome, zwölf rotierbare Bindungen und zwei CIP-R-Stereozentren (3R,5R).","modellgrenze":"Dargestellt ist die freie neutrale Atorvastatinsäure, nicht das häufig verwendete Calciumsalz oder das pH-abhängige Carboxylat. Eine einzelne RDKit-MMFF-Geometrie ist weder die biologisch aktive Bindepose noch eine Vorhersage der Wasserlöslichkeit, Wirksamkeit oder Verteilung.","mini_aufgaben":["Finde und markiere Carboxylgruppe, OH-Gruppen, Amid und Arylfluorid.","Suche die zwei Stereozentren und erkläre, warum sie pharmakochemisch relevant sein können.","Unterscheide die starren aromatischen Ringsysteme von flexiblen Kettensegmenten. Was zeigt ein einzelnes 3D-Modell nicht?"],"experimenteller_bezug":"Geprüfte Atorvastatin-Struktur der freien Säure, nicht Arzneistoffsalz oder Kristallhydratform."},"stoffdaten":{"schmelzpunkt_C":None,"siedepunkt_C":None,"dichte_g_cm3":None,"wasserloeslichkeit":None,"aggregatzustand_raumtemperatur":None,"bedingungen":None,"quelle":"PubChem CID 60823 (Atorvastatin, neutrale freie Säure)","hinweis":"Referenzstruktur C₃₃H₃₅FN₂O₅; M = 558,65 g/mol; (3R,5R). Keine ungesicherten experimentellen Werte ergänzt."},"reaktivitaet":{"hinweis":"Qualitative Reaktionsmöglichkeiten von Carboxylgruppe, OH-Gruppen und Amid.","warum":"Die Säuregruppe ist ionisierbar; die beiden OH-Gruppen und Amidfunktion erlauben weitere chemische Wechselwirkungen.","reaktionsidee":"Säure-Base-Verhalten und Salzbildung an der Carboxylgruppe; keine Vorhersage von Wirkmechanismus oder Metabolismus.","modellgrenze":"Nur qualitative Gruppenchemie, keine Reaktionsvorhersage oder Proteinbindungssimulation."}}]}, 'isomer_pairs': {'Ethanol': 'Dimethylether', 'Dimethylether': 'Ethanol', 'Propanal': 'Aceton', 'Aceton': 'Propanal', 'Butan': 'Isobutan', 'Isobutan': 'Butan', 'Propansäure': 'Essigsäuremethylester', 'Essigsäuremethylester': 'Propansäure', 'D-Alanin': 'L-Alanin', 'L-Alanin': 'D-Alanin', 'Glucose offen': 'Fructose offen', 'Fructose offen': 'Glucose offen', 'Maleinsäure': 'Fumarsäure', 'Fumarsäure': 'Maleinsäure'}, 'polarity_pairs': {'Wasser vs. Schwefelwasserstoff': {'a': 'Wasser', 'b': 'Schwefelwasserstoff', 'fehlvorstellung': 'Ähnliche Summenformel und ähnliche Molekülform bedeuten ähnliche Eigenschaften.', 'leitfrage': 'Warum ist Wasser bei Raumtemperatur flüssig, Schwefelwasserstoff aber gasförmig?', 'mini_aufgaben': ['Vergleiche die Molekülform beider Moleküle.', 'Suche mögliche Wasserstoffbrücken.', 'Erkläre, warum Wasser ein starkes Netzwerk zwischen den Molekülen bilden kann.'], 'kernaussage': 'Wasser bildet starke Wasserstoffbrücken; Schwefelwasserstoff hat deutlich schwächere zwischenmolekulare Wechselwirkungen.', 'experiment': 'Wasser ist bei Raumtemperatur flüssig; Schwefelwasserstoff ist gasförmig.', 'modellgrenze': 'Für kleine anorganische Moleküle sind logP/TPSA nur eingeschränkt aussagekräftig. Hier zählt der experimentelle Bezug besonders stark.', 'id': 'wasser_vs_schwefelwasserstoff', 'typ': 'comparison', 'modul': 'Polarität und Löslichkeit', 'experimenteller_bezug': 'Wasser ist bei Raumtemperatur flüssig; Schwefelwasserstoff ist gasförmig.', 'stoffdaten_hinweis': 'Stoffdaten sind als kuratierte Unterrichtswerte bei den Einzelmolekülen hinterlegt, wo sie didaktisch besonders hilfreich sind.', 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, 'Methan vs. Hexan': {'a': 'Methan', 'b': 'Hexan', 'fehlvorstellung': 'Unpolar ist unpolar – daher müssten unpolare Stoffe ähnliche Eigenschaften besitzen.', 'leitfrage': 'Warum ist Methan gasförmig, Hexan aber flüssig, obwohl beide unpolar sind?', 'mini_aufgaben': ['Vergleiche die Molekülgröße.', 'Betrachte die unpolare Oberfläche.', 'Erkläre den Einfluss der Molekülgröße auf London-Dispersionskräfte.'], 'kernaussage': 'Auch unpolare Moleküle ziehen einander an. Größere Moleküle besitzen stärkere London-Kräfte.', 'experiment': 'Methan ist gasförmig; Hexan ist flüssig und kaum wasserlöslich.', 'modellgrenze': 'London-Kräfte werden nicht direkt berechnet, sondern nur über Größe und Oberfläche erschlossen.', 'id': 'methan_vs_hexan', 'typ': 'comparison', 'modul': 'Polarität und Löslichkeit', 'experimenteller_bezug': 'Methan ist gasförmig; Hexan ist flüssig und kaum wasserlöslich.', 'stoffdaten_hinweis': 'Stoffdaten sind als kuratierte Unterrichtswerte bei den Einzelmolekülen hinterlegt, wo sie didaktisch besonders hilfreich sind.', 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, 'Ethanol vs. 1-Hexanol': {'a': 'Ethanol', 'b': '1-Hexanol', 'fehlvorstellung': 'Alkohole sind wegen der OH-Gruppe immer gut wasserlöslich.', 'leitfrage': 'Warum ist Ethanol gut wasserlöslich, 1-Hexanol aber deutlich schlechter?', 'mini_aufgaben': ['Markiere in beiden Molekülen die OH-Gruppe.', 'Vergleiche die Größe der unpolaren Kohlenwasserstoffreste.', 'Begründe die unterschiedliche Wasserlöslichkeit.'], 'kernaussage': 'Die OH-Gruppe ermöglicht H-Brücken, aber bei längerer Kohlenstoffkette wird der unpolare Anteil immer wichtiger.', 'experiment': 'Ethanol ist mit Wasser mischbar; 1-Hexanol ist nur begrenzt wasserlöslich.', 'modellgrenze': 'Die Skala ist eine Orientierung und ersetzt keine Messung der Löslichkeit.', 'id': 'ethanol_vs_1_hexanol', 'typ': 'comparison', 'modul': 'Polarität und Löslichkeit', 'experimenteller_bezug': 'Ethanol ist mit Wasser mischbar; 1-Hexanol ist nur begrenzt wasserlöslich.', 'stoffdaten_hinweis': 'Stoffdaten sind als kuratierte Unterrichtswerte bei den Einzelmolekülen hinterlegt, wo sie didaktisch besonders hilfreich sind.', 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, 'Essigsäure vs. Palmitinsäure': {'a': 'Essigsäure', 'b': 'Palmitinsäure', 'fehlvorstellung': 'Carbonsäuren sind wegen der COOH-Gruppe grundsätzlich gut wasserlöslich.', 'leitfrage': 'Warum ist Essigsäure gut wasserlöslich, Palmitinsäure aber kaum?', 'mini_aufgaben': ['Suche die Carboxylgruppe in beiden Molekülen.', 'Vergleiche die unpolaren Molekülteile.', 'Erkläre, warum dieselbe funktionelle Gruppe nicht dieselbe Stoffeigenschaft garantiert.'], 'kernaussage': 'Eine polare Gruppe kann durch einen sehr großen unpolaren Rest in ihrer Wirkung überlagert werden.', 'experiment': 'Essigsäure ist gut wasserlöslich; Palmitinsäure ist in Wasser praktisch unlöslich.', 'modellgrenze': 'pH-Effekte, Salzbildung, Kristallstruktur und Aggregation werden nicht abgebildet.', 'id': 'essigsaeure_vs_palmitinsaeure', 'typ': 'comparison', 'modul': 'Polarität und Löslichkeit', 'experimenteller_bezug': 'Essigsäure ist gut wasserlöslich; Palmitinsäure ist in Wasser praktisch unlöslich.', 'stoffdaten_hinweis': 'Stoffdaten sind als kuratierte Unterrichtswerte bei den Einzelmolekülen hinterlegt, wo sie didaktisch besonders hilfreich sind.', 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, 'Diethylether vs. Diethylenglycol': {'a': 'Diethylether', 'b': 'Diethylenglycol', 'fehlvorstellung': 'Ein Sauerstoffatom reicht aus, um Wasserlöslichkeit zuverlässig vorherzusagen.', 'leitfrage': 'Warum ist Diethylenglycol deutlich wasserfreundlicher als Diethylether?', 'mini_aufgaben': ['Zähle die O-Atome.', 'Suche OH-Gruppen.', 'Vergleiche H-Brücken-Donoren und -Akzeptoren.'], 'kernaussage': 'Diethylether kann H-Brücken nur aufnehmen; Diethylenglycol kann durch OH-Gruppen auch spenden und besitzt eine größere polare Oberfläche.', 'experiment': 'Diethylether ist nur begrenzt wasserlöslich; Diethylenglycol ist sehr wasserfreundlich.', 'modellgrenze': 'Toxizität und experimentelle Verwendung werden hier nicht behandelt.', 'id': 'diethylether_vs_diethylenglycol', 'typ': 'comparison', 'modul': 'Polarität und Löslichkeit', 'experimenteller_bezug': 'Diethylether ist nur begrenzt wasserlöslich; Diethylenglycol ist sehr wasserfreundlich.', 'stoffdaten_hinweis': 'Stoffdaten sind als kuratierte Unterrichtswerte bei den Einzelmolekülen hinterlegt, wo sie didaktisch besonders hilfreich sind.', 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, 'Methanol vs. Methanal': {'a': 'Methanol', 'b': 'Methanal', 'fehlvorstellung': 'Wenn beide Moleküle ein O-Atom besitzen, müssen sie ähnlich mit Wasser wechselwirken.', 'leitfrage': 'Warum unterscheiden sich Methanol und Methanal trotz ähnlicher Größe in ihren Wechselwirkungen?', 'mini_aufgaben': ['Suche OH-Gruppe bzw. Carbonylgruppe.', 'Vergleiche H-Brücken-Donoren und -Akzeptoren.', 'Erkläre den Unterschied zwischen Alkohol und Aldehyd.'], 'kernaussage': 'Methanol kann H-Brücken spenden und aufnehmen; Methanal kann nur aufnehmen und ist als Carbonylverbindung anders reaktiv.', 'experiment': 'Beide sind klein und wasserfreundlich, aber chemisch deutlich verschieden.', 'modellgrenze': 'Methanal liegt in Wasser teilweise hydratisiert vor; das einfache Modell bildet dies nicht ab.', 'id': 'methanol_vs_methanal', 'typ': 'comparison', 'modul': 'Polarität und Löslichkeit', 'experimenteller_bezug': 'Beide sind klein und wasserfreundlich, aber chemisch deutlich verschieden.', 'stoffdaten_hinweis': 'Stoffdaten sind als kuratierte Unterrichtswerte bei den Einzelmolekülen hinterlegt, wo sie didaktisch besonders hilfreich sind.', 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, 'Aceton vs. Essigsäure': {'a': 'Aceton', 'b': 'Essigsäure', 'fehlvorstellung': 'Eine C=O-Gruppe bestimmt die Eigenschaften allein.', 'leitfrage': 'Warum verhalten sich Aceton und Essigsäure trotz Carbonylgruppe unterschiedlich?', 'mini_aufgaben': ['Suche die Carbonylgruppe.', 'Suche zusätzliche funktionelle Gruppen.', 'Vergleiche H-Brücken und Säureeigenschaft.'], 'kernaussage': 'Aceton ist polar aprotisch; Essigsäure besitzt zusätzlich eine OH-Gruppe, kann H-Brücken spenden und ist sauer.', 'experiment': 'Beide sind gut wasserlöslich/mischbar, aber Essigsäure zeigt Säure-Base-Verhalten und starke H-Brücken.', 'modellgrenze': 'pH-Gleichgewichte und vollständige Säuredissoziation werden nicht berechnet.', 'id': 'aceton_vs_essigsaeure', 'typ': 'comparison', 'modul': 'Polarität und Löslichkeit', 'experimenteller_bezug': 'Beide sind gut wasserlöslich/mischbar, aber Essigsäure zeigt Säure-Base-Verhalten und starke H-Brücken.', 'stoffdaten_hinweis': 'Stoffdaten sind als kuratierte Unterrichtswerte bei den Einzelmolekülen hinterlegt, wo sie didaktisch besonders hilfreich sind.', 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}}, 'future_datasets': {'basis': None, 'standard': 'molecules_v0_4_2.json', 'plus': None}}

def load_teaching_data():
    """Lädt Unterrichtsdaten aus lokaler JSON, GitHub Raw oder Fallback.

    Diese Struktur bereitet spätere alternative JSON-Datenpakete vor. Für die App
    bleiben die Oberflächenelemente bewusst schlank.
    """
    import os, urllib.request
    candidates = [LOCAL_DATA_FILE, f"/content/{LOCAL_DATA_FILE}"]
    for path in candidates:
        if os.path.exists(path):
            try:
                with open(path, "r", encoding="utf-8") as f:
                    return json.load(f)
            except Exception as exc:
                print(f"Hinweis: Lokale Daten konnten nicht geladen werden: {exc}")
    try:
        with urllib.request.urlopen(DATA_URL, timeout=8) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception as exc:
        print("Hinweis: Nutze eingebettete Fallback-Daten, da die externe JSON nicht erreichbar ist.")
        return FALLBACK_DATA

DATA = load_teaching_data()
# V0.5 review branch: mirror the four sourced source-file additions into the
# embedded fallback path. Colab's DATA_URL still points to stable main while
# this notebook is being reviewed. This patch is harmless once data are merged.
ISOMER_V05_SOURCE_UPDATES = json.loads("{\"Butan\":{\"stoffdaten\":{\"schmelzpunkt_C\":\"−138\",\"siedepunkt_C\":\"−0,5\",\"dichte_g_cm3\":null,\"wasserloeslichkeit\":\"sehr gering (~0,0061 g/100 mL bei 20 °C)\",\"aggregatzustand_raumtemperatur\":\"gasförmig\",\"bedingungen\":\"Schmelz-/Siedepunkt bei ca. 1013 hPa; Löslichkeit 20 °C\",\"quelle\":\"WHO/ILO/UNEP International Chemical Safety Card ICSC 0232, https://www.inchem.org/documents/icsc/icsc/eics0232.htm\",\"hinweis\":\"Dichte des Flüssiggases ist ohne geeignete Temperatur-/Druckangabe nicht vergleichbar.\"},\"reaktivitaet\":{\"hinweis\":\"relativ reaktionsträges, unpolares Alkan\",\"warum\":\"C–C- und C–H-σ-Bindungen; keine stark polaren Gruppen.\",\"reaktionsidee\":\"Verbrennung oder radikalische Halogenierung bei geeigneter Aktivierung.\",\"modellgrenze\":\"Die Molekülstruktur alleine liefert keine Flammen- oder Reaktionsgeschwindigkeitsprognose.\"}},\"Isobutan\":{\"stoffdaten\":{\"schmelzpunkt_C\":\"−160\",\"siedepunkt_C\":\"−12\",\"dichte_g_cm3\":null,\"wasserloeslichkeit\":\"sehr gering (ca. 49 mg/L bei 25 °C; anderer Quellen-/Messkontext)\",\"aggregatzustand_raumtemperatur\":\"gasförmig\",\"bedingungen\":\"Schmelz-/Siedepunkt bei ca. 1013 hPa; Löslichkeit ca. 25 °C\",\"quelle\":\"WHO/ILO/UNEP ICSC 0901, https://www.ilo.org/dyn/icsc/showcard.display?p_card_id=0901&p_lang=en; TCEQ Butanes DSD, https://www.tceq.texas.gov/downloads/toxicology/dsd/final/butanes.pdf/@@download/file/butanes.pdf\",\"hinweis\":\"Die beiden Löslichkeitswerte haben unterschiedliche Quellen und Temperaturen; sie begründen keinen präzisen quantitativen Unterschied.\"},\"reaktivitaet\":{\"hinweis\":\"relativ reaktionsträges, verzweigtes Alkan\",\"warum\":\"Gleiche Summenformel wie Butan, andere C-C-Verknüpfung und Molekülgestalt.\",\"reaktionsidee\":\"Verbrennung; radikalische Substitution bei geeigneter Aktivierung.\",\"modellgrenze\":\"Die unterschiedliche Siedetemperatur ist keine Aussage über relative chemische Reaktionsgeschwindigkeit.\"}},\"Propansäure\":{\"stoffdaten\":{\"schmelzpunkt_C\":\"−21\",\"siedepunkt_C\":\"141\",\"dichte_g_cm3\":null,\"wasserloeslichkeit\":\"sehr gut; mit Wasser mischbar\",\"aggregatzustand_raumtemperatur\":\"flüssig\",\"bedingungen\":\"ca. 1013 hPa; Stoffwerte um Raumtemperatur\",\"quelle\":\"WHO/ILO/UNEP ICSC 0806, https://www.inchem.org/documents/icsc/icsc/eics0806.htm; USDA Propionic acid report, https://www.ams.usda.gov/sites/default/files/media/Prop%20acid%20report.pdf\",\"hinweis\":null},\"reaktivitaet\":{\"hinweis\":\"Carbonsäure; schwach sauer\",\"warum\":\"Carboxylgruppe mit abspaltbarem O–H-Proton; Carboxylat ist mesomeriestabilisiert.\",\"reaktionsidee\":\"Säure-Base-Reaktion mit Basen, Esterbildung unter geeigneten Bedingungen.\",\"modellgrenze\":\"Säurestärke und Veresterung hängen von Lösungsmittel und Reaktionsbedingungen ab.\"}},\"Essigsäuremethylester\":{\"stoffdaten\":{\"schmelzpunkt_C\":\"−98\",\"siedepunkt_C\":\"57\",\"dichte_g_cm3\":null,\"wasserloeslichkeit\":\"24,4 g/100 mL Wasser bei 20 °C (begrenzt mischbar)\",\"aggregatzustand_raumtemperatur\":\"flüssig\",\"bedingungen\":\"Siedepunkt ca. 1013 hPa; Löslichkeit bei 20 °C\",\"quelle\":\"WHO/ILO/UNEP International Chemical Safety Card ICSC 0507, https://inchem.org/documents/icsc/icsc/eics0507.htm\",\"hinweis\":null},\"reaktivitaet\":{\"hinweis\":\"Ester; keine freie Carboxyl-O–H-Gruppe\",\"warum\":\"Carbonyl-C ist über O an einen weiteren C-Rest gebunden.\",\"reaktionsidee\":\"Esterhydrolyse unter sauren oder basischen Bedingungen.\",\"modellgrenze\":\"Esterhydrolyse und Gleichgewicht erfordern Reaktionsbedingungen; nicht allein aus einer Strukturansicht ablesbar.\"}}}")
for _isocard in DATA.get("modules", {}).get("Isomerie", []):
    _update = ISOMER_V05_SOURCE_UPDATES.get(_isocard.get("name"))
    if _update:
        _isocard.setdefault("stoffdaten", {}).update(_update["stoffdaten"])
        _isocard.setdefault("reaktivitaet", {}).update(_update["reaktivitaet"])
# Vergleichs-MC: zusätzlich zu den Einzelmolekül-Fragen; auch bei Fallback-Daten verfügbar.
PAIR_QUIZZES = {"Wasser vs. Schwefelwasserstoff": "{\"frage\":\"Welche Wechselwirkung erklärt den großen Unterschied zwischen Wasser und Schwefelwasserstoff?\",\"antworten\":[\"Beide Stoffe bilden gleich starke Wasserstoffbrücken.\",\"Wasser bildet ein ausgeprägtes Netzwerk aus Wasserstoffbrücken, Schwefelwasserstoff nicht.\",\"Wasser ist aus Ionen aufgebaut, Schwefelwasserstoff nicht.\",\"Die Moleküle haben dieselbe molare Masse.\"],\"richtig\":1,\"feedback\":\"Die starken Wasserstoffbrücken zwischen H₂O-Molekülen erhöhen die Siedetemperatur deutlich.\",\"erklaerung\":\"Die starken Wasserstoffbrücken zwischen H₂O-Molekülen erhöhen die Siedetemperatur deutlich.\"}", "Methan vs. Hexan": "{\"frage\":\"Warum hat Hexan bei Normaldruck einen viel höheren Siedepunkt als Methan?\",\"antworten\":[\"Weil Hexan ionisch und Methan kovalent ist.\",\"Weil Hexan Wasserstoffbrücken ausbildet.\",\"Weil bei den größeren Hexanmolekülen die London-Dispersionskräfte stärker sind.\",\"Weil Methan nicht zwischenmolekular wechselwirkt.\"],\"richtig\":2,\"feedback\":\"Auch unpolare Moleküle wechselwirken über London-Kräfte; die Stärke nimmt bei größeren, leichter polarisierbaren Molekülen zu.\",\"erklaerung\":\"Auch unpolare Moleküle wechselwirken über London-Kräfte; die Stärke nimmt bei größeren, leichter polarisierbaren Molekülen zu.\"}", "Ethanol vs. 1-Hexanol": "{\"frage\":\"Warum ist 1-Hexanol in Wasser deutlich schlechter löslich als Ethanol?\",\"antworten\":[\"Der längere unpolare Kohlenwasserstoffrest gewinnt bei 1-Hexanol stärker an Bedeutung.\",\"1-Hexanol besitzt keine OH-Gruppe mehr.\",\"Ethanol ist ionisch, 1-Hexanol kovalent.\",\"Die C–C-Bindungen von Ethanol bilden Wasserstoffbrücken.\"],\"richtig\":0,\"feedback\":\"Beide Alkohole besitzen eine OH-Gruppe, aber der größere hydrophobe Molekülteil verringert die Wasserlöslichkeit.\",\"erklaerung\":\"Beide Alkohole besitzen eine OH-Gruppe, aber der größere hydrophobe Molekülteil verringert die Wasserlöslichkeit.\"}", "Essigsäure vs. Palmitinsäure": "{\"frage\":\"Warum löst sich Palmitinsäure trotz Carboxylgruppe nur sehr schlecht in Wasser?\",\"antworten\":[\"Carboxylgruppen sind grundsätzlich unpolar.\",\"Essigsäure hat keine Kohlenstoffatome.\",\"Palmitinsäure kann keine Säure-Base-Reaktionen eingehen.\",\"Die lange unpolare Kette dominiert die Eigenschaften des Moleküls.\"],\"richtig\":3,\"feedback\":\"Die lange Alkylkette begrenzt die Wasserlöslichkeit, obwohl die Carboxylgruppe polar und ionisierbar ist.\",\"erklaerung\":\"Die lange Alkylkette begrenzt die Wasserlöslichkeit, obwohl die Carboxylgruppe polar und ionisierbar ist.\"}", "Diethylether vs. Diethylenglycol": "{\"frage\":\"Welche Strukturänderung erklärt die größere Wasserfreundlichkeit des Diethylenglycols?\",\"antworten\":[\"Diethylenglycol ist ein Salz.\",\"Diethylenglycol besitzt zwei OH-Gruppen und kann H-Brücken spenden und aufnehmen.\",\"Diethylether enthält gar keinen Sauerstoff.\",\"Diethylenglycol kann keine Wasserstoffbrücken ausbilden.\"],\"richtig\":1,\"feedback\":\"Die zusätzlichen Hydroxygruppen erhöhen die Möglichkeiten für Wasserstoffbrücken mit Wasser.\",\"erklaerung\":\"Die zusätzlichen Hydroxygruppen erhöhen die Möglichkeiten für Wasserstoffbrücken mit Wasser.\"}", "Methanol vs. Methanal": "{\"frage\":\"Welche Aussage über die Wasserstoffbrücken-Fähigkeit der beiden Moleküle ist korrekt?\",\"antworten\":[\"Beide besitzen eine OH-Gruppe.\",\"Methanal spendet stärkere H-Brücken als Methanol.\",\"Methanol kann H-Brücken spenden und aufnehmen; Methanal besitzt einen Carbonylsauerstoff als Akzeptor.\",\"Methanol enthält keinen Sauerstoff.\"],\"richtig\":2,\"feedback\":\"Die OH-Gruppe ist Donor und Akzeptor, während die Carbonylgruppe vor allem als Akzeptor wirkt.\",\"erklaerung\":\"Die OH-Gruppe ist Donor und Akzeptor, während die Carbonylgruppe vor allem als Akzeptor wirkt.\"}", "Aceton vs. Essigsäure": "{\"frage\":\"Warum ist Essigsäure eine schwache Säure, Aceton dagegen nicht in vergleichbarer Weise?\",\"antworten\":[\"Aceton enthält keine polare Bindung.\",\"Essigsäure besitzt eine Carboxylgruppe mit abspaltbarem Proton; Aceton ist ein Keton.\",\"Essigsäure ist ionisch und Aceton ein Alkan.\",\"Beide haben dieselbe funktionelle Gruppe.\"],\"richtig\":1,\"feedback\":\"Die Carboxylgruppe enthält ein saures O–H, dessen konjugierte Base mesomeriestabilisiert ist.\",\"erklaerung\":\"Die Carboxylgruppe enthält ein saures O–H, dessen konjugierte Base mesomeriestabilisiert ist.\"}"}
for _pair_name, _quiz_json in PAIR_QUIZZES.items():
    _pair = (DATA.get("polarity_pairs") or {}).get(_pair_name)
    if _pair is not None and not _pair.get("quiz"):
        _pair["quiz"] = json.loads(_quiz_json)

if "modules" in DATA:
    MOLECULES = DATA["modules"]
    ISOMER_PAIRS = DATA.get("isomer_pairs", {})
    POLARITY_PAIRS = DATA.get("polarity_pairs", {})
else:
    # Abwärtskompatibilität zu älteren JSON-Dateien.
    MOLECULES = {k:v for k,v in DATA.items() if isinstance(v, list)}
    ISOMER_PAIRS = {}
    POLARITY_PAIRS = {}


FUNCTIONAL_GROUPS = {
    "Alkohol": "[OX2H][CX4]",
    "Phenol": "[OX2H]c",
    "Aldehyd": "[CX3;H1,H2](=[OX1])",
    "Keton": "[#6][CX3](=O)[#6]",
    "Carboxylgruppe": "C(=O)[OX2H1]",
    "Ester": "[CX3](=[OX1])[OX2][#6;!$(C=O)]",
    "Ether": "[#6;!$(C=O)][OD2][#6;!$(C=O)]",
    "Carbonsäureanhydrid": "C(=O)OC(=O)",
    "Amin": "[NX3;H2,H1,H0;!$(NC=O)]",
    "Amid": "C(=O)N",
    "Sulfonamid": "[SX4](=[OX1])(=[OX1])[NX3]",
    "Arylfluorid": "[F][c]",
    "Aromatischer Ring": "a",
    "Carbonylgruppe": "[#6]=[OX1]",
}

ISOMER_QUIZZES = {
    "Ethanol": {
        "frage": "Wodurch unterscheidet sich Ethanol am wichtigsten von Dimethylether?",
        "antworten": [
            "Ethanol kann wegen seiner O–H-Bindung untereinander Wasserstoffbrücken spenden; Dimethylether nicht.",
            "Beide besitzen gleichartige O–H-Bindungen und müssen gleich siedend sein.",
            "Nur Dimethylether enthält Sauerstoff.",
            "Die Summenformel bestimmt allein die Siedetemperatur."
        ],
        "richtig": 0,
        "feedback": "Beide haben C₂H₆O; die OH-Gruppe bei Ethanol ermöglicht starke intermolekulare Wasserstoffbrücken."
    },
    "Propanal": {
        "frage": "Welche Aussage erklärt den wichtigsten Unterschied zwischen Propanal und Aceton?",
        "antworten": [
            "Aceton besitzt eine Aldehydgruppe, Propanal eine Ketogruppe.",
            "Propanal trägt am Carbonyl-C ein H-Atom und ist ein Aldehyd; Aceton trägt dort zwei C-Reste und ist ein Keton.",
            "Nur Propanal besitzt eine C=O-Doppelbindung.",
            "Da beide C₃H₆O haben, müssen sie identisch reagieren."
        ],
        "richtig": 1,
        "feedback": "Die unterschiedliche Bindungsumgebung am Carbonylkohlenstoff führt u. a. zu unterschiedlicher Oxidierbarkeit."
    },
    "Butan": {
        "frage": "Warum besitzen Butan und Isobutan unterschiedliche Siedepunkte?",
        "antworten": [
            "Isobutan ist ionisch, Butan kovalent.",
            "Nur Butan kann Wasserstoffbrücken ausbilden.",
            "Die Verzweigung ändert die Molekülgestalt und die London-Wechselwirkungen trotz gleicher Summenformel.",
            "Der Unterschied entsteht durch eine zusätzliche Doppelbindung in Isobutan."
        ],
        "richtig": 2,
        "feedback": "Beide sind unpolare Alkane mit C₄H₁₀; die Gestalt verändert die effektive Kontaktfläche und damit die Dispersionswechselwirkungen."
    },
    "Propansäure": {
        "frage": "Welcher Strukturunterschied trennt Propansäure von Essigsäuremethylester?",
        "antworten": [
            "Beide besitzen eine freie Carboxyl-O–H-Bindung.",
            "Propansäure ist ein Ester, Essigsäuremethylester eine Carbonsäure.",
            "Der Ester besitzt eine C=C-Doppelbindung, die Säure nicht.",
            "Propansäure trägt eine Carboxylgruppe mit O–H; der Ester besitzt eine C(=O)–O–C-Verknüpfung."
        ],
        "richtig": 3,
        "feedback": "Die Carbonsäure kann ein Carboxylproton abgeben; der Ester besitzt keine freie Carboxyl-OH-Gruppe."
    },
    "D-Alanin": {
        "frage": "Wie unterscheiden sich D-Alanin und L-Alanin bei gleicher Konnektivität?",
        "antworten": [
            "Sie sind nicht deckungsgleiche Spiegelbilder; in achiralen Umgebungen können viele Stoffwerte gleich sein, biologische Wechselwirkungen aber verschieden.",
            "Sie haben verschiedene Summenformeln.",
            "Sie sind nur unterschiedlich um eine gewöhnliche C–C-Einfachbindung gedreht.",
            "Die eine Form hat kein Stickstoffatom."
        ],
        "richtig": 0,
        "feedback": "Das chirale Zentrum hat entgegengesetzte Konfiguration. Eine Rotation führt nicht vom einen Enantiomer zum anderen."
    },
    "Glucose offen": {
        "frage": "Was ist der wesentliche Unterschied der hier gezeigten offenen Glucose- und Fructosemodelle?",
        "antworten": [
            "Beide besitzen nur Kohlenstoff-Kohlenstoff-Bindungen.",
            "Offenkettige Glucose ist ein Aldehyd-Zucker, offenkettige Fructose ein Keton-Zucker; in Wasser liegen überwiegend Ringformen vor.",
            "Nur Fructose besitzt Hydroxygruppen.",
            "Sie sind Enantiomere mit identischer Bindungsverknüpfung."
        ],
        "richtig": 1,
        "feedback": "Die Carbonylgruppe ist unterschiedlich angeordnet. Die offenen Darstellungen sind vereinfachte Strukturmodelle, nicht die alleinigen dominierenden Spezies in Wasser."
    },
    "Maleinsäure": {
        "frage": "Welche Aussage beschreibt Maleinsäure und Fumarsäure korrekt?",
        "antworten": [
            "Beide unterscheiden sich nur durch eine frei drehbare C–C-Einfachbindung.",
            "Fumarsäure hat eine zusätzliche Carboxylgruppe.",
            "Die Anordnung der Carboxylgruppen an der C=C-Bindung ist cis (Maleinsäure) bzw. trans (Fumarsäure); dies beeinflusst Eigenschaften und Kristallpackung.",
            "Die Moleküle besitzen unterschiedliche molare Massen."
        ],
        "richtig": 2,
        "feedback": "Bei einer C=C-Doppelbindung ist die Konfiguration nicht durch einfache Rotation ineinander überführbar."
    }
}

current_report = {}

GLUCOSE_FORMS = {
    # Stereochemically specified D-glucose structures.
    # CID 107526: open-chain aldehydo-D-glucose; CID 79025 alpha,
    # CID 64689 beta-D-glucopyranose.
    # Highlight indices are verified against the specific SMILES and tested.
    "open": {
        "name": "Offenkettige D-Glucose",
        "smiles": "OC[C@@H](O)[C@@H](O)[C@H](O)[C@@H](O)C=O",
        "focus": [3, 10, 11],  # C5–OH oxygen, aldehyde C1 and O
        "quelle": "PubChem CID 107526",
        "beschreibung": (
            "<b>Offene Kette:</b> C1 ist eine planare Aldehydgruppe; "
            "das Sauerstoffatom der OH-Gruppe an C5 kann C1 intramolekular "
            "angreifen. C2 bis C5 besitzen festgelegte D-Glucose-Konfiguration."
        )
    },
    "alpha": {
        "name": "α-D-Glucopyranose",
        "smiles": "C([C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)O)O)O)O)O",
        "focus": [5, 6, 7],  # new anomeric C1, ring O5, C1 OH
        "quelle": "PubChem CID 79025",
        "beschreibung": (
            "<b>α-Anomer:</b> Der C5-Sauerstoff ist nun Ringatom, "
            "die C1=O-Doppelbindung wurde zur C1–OH-Gruppe. "
            "C1 ist ein neues stereogenes Zentrum. In üblicher Haworth-"
            "Projektion liegt die anomere OH-Gruppe bei α-D-Glucose "
            "auf der Gegenseite zur CH₂OH-Gruppe."
        )
    },
    "beta": {
        "name": "β-D-Glucopyranose",
        "smiles": "C([C@@H]1[C@H]([C@@H]([C@H]([C@@H](O1)O)O)O)O)O",
        "focus": [5, 6, 7],
        "quelle": "PubChem CID 64689",
        "beschreibung": (
            "<b>β-Anomer:</b> Gleiches Sechsring-Gerüst mit fünf C- und "
            "einem O-Atom; nur die Konfiguration am anomeren C1 ist "
            "gegenüber α verschieden. In der üblichen Haworth-Darstellung "
            "liegen C1–OH und C5–CH₂OH auf derselben Ringseite."
        )
    }
}

GLUCOSE_QUIZ = {
    "frage": "Warum gibt es α- und β-D-Glucopyranose, und wie können sie ineinander übergehen?",
    "antworten": [
        "Die beiden Formen unterscheiden sich durch freie Rotation um eine C–C-Einfachbindung.",
        "Die Ringbildung erzeugt am zuvor planaren Carbonyl-C1 ein neues Stereozentrum; in Wasser ist die Umwandlung über die offene Kette möglich.",
        "α- und β-D-Glucopyranose besitzen unterschiedliche Summenformeln.",
        "Mutarotation ist ausschließlich eine Reaktion mit Sauerstoff aus der Luft."
    ],
    "richtig": 1,
    "feedback": (
        "Der intramolekulare Angriff von C5–OH auf das Aldehyd-C1 "
        "erzeugt ein Halbacetal. Das anomere Zentrum C1 kann "
        "in zwei Konfigurationen entstehen; über die offene Form "
        "können sich die Anomere austauschen."
    )
}


def mol_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Ungültiger SMILES-Code.")
    return mol

def make_3d(mol):
    mol_h = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    status = AllChem.EmbedMolecule(mol_h, params)
    if status != 0:
        AllChem.EmbedMolecule(mol_h, randomSeed=42)
    try:
        AllChem.MMFFOptimizeMolecule(mol_h, maxIters=300)
    except Exception:
        try:
            AllChem.UFFOptimizeMolecule(mol_h, maxIters=300)
        except Exception:
            pass
    return mol_h

def calc_properties(mol):
    return {
        "Summenformel": rdMolDescriptors.CalcMolFormula(mol),
        "Molmasse / g mol⁻¹": round(Descriptors.MolWt(mol), 2),
        "H-Brücken-Donoren": rdMolDescriptors.CalcNumHBD(mol),
        "H-Brücken-Akzeptoren": rdMolDescriptors.CalcNumHBA(mol),
        "logP (Modellwert)": round(Crippen.MolLogP(mol), 2),
        "TPSA / Å²": round(rdMolDescriptors.CalcTPSA(mol), 2),
        "rotierbare Bindungen": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "Ringanzahl": rdMolDescriptors.CalcNumRings(mol),
        "schwere Atome": mol.GetNumHeavyAtoms(),
    }

def _free_carbonyl_matches(mol):
    """C=O fragments that are not part of a higher-specificity acyl group."""
    carbonyl = Chem.MolFromSmarts(FUNCTIONAL_GROUPS["Carbonylgruppe"])
    excluded = set()
    for group in ("Carboxylgruppe", "Ester", "Amid", "Carbonsäureanhydrid"):
        patt = Chem.MolFromSmarts(FUNCTIONAL_GROUPS[group])
        if patt is not None:
            for match in mol.GetSubstructMatches(patt):
                excluded.update(match)
    return [match for match in mol.GetSubstructMatches(carbonyl)
            if match[0] not in excluded]


def _functional_group_highlight_atoms(mol, group):
    """Return the selected motif's atom and bond indices for RDKit highlighting."""
    if not group or group not in FUNCTIONAL_GROUPS:
        return [], []
    if group == "Aromatischer Ring":
        atoms = [a.GetIdx() for a in mol.GetAtoms() if a.GetIsAromatic()]
        bonds = [b.GetIdx() for b in mol.GetBonds()
                 if b.GetIsAromatic() and b.IsInRing()]
        return atoms, bonds
    patt = Chem.MolFromSmarts(FUNCTIONAL_GROUPS[group])
    if patt is None:
        return [], []
    matches = (_free_carbonyl_matches(mol) if group == "Carbonylgruppe"
               else mol.GetSubstructMatches(patt, uniquify=True))
    atoms, bonds = set(), set()
    for match in matches:
        atoms.update(match)
        for pb in patt.GetBonds():
            bond = mol.GetBondBetweenAtoms(match[pb.GetBeginAtomIdx()],
                                           match[pb.GetEndAtomIdx()])
            if bond is not None:
                bonds.add(bond.GetIdx())
    return sorted(atoms), sorted(bonds)


def detect_functional_groups(mol):
    """Prioritize named complete acyl groups over redundant carbonyl fragments."""
    return sorted(name for name in FUNCTIONAL_GROUPS
                  if _functional_group_highlight_atoms(mol, name)[0])


def _functional_group_display_label(mol, group):
    """Distinguish aromatic fused systems and unassigned ring carbonyl groups."""
    if group == "Aromatischer Ring":
        aromatic_rings = [
            set(ring) for ring in mol.GetRingInfo().AtomRings()
            if all(mol.GetAtomWithIdx(i).GetIsAromatic() for i in ring)
        ]
        fused = any(len(left & right) >= 2
                    for i,left in enumerate(aromatic_rings)
                    for right in aromatic_rings[i+1:])
        return ("Aromatisches Ringsystem (RDKit)" if fused
                else "Aromatischer Ring (RDKit)")
    if group == "Carbonylgruppe":
        matches = _free_carbonyl_matches(mol)
        if len(matches) >= 2 and all(mol.GetAtomWithIdx(m[0]).GetIsAromatic()
                                     for m in matches):
            return "Ring-Carbonylgruppen (C=O)"
        return "Carbonylgruppen (C=O)" if len(matches) > 1 else "Carbonylgruppe (C=O)"
    return group


def mol_png_data_uri(mol, width=340, height=260, highlight_group=None, highlight_atoms=None, highlight_bonds=None):
    """RDKit drawing. Highlight only the chosen functional group, never infer ESP."""
    atoms, bonds = _functional_group_highlight_atoms(mol, highlight_group)
    atoms = sorted(set(atoms) | set(highlight_atoms or []))
    bonds = sorted(set(bonds) | set(highlight_bonds or []))
    img = Draw.MolToImage(mol, size=(width, height),
                          highlightAtoms=atoms, highlightBonds=bonds)
    buf = BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/png;base64,{b64}"


def functional_group_highlight_widget(mol, groups=None):
    """Colab-safe switch for highlighting one SMARTS match in the 2D structure."""
    groups = list(groups if groups is not None else detect_functional_groups(mol))
    # Readable structural formulas without changing the UI of the usual small molecules.
    large = mol.GetNumHeavyAtoms() >= 38
    image_w, image_h = (490, 340) if large else (340, 260)
    widget_w = "510px" if large else "365px"
    view_w = "500px" if large else "350px"
    chooser = widgets.Dropdown(
        options=[("— Gruppe hervorheben —", None)] + [(_functional_group_display_label(mol, g), g) for g in groups],
        value=None, description="Gruppe:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="min(100%, 500px)" if large else "min(100%, 345px)")
    )
    def image_for(group):
        uri = mol_png_data_uri(mol, image_w, image_h, highlight_group=group)
        return ("<img alt='2D-Struktur mit ausgewählter funktioneller Gruppe'"
                f" src='{uri}' style='max-width:{image_w}px;width:100%;height:auto;' />")
    view = widgets.HTML(value=image_for(None), layout=widgets.Layout(width=view_w))
    def refresh(change):
        if change["name"] == "value":
            view.value = image_for(change["new"])
    chooser.observe(refresh, names="value")
    return widgets.VBox([
        widgets.HTML("<h3 style='margin:0 0 6px;'>2D-Struktur – Gruppen markieren</h3>"),
        chooser, view,
        widgets.HTML(
            "<p style='font-size:0.8rem;color:#506679;'>Markierung anhand "
            "kuratierter SMARTS-Strukturmuster. Carbonylgruppen von "
            "Carboxyl-, Ester-, Amid- und Anhydridgruppen werden nicht "
            "doppelt gelistet. Aromatizität folgt dem RDKit-Ringmodell; "
            "C=O-Gruppen können daneben erhalten bleiben.</p>")
    ], layout=widgets.Layout(width=widget_w, max_width="100%"))


def show_2d(mol, width=360, height=260):
    display(HTML(f"<img src='{mol_png_data_uri(mol, width, height)}' style='max-width:100%;'>"))

def _style_for_3d(style_name):
    if style_name == "Kalottenmodell":
        return {"sphere": {"scale": 1.0}}
    if style_name == "Stabmodell":
        return {"stick": {"radius": 0.16}}
    # Default: Kugel-Stab
    return {"stick": {"radius": 0.12}, "sphere": {"scale": 0.25}}

def _style_js_for_3d(style_name):
    if style_name == "Kalottenmodell":
        return '{sphere:{scale:1.0}}'
    if style_name == "Stabmodell":
        return '{stick:{radius:0.16}}'
    if style_name == "Didaktische Polaritätsoberfläche":
        # Die Oberfläche selbst wird unten zusätzlich erzeugt.
        return '{stick:{radius:0.08}, sphere:{scale:0.18}}'
    return '{stick:{radius:0.12}, sphere:{scale:0.25}}'

def _is_polar_surface(style_name):
    return style_name == "Didaktische Polaritätsoberfläche"

def _custom_3dmol_iframe_html(molblock, width=520, height=380, style="Kugel-Stab"):
    # Schlanke 3Dmol.js-Ausgabe ohne py3Dmol-Wrapper.
    # v0.2: optional mit didaktischer Polaritätsoberfläche.
    div_id = "mol3d_" + uuid.uuid4().hex
    mol_js = json.dumps(molblock)
    style_js = _style_js_for_3d(style)
    add_surface_js = ""
    if _is_polar_surface(style):
        add_surface_js = """
      try {
        // Didaktische Oberfläche: elementbasierte VDW-Oberfläche, keine exakte ESP-Fläche.
        viewer.addSurface($3Dmol.SurfaceType.VDW, {opacity:0.72, colorscheme:'Jmol'}, {});
      } catch(surfaceErr) {
        console.log('Oberfläche konnte nicht erzeugt werden', surfaceErr);
      }
        """
    inner = f'''
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin:0; padding:0; width:100%; height:100%; overflow:hidden; background:white; }}
  #{div_id} {{ width:100%; height:100%; }}
  .msg {{ font-family:Arial, sans-serif; font-size:12px; padding:8px; color:#666; }}
</style>
<script src="https://cdn.jsdelivr.net/npm/3dmol@2.5.5/build/3Dmol-min.js"></script>
</head>
<body>
<div id="{div_id}"><div class="msg">3D-Modell wird geladen …</div></div>
<script>
(function() {{
  const molblock = {mol_js};
  const styleObj = {style_js};
  function init(attempt) {{
    const el = document.getElementById('{div_id}');
    if (!el) return;
    if (typeof $3Dmol === 'undefined') {{
      if (attempt < 20) {{ window.setTimeout(function() {{ init(attempt+1); }}, 150); }}
      else {{ el.innerHTML = '<div class="msg">3Dmol.js konnte nicht geladen werden. Bitte erneut analysieren oder 3D-Engine wechseln.</div>'; }}
      return;
    }}
    el.innerHTML = '';
    try {{
      const viewer = $3Dmol.createViewer(el, {{backgroundColor:'white'}});
      viewer.addModel(molblock, 'sdf');
      viewer.setStyle({{}}, styleObj);
      {add_surface_js}
      viewer.zoomTo();
      viewer.render();
      window.addEventListener('resize', function() {{ viewer.resize(); viewer.render(); }});
    }} catch(e) {{
      el.innerHTML = '<div class="msg">3D-Anzeige fehlgeschlagen: ' + e + '</div>';
    }}
  }}
  init(0);
}})();
</script>
</body>
</html>
'''
    escaped = html.escape(inner, quote=True)
    return f'<iframe class="mol-lab-3d-frame" srcdoc="{escaped}" width="{width}" height="{height}" loading="eager" style="border:1px solid #ddd; border-radius:8px; background:white; max-width:100%;"></iframe>'

def _py3dmol_iframe_html(viewer, width=520, height=380):
    # Fallback-Ausgabe mit py3Dmol-eigenem HTML im iframe.
    try:
        inner = viewer._make_html()
        escaped = html.escape(inner, quote=True)
        return f'<iframe class="mol-lab-3d-frame" srcdoc="{escaped}" width="{width}" height="{height}" loading="eager" style="border:1px solid #ddd; border-radius:8px; background:white; max-width:100%;"></iframe>'
    except Exception as e:
        return f"<p style='color:#b00020'>3D-Anzeige konnte nicht erzeugt werden: {html.escape(str(e))}</p>"

def _viewer_for_molblock(molblock, width=520, height=380, style="Kugel-Stab"):
    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(molblock, "sdf")
    viewer.setStyle(_style_for_3d(style))
    if _is_polar_surface(style):
        try:
            viewer.addSurface(py3Dmol.VDW, {"opacity":0.72, "colorscheme":"Jmol"})
        except Exception:
            pass
    viewer.setBackgroundColor("white")
    viewer.zoomTo()
    return viewer

def molblock_3d_html(molblock, width=520, height=380, style="Kugel-Stab", mode="custom"):
    if mode == "py3dmol":
        viewer = _viewer_for_molblock(molblock, width=width, height=height, style=style)
        return _py3dmol_iframe_html(viewer, width=width+20, height=height+30)
    return _custom_3dmol_iframe_html(molblock, width=width+20, height=height+30, style=style)

def mol3d_iframe_html(mol3d, width=520, height=380, style="Kugel-Stab", mode="custom"):
    mb = Chem.MolToMolBlock(mol3d)
    return molblock_3d_html(mb, width=width, height=height, style=style, mode=mode)

def show_3d(mol3d, width=520, height=380, style="Kugel-Stab", mode="custom"):
    display(HTML(mol3d_iframe_html(mol3d, width=width, height=height, style=style, mode=mode)))



# V0.9a: isolated, curated dipole-learning viewer. The normal analysis
# and pair/conformer viewers keep their existing rendering paths.
DIPOLE_EN_PAULING = {"H": 2.20, "C": 2.55, "N": 3.04, "O": 3.44}
DIPOLE_SETS = {
    "basis": {"title": "Geometrie und Symmetrie",
              "members": ["co2", "h2o", "nh3", "ch4"]}
}
DIPOLE_CASES = {
    "co2": {"name": "Kohlenstoffdioxid", "smiles": "O=C=O", "ready": True},
    "h2o": {"name": "Wasser", "smiles": "O", "ready": True},
    "nh3": {"name": "Ammoniak", "smiles": "N", "ready": True},
    "ch4": {"name": "Methan", "smiles": "C", "ready": True}
}

# NIST CCCBDB, gas-phase permanent dipole moments. Reference magnitudes
# are deliberately separate from the unit-length EN vector model.
DIPOLE_REFERENCE = {
    "co2": {"value_D": 0.0, "display": "0 D", "kind": "Symmetrie-Referenz",
            "source": "NIST CCCBDB · NSRDS-NBS10",
            "note": "Im linearen CO₂ (D∞h) verschwindet das permanente Dipolmoment aus Symmetriegründen."},
    "h2o": {"value_D": 1.85498, "display": "≈ 1,855 D", "kind": "Experiment · μ₀",
            "source": "NIST CCCBDB · Shostak, Ebenstein, Muenter (1991)",
            "note": "Gasphase, Schwingungsgrundzustand (μ₀); Mikrowellenspektroskopie."},
    "nh3": {"value_D": 1.476, "display": "≈ 1,476 D", "kind": "Experiment · μ₀",
            "source": "NIST CCCBDB · Shimizu (1970)",
            "note": "Gasphase; spektroskopischer Stark-Effekt. NIST führt 1,476 D (± 0,0029 D)."},
    "ch4": {"value_D": 0.0, "display": "0 D", "kind": "Symmetrie-Referenz",
            "source": "NIST CCCBDB · NSRDS-NBS10",
            "note": "Im ideal tetraedrischen CH₄ (Td) verschwindet das permanente Dipolmoment aus Symmetriegründen."}
}

def dipole_case_id(module, card, mol, custom=False):
    """Only ready curated single molecules, not pairs or arbitrary SMILES."""
    if module != "Polarität und Löslichkeit" or custom:
        return None
    canonical = Chem.MolToSmiles(Chem.RemoveHs(mol))
    for key, case in DIPOLE_CASES.items():
        if case["ready"] and card.get("name") == case["name"] and canonical == case["smiles"]:
            return key
    return None

def _dip_vadd(a, b):
    return tuple(x+y for x,y in zip(a,b))

def _dip_vsub(a, b):
    return tuple(x-y for x,y in zip(a,b))

def _dip_vscale(a, k):
    return tuple(x*k for x in a)

def _dip_dot(a, b):
    return sum(x*y for x,y in zip(a,b))

def _dip_unit(a):
    length = math.sqrt(_dip_dot(a,a))
    if length < 1e-8 or not math.isfinite(length):
        raise ValueError("Keine eindeutige Vektorrichtung.")
    return _dip_vscale(a, 1.0/length)

def _dip_xyz(v):
    return dict(zip(("x","y","z"), v))

def _dip_perp(u):
    """Deterministic perpendicular even for linear and symmetric molecules."""
    axis = min(((1.,0.,0.),(0.,1.,0.),(0.,0.,1.)),key=lambda a:abs(_dip_dot(a,u)))
    return _dip_unit(_dip_vsub(axis,_dip_vscale(u,_dip_dot(axis,u))))

def _dip_resultant_anchor(center, total, coords, vectors):
    """Sideways placement in molecule coordinates, not a screen overlay.
    Keep the entire arrow outside the atomic span along a transverse axis.
    This also works for a future nonzero NH3 resultant."""
    if math.sqrt(_dip_dot(total,total)) < 1e-8:
        return center
    u = _dip_unit(total)
    # Prefer the plane spanned by the first nonparallel bond and the sum.
    # This preserves the intuitive frontal view of water.
    side = None
    for v in vectors:
        transverse = _dip_vsub(v,_dip_vscale(u,_dip_dot(v,u)))
        if _dip_dot(transverse,transverse)>1e-10:
            side = _dip_unit(transverse)
            break
    if side is None:
        side = _dip_perp(u)
    extent = max(_dip_dot(_dip_vsub(pos,center),side) for pos in coords)
    return _dip_vadd(center,_dip_vscale(side,max(0.,extent)+.60))

def _dip_cross(a,b):
    return (a[1]*b[2]-a[2]*b[1],a[2]*b[0]-a[0]*b[2],a[0]*b[1]-a[1]*b[0])

def _dip_symmetric_hydrides(coords, atoms, case_id):
    """Symmetric teaching geometry, preserving approximate RDKit orientation
    and mean bond length. NH3 keeps its mean axial projection, CH4 exact Td."""
    central = "N" if case_id=="nh3" else "C"
    ci = next(a.GetIdx() for a in atoms if a.GetSymbol()==central)
    hi = [a.GetIdx() for a in atoms if a.GetSymbol()=="H"]
    c = coords[ci]
    raw = [_dip_vsub(coords[i],c) for i in hi]
    length = sum(math.sqrt(_dip_dot(v,v)) for v in raw)/len(raw)
    dirs = [_dip_unit(v) for v in raw]
    if case_id=="nh3":
        z = _dip_unit(tuple(sum(v[k] for v in dirs) for k in range(3)))
        axial = sum(_dip_dot(v,z) for v in dirs)/3
        ring = dirs
        lead = []
    else:
        z = dirs[0]
        axial = -1./3.
        ring = dirs[1:]
        lead = [z]
    x = _dip_unit(_dip_vsub(ring[0],_dip_vscale(z,_dip_dot(ring[0],z))))
    y = _dip_unit(_dip_cross(z,x))
    if _dip_dot(ring[1],y)<0: y = _dip_vscale(y,-1.)
    radial = math.sqrt(max(0.,1.-axial*axial))
    directions = lead + [_dip_vadd(_dip_vscale(z,axial),
        _dip_vscale(_dip_vadd(_dip_vscale(x,math.cos(t)),_dip_vscale(y,math.sin(t))),radial))
        for t in (0.,2*math.pi/3,4*math.pi/3)]
    result = list(coords)
    for i,v in zip(hi,directions):
        result[i]=_dip_vadd(c,_dip_vscale(v,length))
    return result,z

def dipole_learning_payload(mol3d, case_id):
    """Model directions from EN and actual displayed coordinates; no Debye."""
    case = DIPOLE_CASES.get(case_id)
    if not case or not case["ready"]:
        raise ValueError("Dieses Dipol-Beispiel ist noch nicht freigegeben.")
    if (Chem.MolToSmiles(Chem.RemoveHs(mol3d)) != case["smiles"]
            or Chem.GetFormalCharge(mol3d) != 0
            or not mol3d.GetNumConformers()):
        raise ValueError("Das 3D-Modell passt nicht zum kuratierten Beispiel.")
    conf = mol3d.GetConformer()
    atoms = list(mol3d.GetAtoms())
    expected = {"h2o":["H","H","O"],"co2":["C","O","O"],"nh3":["H","H","H","N"],"ch4":["C","H","H","H","H"]}[case_id]
    if not conf.Is3D() or sorted(a.GetSymbol() for a in atoms) != expected:
        raise ValueError("Das vollständige 3D-Modell passt nicht zum Beispiel.")
    coords = [tuple(conf.GetAtomPosition(a.GetIdx())) for a in atoms]
    if not all(math.isfinite(x) for v in coords for x in v):
        raise ValueError("Nicht endliche 3D-Koordinaten.")
    # For CO2 explicitly use a symmetric linear teaching geometry.
    # Preserve the axis of the RDKit model, average the two bond lengths.
    # Only the learning viewer is modified; the normal analysis is untouched.
    if case_id == "co2":
        mol3d = Chem.Mol(mol3d)
        conf = mol3d.GetConformer()
        ci = next(a.GetIdx() for a in atoms if a.GetSymbol()=="C")
        oi = [a.GetIdx() for a in atoms if a.GetSymbol()=="O"]
        c = coords[ci]
        axis = _dip_unit(_dip_vsub(coords[oi[0]],coords[oi[1]]))
        length = sum(math.sqrt(_dip_dot(_dip_vsub(coords[i],c),_dip_vsub(coords[i],c))) for i in oi)/2
        for i,sign in zip(oi,(1,-1)):
            coords[i] = _dip_vadd(c,_dip_vscale(axis,sign*length))
            conf.SetAtomPosition(i,coords[i])
    if case_id in ("nh3","ch4"):
        mol3d = Chem.Mol(mol3d)
        conf = mol3d.GetConformer()
        coords,axis = _dip_symmetric_hydrides(coords,atoms,case_id)
        for i,pos in enumerate(coords): conf.SetAtomPosition(i,pos)
    center = tuple(sum(v[k] for v in coords)/len(coords) for k in range(3))
    labels = []
    for a, pos in zip(atoms, coords):
        symbol = a.GetSymbol()
        radial = _dip_vsub(pos,center)
        away = (_dip_perp(axis) if case_id in ("co2","ch4") and symbol=="C"
                else _dip_unit(radial))
        labels.append({"symbol": symbol, "en": DIPOLE_EN_PAULING[symbol],
                       "position": _dip_xyz(_dip_vadd(pos, _dip_vscale(away, .46)))})
    bonds = []
    for bond in mol3d.GetBonds():
        a,b = bond.GetBeginAtom(),bond.GetEndAtom()
        ea,eb = DIPOLE_EN_PAULING[a.GetSymbol()],DIPOLE_EN_PAULING[b.GetSymbol()]
        if ea == eb:
            continue
        low,high = (a,b) if ea<eb else (b,a)
        p,q = coords[low.GetIdx()],coords[high.GetIdx()]
        u = _dip_unit(_dip_vsub(q,p))
        # Move the arrow parallel to its bond, away from the triangle
        # centroid, so it remains visible beside the ball-and-stick model.
        middle = _dip_vscale(_dip_vadd(p,q),.5)
        radial = _dip_vsub(middle,center)
        transverse = _dip_vsub(radial,_dip_vscale(u,_dip_dot(radial,u)))
        normal = (_dip_vscale(_dip_perp(axis),-1.) if case_id=="co2"
                  else _dip_unit(transverse) if _dip_dot(transverse,transverse)>1e-10
                  else _dip_perp(u))
        start = _dip_vadd(_dip_vadd(middle,_dip_vscale(normal,.40)),_dip_vscale(u,-.43))
        end = _dip_vadd(start,_dip_vscale(u,.86))
        cross_a = _dip_vadd(start,_dip_vscale(normal,.10))
        cross_b = _dip_vadd(start,_dip_vscale(normal,-.10))
        bonds.append({"low":low.GetSymbol(),"high":high.GetSymbol(),
                      "delta_en":round(abs(ea-eb),2),"direction":_dip_xyz(u),
                      "start":_dip_xyz(start),"end":_dip_xyz(end),
                      "cross_a":_dip_xyz(cross_a),"cross_b":_dip_xyz(cross_b)})
    low_symbol,high_symbol,n_bonds = {"h2o":("H","O",2),"co2":("C","O",2),
        "nh3":("H","N",3),"ch4":("H","C",4)}[case_id]
    if len(bonds)!=n_bonds or any(b["low"]!=low_symbol or b["high"]!=high_symbol for b in bonds):
        raise ValueError("Unerwartete Bindungstopologie.")
    vectors = [_dip_vscale(tuple(b["direction"].values()),.86) for b in bonds]
    total = tuple(sum(v[k] for v in vectors) for k in range(3))
    magnitude = math.sqrt(_dip_dot(total,total))
    zero = magnitude < 1e-8
    # In the addition stage use free vectors, translated away from the
    # molecule without changing their direction or magnitude.
    if case_id in ("nh3","ch4"):
        # Translate the entire open/closed polygon to a clear region beside
        # the molecule. Keep all edge vectors exactly unchanged.
        side = _dip_perp(axis)
        partial = (0.,0.,0.)
        offsets = [partial]
        for v in vectors:
            partial = _dip_vadd(partial,v)
            offsets.append(partial)
        extent = max(_dip_dot(_dip_vsub(p,center),side) for p in coords)
        lower = min(_dip_dot(p,side) for p in offsets)
        origin = _dip_vadd(center,_dip_vscale(side,extent-lower+.70))
        mean = tuple(sum(p[k] for p in offsets)/len(offsets) for k in range(3))
        origin = _dip_vsub(origin,_dip_vsub(mean,_dip_vscale(side,_dip_dot(mean,side))))
        if zero: total=(0.,0.,0.)
    elif zero:
        origin = _dip_vadd(center,_dip_vscale(_dip_perp(axis),.95))
        origin = _dip_vsub(origin,_dip_vscale(vectors[0],.5))
        total = (0.,0.,0.)
    else:
        origin = _dip_vsub(center,_dip_vscale(_dip_unit(total),1.25))
    points = [origin]
    for v in vectors:
        points.append(_dip_vadd(points[-1],v))
    if zero: points[-1] = origin
    # Translate the resultant next to the molecule, without changing its
    # vector. Atomic spheres no longer intersect its shaft or tip in 3D.
    # As with any 3D geometry, projection can still overlap from some views.
    result_anchor = _dip_resultant_anchor(center,total,coords,vectors)
    result_start = _dip_vsub(result_anchor,_dip_vscale(total,.5))
    result_end = _dip_vadd(result_start,total)
    return {"case_id":case_id,"name":case["name"],"molblock":Chem.MolToMolBlock(mol3d),
            "labels":labels,"bonds":bonds,"zero":zero,
            "points":[_dip_xyz(p) for p in points],
            "result_start":_dip_xyz(result_start),"result_end":_dip_xyz(result_end),
            "sum_norm":0. if zero else magnitude/.86,
            "idealized":case_id in ("co2","nh3","ch4"),
            "reference":DIPOLE_REFERENCE[case_id]}

def dipole_learning_iframe(mol3d, case_id):
    payload = dipole_learning_payload(mol3d, case_id)
    # Controls live inside this iframe: no Python callback rebuilds the viewer
    # when changing layers. Only the overlay shapes/labels are replaced.
    template = r'''<!doctype html>
<html lang="de"><head><meta charset="utf-8">
<style>
*{box-sizing:border-box}body{margin:0;padding:10px;font:14px Arial,sans-serif;color:#203246;background:#fff}
.toolbar{display:flex;gap:6px;flex-wrap:wrap;margin-bottom:8px}
button{border:1px solid #9aafbf;border-radius:6px;background:#fff;padding:8px 11px;color:#203246;font:inherit;cursor:pointer}
button[aria-pressed="true"]{background:#245d82;color:#fff;border-color:#245d82}
button:disabled{opacity:.55;cursor:default}
#dip-view{position:relative;width:100%;height:315px;border:1px solid #d5dfe6;border-radius:8px;overflow:hidden}
#dip-info{min-height:76px;line-height:1.45;margin-top:8px}
#dip-add-controls{margin-top:8px;display:flex;gap:8px;align-items:center;flex-wrap:wrap}
#dip-add-controls[hidden]{display:none}
#dip-add-count{font-size:13px;font-weight:bold}
#dip-reference{margin-top:8px;padding:8px 10px;border:1px solid #d5dfe6;border-radius:8px;background:#f7fafc}
#dip-reference[hidden]{display:none}
#dip-reference summary{cursor:pointer;font-weight:bold}
#dip-reference p{margin:5px 0;font-size:13px;line-height:1.4}
#dip-reference-value{font-size:17px;font-weight:bold;color:#245d82}
.note{font-size:12px;color:#506376;line-height:1.4;margin-top:6px}
</style>
<script src="https://cdn.jsdelivr.net/npm/3dmol@2.5.5/build/3Dmol-min.js"></script>
</head><body>
<div class="toolbar" role="group" aria-label="Darstellungsstufen">
<button id="dip-stage-0" type="button" aria-pressed="true" disabled>1 · Geometrie</button>
<button id="dip-stage-1" type="button" aria-pressed="false" disabled>2 · EN-Werte</button>
<button id="dip-stage-2" type="button" aria-pressed="false" disabled>3 · Bindungsdipole</button>
<button id="dip-stage-3" type="button" aria-pressed="false" disabled>4 · Vektoraddition</button>
</div>
<div id="dip-view">3D-Modell wird geladen …</div>
<div id="dip-add-controls" hidden>
<button id="dip-prev" type="button" disabled>← Zurück</button>
<span id="dip-add-count" aria-live="polite">1 von 4</span>
<button id="dip-next" type="button" disabled>Weiter →</button>
</div>
<div id="dip-info" role="status">Drehe das Molekül und beschreibe seine räumliche Form.</div>
<div class="note">Didaktisches Modell · EN nach Pauling · Pfeillängen nur zur Veranschaulichung, keine Debye-Werte.</div>
<details id="dip-reference" hidden>
<summary>Experimenteller Bezug · Referenzwert vergleichen</summary>
<p><strong id="dip-reference-value"></strong> · <span id="dip-reference-kind"></span></p>
<p id="dip-reference-note"></p>
<p id="dip-reference-compare"></p>
<p><b>Modellgrenze:</b> Die EN-Pfeile sind gleich lange qualitative Modellvektoren innerhalb eines Moleküls. Ihre Summe hat keine Einheit Debye und erlaubt keinen quantitativen Vergleich der Pfeillängen verschiedener Moleküle. Die reale Elektronendichte einschließlich freier Elektronenpaare bestimmt das physikalische Dipolmoment. Auch unpolare Moleküle sind polarisierbar.</p>
<p>Quelle: <a id="dip-reference-source" href="https://cccbdb.nist.gov/diplistx.asp" target="_blank" rel="noopener noreferrer"></a> · 1 D ≈ 3,336 × 10⁻³⁰ C m.</p>
</details>
<script>
(function(){
"use strict";
const data = __DIPOLE_PAYLOAD__;
const el=document.getElementById("dip-view");
const info=document.getElementById("dip-info");
const buttons=[0,1,2,3].map(i=>document.getElementById("dip-stage-"+i));
const addControls=document.getElementById("dip-add-controls");
const reference=document.getElementById("dip-reference");
const prev=document.getElementById("dip-prev"),next=document.getElementById("dip-next"),count=document.getElementById("dip-add-count");
let viewer=null, labels=[], shapes=[], addition=0;
const notesByCase={h2o:[
  "Drehe das Molekül und beschreibe seine räumliche Form. Wie liegen die H-Atome zueinander?",
  "Vergleiche die EN-Werte an den Atomen: H = 2,20; O = 3,44. Welche Richtung erwartest du für die Bindungspolarisation?",
  "O–H: ΔEN = 1,24. Beide blauen chemischen Dipolpfeile zeigen von H (δ+) zu O (δ−). Der Querstrich markiert die positive Seite. Die physikalische Dipolmomentkonvention ist entgegengesetzt."
],co2:[
  "Drehe das Molekül: Wie sind die beiden O-Atome um das C-Atom angeordnet? Im Lernmodus ist CO₂ idealisiert linear und symmetrisch.",
  "Vergleiche die EN-Werte: C = 2,55; O = 3,44. Welche Richtung erwartest du für die Bindungspolarisation?",
  "C=O: ΔEN = 0,89. Die beiden chemischen Dipolpfeile zeigen von C zu O. Der Querstrich markiert die positive Seite. Die physikalische Dipolmomentkonvention ist entgegengesetzt."
],nh3:[
  "Drehe das Molekül. Wie sind die drei H-Atome um N angeordnet? Das Lernmodell ist dreizählig symmetrisch.",
  "Vergleiche die EN-Werte: H = 2,20; N = 3,04. Welche Richtung erwartest du?",
  "N–H: ΔEN = 0,84. Drei gleich lange chemische Dipolpfeile zeigen von H zu N. Der Querstrich markiert die positive Seite. Die physikalische Konvention ist entgegengesetzt."
],ch4:[
  "Drehe das Molekül. Welche räumliche Anordnung bilden die vier H-Atome? Das Lernmodell ist ideal tetraedrisch.",
  "Vergleiche die EN-Werte: H = 2,20; C = 2,55. Welche Richtung erwartest du?",
  "C–H: ΔEN = 0,35. Die vier schwach polarisierten Bindungen werden durch gleich lange chemische Dipolpfeile H → C modelliert. Die physikalische Konvention ist entgegengesetzt."
]};
const notes=notesByCase[data.case_id];
const n=data.bonds.length;
const colors=["#2466bb","#db7c15","#8756a8","#007f83"];
const colorNames=["blaue","orangefarbene","violette","türkise"];
const addNotes=Array.from({length:n},(_,i)=>i===0?
  "1. Erster Bindungsdipol: Der blaue Vektor wurde parallel neben das Molekül verschoben; Länge und Richtung bleiben gleich.":
  (i+1)+". Der "+colorNames[i]+" Vektor beginnt an der Spitze des vorherigen. Drehe die räumliche Konstruktion.");
addNotes.push(data.zero?
  (n+1)+". Die Vektorkette führt genau zum Start zurück. Alle Beiträge kompensieren sich: Σ = 0.":
  (n+1)+". Der grüne Vektor verbindet Start und Ende der Konstruktion: Die Resultierende ist nicht null.");
addNotes.push(data.zero?
  (n+2)+". Kein permanenter Moleküldipol im symmetrischen "+(data.case_id==="co2"?"CO₂":"CH₄")+". Ein Nullvektor hat keine Richtung.":
  (n+2)+". Der grüne Modellvektor liegt parallel versetzt neben dem Molekül. Seine Richtung folgt "+(data.case_id==="h2o"?"der Winkelhalbierenden":"der dreizähligen Symmetrieachse in Richtung N")+". Länge und Richtung bleiben unverändert. Kein gemessener Dipolbetrag.");
function arrow(start,end,color,radius){
  shapes.push(viewer.addArrow({start,end,color,radius,mid:.73}));
}
function label(text,position,color){
  labels.push(viewer.addLabel(text,{position,fontSize:13,fontColor:color,
    backgroundColor:"#ffffff",backgroundOpacity:.92,borderColor:"#9aafbf",borderThickness:1,inFront:true}));
}
function drawAddition(){
  const p=data.points;
  if(addition<=n){
    for(let i=0;i<Math.min(addition+1,n);i++){
      arrow(p[i],p[i+1],colors[i],i===0?.052:(data.case_id==="co2"?.031:.047));
    }
    if(addition===n&&!data.zero)arrow(p[0],p[n],"#17814a",.064);
    if(addition===n&&data.zero){
      shapes.push(viewer.addSphere({center:p[0],radius:.09,color:"#17814a"}));
      label("Σ = 0",p[0],"#17814a");
    }
  }else if(!data.zero){
    arrow(data.result_start,data.result_end,"#17814a",.064);
  }else{
    label("Σ = 0",p[0],"#17814a");
  }
  prev.disabled=addition===0;next.disabled=addition===n+1;
  count.textContent=(addition+1)+" von "+(n+2);
}
function stage(k){
  if(!viewer || k<0 || k>3)return;
  try{
    labels.forEach(label=>viewer.removeLabel(label));
    shapes.forEach(shape=>viewer.removeShape(shape));
    labels=[];shapes=[];
    addControls.hidden=k!==3;
    const showReference=k===3&&addition===n+1;
    reference.hidden=!showReference;
    if(!showReference)reference.open=false;
    if(k===1||k===2){
      data.labels.forEach(a=>{
        const text=a.symbol+"  EN "+a.en.toFixed(2).replace(".",",");
        labels.push(viewer.addLabel(text,{position:a.position,fontSize:13,fontColor:"#203246",
          backgroundColor:"#ffffff",backgroundOpacity:.90,borderColor:"#9aafbf",borderThickness:1,inFront:true}));
      });
    }
    if(k===2){
      data.bonds.forEach(b=>{
        shapes.push(viewer.addArrow({start:b.start,end:b.end,color:"#2466bb",radius:.047,mid:.73}));
        shapes.push(viewer.addCylinder({start:b.cross_a,end:b.cross_b,color:"#2466bb",radius:.028,fromCap:1,toCap:1}));
      });
    }
    buttons.forEach((b,i)=>b.setAttribute("aria-pressed",String(i===k)));
    if(k===3)drawAddition();
    info.textContent=k===3?addNotes[addition]:notes[k];
    viewer.render(); // deliberately no zoomTo, rotation or model reload
  }catch(err){
    info.textContent="Darstellung konnte nicht aktualisiert werden: "+err.message;
  }
}
function init(attempt){
  if(typeof $3Dmol==="undefined"){
    if(attempt<20){window.setTimeout(()=>init(attempt+1),150);return;}
    el.textContent="3Dmol.js konnte nicht geladen werden. Bitte Analyse neu starten.";
    return;
  }
  try{
    el.textContent="";
    viewer=$3Dmol.createViewer(el,{backgroundColor:"white"});
    viewer.addModel(data.molblock,"sdf");
    viewer.setStyle({},{stick:{radius:.10},sphere:{scale:.25}});
    viewer.zoomTo();
    viewer.zoom(n>2?.60:.77);
    viewer.render();
    document.getElementById("dip-reference-value").textContent=data.reference.display;
    document.getElementById("dip-reference-kind").textContent=data.reference.kind;
    document.getElementById("dip-reference-note").textContent=data.reference.note;
    document.getElementById("dip-reference-source").textContent=data.reference.source;
    document.getElementById("dip-reference-compare").textContent=data.zero?"Das qualitative Modell und der Referenzwert stimmen in der Nullsumme überein.":"Das qualitative Modell und der Referenzwert stimmen darin überein, dass ein permanentes Dipolmoment vorliegt. Der Modellvektor liefert nicht den gemessenen Betrag.";
    stage(0);
    buttons.forEach((button,k)=>{button.disabled=false;button.addEventListener("click",()=>{if(k===3)addition=0;stage(k);});});
    prev.addEventListener("click",()=>{if(addition>0){addition--;stage(3);}});
    next.addEventListener("click",()=>{if(addition<n+1){addition++;stage(3);}});
    window.addEventListener("resize",()=>{viewer.resize();viewer.render();});
  }catch(err){
    el.textContent="3D-Anzeige fehlgeschlagen: "+err.message;
    buttons.forEach(button=>button.disabled=true);
  }
}
init(0);
})();
</script></body></html>'''
    inner=template.replace("__DIPOLE_PAYLOAD__",json.dumps(payload,ensure_ascii=True,allow_nan=False))
    return ("<iframe class='dipole-learning-frame' title='Dipol-Lernmodus – "+html.escape(payload["name"],quote=True)+"' "
            "srcdoc='"+html.escape(inner,quote=True)+"' width='680' height='805' "
            "style='width:100%;max-width:680px;border:1px solid #cbd9e2;border-radius:8px;background:white;' "
            "loading='eager'></iframe>")

def dipole_learning_widget(mol3d, case_id):
    """Lazy opt-in; toggling inside the frame never recomputes coordinates."""
    button = widgets.Button(description="Bindungsdipole untersuchen",
                            layout=widgets.Layout(width="auto"),button_style="info")
    panel = widgets.HTML(value="",layout=widgets.Layout(width="100%",max_width="690px"))
    def toggle(_):
        if panel.value:
            panel.value=""
            button.description="Bindungsdipole untersuchen"
        else:
            try:
                panel.value=dipole_learning_iframe(mol3d,case_id)
                button.description="Dipol-Lernmodus schließen"
            except Exception as exc:
                panel.value="<p>Dipol-Lernmodus nicht verfügbar: "+html.escape(str(exc))+"</p>"
                button.description="Dipol-Lernmodus schließen"
    button.on_click(toggle)
    return widgets.VBox([button,panel],layout=widgets.Layout(width="100%",max_width="700px"))


def _label_bar_html(labels, active_index):
    """Kleine qualitative Skala als HTML."""
    cells=[]
    for i,lab in enumerate(labels):
        if i == active_index:
            cells.append(f"<span style='display:inline-block; padding:3px 7px; margin:1px; border-radius:10px; background:#e8f0fe; border:1px solid #5f7fcf; font-weight:bold;'>{html.escape(lab)}</span>")
        else:
            cells.append(f"<span style='display:inline-block; padding:3px 7px; margin:1px; border-radius:10px; background:#f6f6f6; border:1px solid #ddd; color:#666;'>{html.escape(lab)}</span>")
    return " ".join(cells)

def _formal_charge(mol):
    try:
        return sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    except Exception:
        return 0

def _hetero_heavy_count(mol):
    return sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() not in (1,6))

def qualitative_polarity_assessment(mol, props):
    """Didaktische Näherung für Polarität und Wasserlöslichkeit.

    Absichtlich keine exakte Stoffdatenvorhersage: Die Skalen übersetzen einfache
    RDKit-Modellgrößen in eine schülernahe Orientierung.
    """
    logp = float(props.get("logP (Modellwert)", 0) or 0)
    tpsa = float(props.get("TPSA / Å²", 0) or 0)
    hbd = int(props.get("H-Brücken-Donoren", 0) or 0)
    hba = int(props.get("H-Brücken-Akzeptoren", 0) or 0)
    heavy = max(int(props.get("schwere Atome", 1) or 1), 1)
    charge = abs(_formal_charge(mol))
    hetero = _hetero_heavy_count(mol)

    # Polaritätseindruck: eher Dichte polarer Oberfläche + H-Brücken-Möglichkeiten,
    # abgeschwächt durch deutlich lipophile Anteile.
    polar_score = 0.0
    polar_score += min(tpsa / 25.0, 3.0)
    polar_score += min(hbd * 0.8, 1.6)
    polar_score += min(hba * 0.25, 1.0)
    polar_score += min(charge * 1.4, 2.8)
    polar_score -= max(logp, 0) * 0.45
    polar_score -= max((heavy - 2*max(hetero,1)) / 18.0, 0)
    polar_score = max(polar_score, 0)

    if polar_score < 0.4:
        pol_i = 0
    elif polar_score < 1.0:
        pol_i = 1
    elif polar_score < 1.75:
        pol_i = 2
    elif polar_score < 2.8:
        pol_i = 3
    else:
        pol_i = 4

    # Wasserlöslichkeits-Tendenz: H-Brücken und TPSA begünstigen, logP und Größe
    # erschweren. Kleine polare Carbonyl-/Heteroatom-Moleküle erhalten einen Bonus.
    sol_score = 0.0
    sol_score += hbd * 1.2 + hba * 1.1 + tpsa / 25.0
    sol_score += min(charge * 3.0, 4.0)
    sol_score -= max(logp, 0) * 1.15
    sol_score -= max(heavy - 6, 0) * 0.25
    if heavy <= 5 and hba >= 1 and tpsa > 14:
        sol_score += 0.75
    sol_score = max(sol_score, 0)

    if sol_score < 0.45:
        sol_i = 0
    elif sol_score < 1.45:
        sol_i = 1
    elif sol_score < 2.35:
        sol_i = 2
    elif sol_score < 3.5:
        sol_i = 3
    else:
        sol_i = 4

    pol_labels = ["sehr schwach", "schwach", "mittel", "stark", "sehr stark"]
    sol_labels = ["kaum wasserlöslich", "begrenzt", "mäßig", "gut", "sehr gut"]

    reasons=[]
    if hbd > 0:
        reasons.append("kann H-Brücken spenden")
    if hba > 0:
        reasons.append("kann H-Brücken aufnehmen")
    if tpsa >= 70:
        reasons.append("große polare Oberfläche")
    elif tpsa >= 20:
        reasons.append("erkennbare polare Oberfläche")
    if logp >= 2.5:
        reasons.append("deutlich lipophiler Anteil")
    elif logp >= 0.8:
        reasons.append("unpolarer/lipophiler Anteil nimmt zu")
    if charge:
        reasons.append("formale Ladung vorhanden")
    if not reasons:
        reasons.append("kaum polare Gruppen erkennbar")

    return {
        "polarity_index": pol_i,
        "polarity_label": pol_labels[pol_i],
        "solubility_index": sol_i,
        "solubility_label": sol_labels[sol_i],
        "polarity_score": round(polar_score, 2),
        "solubility_score": round(sol_score, 2),
        "reason": "; ".join(reasons) + "."
    }

def _polarity_panel_html(mol, props):
    a = qualitative_polarity_assessment(mol, props)
    pol_labels = ["sehr schwach", "schwach", "mittel", "stark", "sehr stark"]
    sol_labels = ["kaum wasserlöslich", "begrenzt", "mäßig", "gut", "sehr gut"]
    return f"""
    <div style='margin-top:8px; padding:8px; border:1px solid #e0e0e0; border-radius:8px; background:#fbfbfb;'>
      <div style='font-size:0.95em; margin-bottom:5px;'><b>Didaktischer Polaritätseindruck (unvalidierte Heuristik):</b> {html.escape(a['polarity_label'])}</div>
      <div style='font-size:0.82em; margin-bottom:8px;'>{_label_bar_html(pol_labels, a['polarity_index'])}</div>
      <div style='font-size:0.95em; margin-bottom:5px;'><b>Wasserlöslichkeits-Tendenz:</b> {html.escape(a['solubility_label'])}</div>
      <div style='font-size:0.82em; margin-bottom:8px;'>{_label_bar_html(sol_labels, a['solubility_index'])}</div>
      <div style='font-size:0.82em; color:#555;'><b>Kurzbegründung:</b> {html.escape(a['reason'])}</div>
      <details style='font-size:0.78em; color:#555; line-height:1.32; margin-top:6px;'>
        <summary style='cursor:pointer;'>Näherung und Modellgrenzen anzeigen</summary>
        <div style='margin-top:5px;'>Diese Einschätzungen sind didaktische Näherungen. Sie werden aus einfachen Modellwerten wie logP, TPSA, Wasserstoffbrücken-Donoren/-Akzeptoren, Molekülgröße und formaler Ladung abgeleitet. Sie sollen Trends verständlich machen, ersetzen aber keine experimentellen Löslichkeitsdaten. Die tatsächliche Wasserlöslichkeit hängt zusätzlich von Temperatur, Ionisierungszustand, Konzentration, Molekülform und Wechselwirkungen mit dem Lösungsmittel ab.</div>
      </details>
    </div>
    """

def _surface_legend_html(style_name):
    if not _is_polar_surface(style_name):
        return ""
    return """
    <div style='font-size:0.78em; color:#555; line-height:1.3; margin-top:5px;'>
      <b>Didaktische Oberfläche:</b> elementbasierte Näherung, keine exakte elektrostatische Potentialfläche.
      Rot/blau/gelb/grün markieren heteroatomreiche, häufig polarere Bereiche; grau/weiß überwiegend C/H-Bereiche.
    </div>
    """

def _props_table_html(props):
    rows = []
    for k, v in props.items():
        rows.append(f"<tr><td style='padding:4px 8px; border-bottom:1px solid #eee;'>{html.escape(str(k))}</td><td style='padding:4px 8px; border-bottom:1px solid #eee; text-align:right;'><b>{html.escape(str(v))}</b></td></tr>")
    return "<table style='border-collapse:collapse; width:100%; font-size:0.92em;'>" + "".join(rows) + "</table>"

def _descriptor_note_html():
    return """
    <div style='font-size:0.78em; color:#555; line-height:1.42; margin-top:8px; border-top:1px solid #eee; padding-top:6px;'>
      <b>logP (Wildman–Crippen):</b> atombeitragsbasierter RDKit-Modellwert zur Lipophilie,
      kein experimenteller Octanol/Wasser-Verteilungskoeffizient.<br>
      <b>TPSA:</b> topologische polare Oberfläche nach Fragmentbeiträgen, kein Dipolmoment.<br>
      <b>Rotierbare Bindungen:</b> RDKit-Deskriptordefinition; die Rotation von
      endständigen Gruppen wird nicht vollständig abgebildet.
      Gleiche Modellwerte beweisen keine identischen experimentellen Eigenschaften.
    </div>
    """


def _safe_text(x):
    return "" if x is None else str(x)

def _has_stoffdaten(card):
    sd = card.get("stoffdaten") or {}
    return any(sd.get(k) not in (None, "", []) for k in ["schmelzpunkt_C", "siedepunkt_C", "dichte_g_cm3", "wasserloeslichkeit", "aggregatzustand_raumtemperatur", "hinweis"])

def _stoffdaten_compact_html(card):
    sd = card.get("stoffdaten") or {}
    if not _has_stoffdaten(card):
        return ""
    rows=[]
    labels=[
        ("Aggregatzustand", "aggregatzustand_raumtemperatur"),
        ("Schmelzpunkt / °C", "schmelzpunkt_C"),
        ("Siedepunkt / °C", "siedepunkt_C"),
        ("Dichte / g cm⁻³", "dichte_g_cm3"),
        ("Wasserlöslichkeit", "wasserloeslichkeit"),
    ]
    for label,key in labels:
        val = sd.get(key)
        if val not in (None, "", []):
            rows.append(f"<tr><td style='padding:3px 7px;border-bottom:1px solid #eee;'>{html.escape(label)}</td><td style='padding:3px 7px;border-bottom:1px solid #eee;'><b>{html.escape(str(val))}</b></td></tr>")
    if sd.get("hinweis"):
        rows.append(f"<tr><td style='padding:3px 7px;border-bottom:1px solid #eee;'>Hinweis</td><td style='padding:3px 7px;border-bottom:1px solid #eee;'>{html.escape(str(sd.get('hinweis')))}</td></tr>")
    if not rows:
        return ""
    source = sd.get("quelle") or "kuratierte Unterrichtsdaten"
    return "<table style='border-collapse:collapse;width:100%;font-size:0.88em;'>" + "".join(rows) + "</table>" + f"<div style='font-size:0.75em;color:#666;margin-top:4px;'>Quelle/Einordnung: {html.escape(str(source))}</div>"

def _stoffdaten_comparison_html(cards, open_by_default=True):
    if not any(_has_stoffdaten(c) for c in cards):
        return ""
    labels=[
        ("Aggregatzustand", "aggregatzustand_raumtemperatur"),
        ("Schmelzpunkt / °C", "schmelzpunkt_C"),
        ("Siedepunkt / °C", "siedepunkt_C"),
        ("Dichte / g cm⁻³", "dichte_g_cm3"),
        ("Wasserlöslichkeit", "wasserloeslichkeit"),
    ]
    rows=[]
    for label,key in labels:
        vals=[]
        has=False
        for c in cards:
            sd=c.get("stoffdaten") or {}
            val=sd.get(key)
            if val not in (None, "", []): has=True
            vals.append("—" if val in (None, "", []) else html.escape(str(val)))
        if has:
            rows.append("<tr><td style='padding:4px 8px;border-bottom:1px solid #eee;'><b>"+html.escape(label)+"</b></td>"+"".join(f"<td style='padding:4px 8px;border-bottom:1px solid #eee;'>{v}</td>" for v in vals)+"</tr>")
    if not rows:
        return ""
    header="<tr><th style='text-align:left;padding:4px 8px;'>Stoffdatum</th>"+"".join(f"<th style='text-align:left;padding:4px 8px;'>{html.escape(c.get('name',''))}</th>" for c in cards)+"</tr>"
    open_attr=" open" if open_by_default else ""
    return f"""
    <details{open_attr} style='margin:10px 0;'>
      <summary><b>Stoffdaten / experimenteller Bezug</b></summary>
      <div style='font-size:0.9em;color:#333;padding:6px 0;'>
        <table style='border-collapse:collapse;width:100%;'>{header}{''.join(rows)}</table>
        <div style='font-size:0.78em;color:#666;margin-top:5px;'>Kuratierte Unterrichtswerte; Werte können je nach Quelle, Temperatur, Druck und Stoffform leicht abweichen. Sie dienen hier dem Struktur–Eigenschaft-Vergleich.</div>
      </div>
    </details>
    """

def _has_reactivity(card):
    r = card.get("reaktivitaet") or {}
    return any(r.get(k) not in (None, "", []) for k in ["hinweis", "warum", "reaktionsidee"])

def _reactivity_box_html(card, open_by_default=False):
    r = card.get("reaktivitaet") or {}
    if not _has_reactivity(card):
        return ""
    open_attr = " open" if open_by_default else ""
    return f"""
    <details{open_attr} style='margin:10px 0; border:1px solid #e5e5e5; border-radius:8px; padding:7px; background:#fbfbff;'>
      <summary style='cursor:pointer;'><b>Reaktivitäts-Hinweis anzeigen</b></summary>
      <div style='font-size:0.92em; line-height:1.35; padding-top:6px;'>
        <b>Hinweis:</b> {html.escape(_safe_text(r.get('hinweis')))}<br>
        <b>Warum?</b> {html.escape(_safe_text(r.get('warum')))}<br>
        <b>Reaktionsidee:</b> {html.escape(_safe_text(r.get('reaktionsidee')))}
        {('<br><span style="font-size:0.85em;color:#666;"><b>Modellgrenze:</b> '+html.escape(str(r.get('modellgrenze')))+' </span>') if r.get('modellgrenze') else ''}
      </div>
    </details>
    """

def _reactivity_comparison_html(cards):
    if not any(_has_reactivity(c) for c in cards):
        return ""
    rows=[]
    for field,label in [("hinweis","Hinweis"),("warum","Warum?"),("reaktionsidee","Reaktionsidee")]:
        vals=[]; has=False
        for c in cards:
            val=(c.get("reaktivitaet") or {}).get(field)
            if val not in (None,"",[]): has=True
            vals.append("—" if val in (None,"",[]) else html.escape(str(val)))
        if has:
            rows.append("<tr><td style='padding:4px 8px;border-bottom:1px solid #eee;'><b>"+html.escape(label)+"</b></td>"+"".join(f"<td style='padding:4px 8px;border-bottom:1px solid #eee;'>{v}</td>" for v in vals)+"</tr>")
    header="<tr><th style='text-align:left;padding:4px 8px;'>Reaktivität</th>"+"".join(f"<th style='text-align:left;padding:4px 8px;'>{html.escape(c.get('name',''))}</th>" for c in cards)+"</tr>"
    return f"""
    <details style='margin:10px 0;'>
      <summary><b>Reaktivitäts-Hinweise vergleichen</b></summary>
      <div style='font-size:0.9em;color:#333;padding:6px 0;'>
        <table style='border-collapse:collapse;width:100%;'>{header}{''.join(rows)}</table>
        <div style='font-size:0.78em;color:#666;margin-top:5px;'>Qualitative Orientierung anhand funktioneller Gruppen; keine vollständige Reaktionsvorhersage.</div>
      </div>
    </details>
    """

def display_structure_panel(mol, mol3d, props=None, groups=None, style="Kugel-Stab"):
    """Kompakte 2D-/3D-/Eigenschaften-Ansicht in einer Zeile, soweit Bildschirmbreite vorhanden ist."""
    png = mol_png_data_uri(mol, 300, 230)
    iframe = mol3d_iframe_html(mol3d, width=460, height=330, style=style, mode=render_mode_dd.value)
    props_html = _props_table_html(props or {}) if props else ""
    groups_html = ", ".join(_functional_group_display_label(mol, group) for group in (groups or [])) if groups else "keine aus der einfachen Liste erkannt"
    if module_dd.value in ("Funktionelle Gruppen", "Wirkstoffe und Biomoleküle"):
        # Keep the proven 3D viewer and descriptive table; replace only static
        # 2D by an independent, inline-selectable, highlighted RDKit drawing.
        group_view = functional_group_highlight_widget(mol, groups)
        structure_3d = widgets.HTML(
            value=("<h3 style='margin:0 0 8px;'>3D-Modell</h3>" +
                   iframe + _surface_legend_html(style)),
            layout=widgets.Layout(width="510px", max_width="100%")
        )
        display(widgets.HBox([group_view, structure_3d], layout=widgets.Layout(
            display="flex", flex_flow="row wrap", gap="16px",
            width="100%", max_width="1040px", align_items="flex-start"
        )))
        display(HTML(
            "<section style='max-width:1040px;margin:12px 0;'>"
            "<h3>Eigenschaften (RDKit-Modellwerte)</h3>"
            "<div style='border:1px solid #ddd;border-radius:8px;padding:10px;'>"
            + props_html
            + (_descriptor_note_html() if props else "") + "</div>"
            "<h3>Funktionelle Gruppen</h3><p>" + html.escape(groups_html)
            + "</p></section>"
        ))
        return
    display(HTML(f"""
    <div style="display:flex; flex-wrap:wrap; gap:14px; align-items:flex-start; margin-top:8px;">
      <div style="flex:0 0 315px; max-width:100%;">
        <h3 style="margin:0 0 6px 0;">2D-Struktur</h3>
        <div style="border:1px solid #ddd; border-radius:8px; background:white; padding:8px; text-align:center;">
          <img src="{png}" style="max-width:100%; height:auto;">
        </div>
      </div>
      <div style="flex:0 0 500px; max-width:100%;">
        <h3 style="margin:0 0 6px 0;">3D-Modell</h3>
        {iframe}
        {_surface_legend_html(style)}
      </div>
      <div style="flex:1 1 330px; min-width:300px; max-width:520px;">
        <h3 style="margin:0 0 6px 0;">Eigenschaften</h3>
        <div style="border:1px solid #ddd; border-radius:8px; background:white; padding:8px;">
          {props_html}
          {_polarity_panel_html(mol, props) if props and module_dd.value == "Polarität und Löslichkeit" else ""}
          {_descriptor_note_html() if props else ""}
        </div>
        <h3 style="margin:12px 0 6px 0;">Funktionelle Gruppen</h3>
        <div style="border:1px solid #ddd; border-radius:8px; background:white; padding:8px;">{html.escape(groups_html)}</div>
      </div>
    </div>
    """))


def _structure_card_html(card, style="Kugel-Stab", width3d=330, height3d=255):
    """Kompakte HTML-Karte mit 2D- und 3D-Struktur für Vergleichsansichten."""
    mol = mol_from_smiles(card["smiles"])
    mol3d = make_3d(mol)
    png = mol_png_data_uri(mol, 230, 175)
    iframe = mol3d_iframe_html(mol3d, width=width3d, height=height3d, style=style, mode=render_mode_dd.value)
    return f"""
    <div style="flex:1 1 430px; min-width:390px; max-width:520px; border:1px solid #ddd; border-radius:10px; padding:10px; background:#fff;">
      <h3 style="margin-top:0;">{html.escape(card['name'])}</h3>
      <p style="margin:4px 0 8px 0;"><b>SMILES:</b> <code>{html.escape(card['smiles'])}</code></p>
      <div style="display:flex; flex-wrap:wrap; gap:10px; align-items:flex-start;">
        <div style="flex:0 0 240px; text-align:center;">
          <b>2D</b><br>
          <img src="{png}" style="max-width:100%; height:auto;">
        </div>
        <div style="flex:1 1 340px; min-width:320px;">
          <b>3D</b><br>
          {iframe}
          {_surface_legend_html(style)}
        </div>
      </div>
    </div>
    """

def _isomer_comparison_html(card_a, card_b, mol_a, mol_b, props_a, props_b, groups_a, groups_b):
    """One contiguous, readable and source-aware 3-column comparison table."""
    def esc(value):
        return html.escape(str(value))
    def shown(value):
        return "nicht hinterlegt" if value in (None, "", []) else str(value)
    def row(label, value_a, value_b):
        va, vb = shown(value_a), shown(value_b)
        different = va != vb and va != "nicht hinterlegt" and vb != "nicht hinterlegt"
        bg = "background:#edf7f9;" if different else ""
        return (
            f"<tr style='{bg}'><th scope='row' style='text-align:left;vertical-align:top;"
            f"padding:8px 11px;border-bottom:1px solid #e3eaf1;font-weight:600;'>"
            f"{esc(label)}</th><td style='vertical-align:top;padding:8px 11px;"
            f"border-bottom:1px solid #e3eaf1;'>{esc(va)}</td>"
            f"<td style='vertical-align:top;padding:8px 11px;border-bottom:1px solid #e3eaf1;'>"
            f"{esc(vb)}</td></tr>"
        )
    def section(title):
        return (
            "<tr><th colspan='3' style='text-align:left;padding:11px;"
            "background:#e4f1f6;color:#173c58;font-size:0.92rem;'>"
            + esc(title) + "</th></tr>"
        )
    a, b = card_a.get("stoffdaten") or {}, card_b.get("stoffdaten") or {}
    ra, rb = card_a.get("reaktivitaet") or {}, card_b.get("reaktivitaet") or {}
    main = [
        section("Struktur und funktionelle Gruppen"),
        row("Summenformel", props_a["Summenformel"], props_b["Summenformel"]),
        row("Molmasse / g mol⁻¹", props_a["Molmasse / g mol⁻¹"], props_b["Molmasse / g mol⁻¹"]),
        row("Funktionelle Gruppen",
            ", ".join(groups_a) or "keine erkannt", ", ".join(groups_b) or "keine erkannt"),
        row("H-Brücken-Donoren (RDKit)", props_a["H-Brücken-Donoren"], props_b["H-Brücken-Donoren"]),
        row("H-Brücken-Akzeptoren (RDKit)", props_a["H-Brücken-Akzeptoren"], props_b["H-Brücken-Akzeptoren"])
    ]
    fields = [
        ("Aggregatzustand bei Raumtemperatur", "aggregatzustand_raumtemperatur"),
        ("Schmelzpunkt / °C", "schmelzpunkt_C"),
        ("Siedepunkt / °C", "siedepunkt_C"),
        ("Dichte / g cm⁻³", "dichte_g_cm3"),
        ("Wasserlöslichkeit", "wasserloeslichkeit")
    ]
    # Keep scientific content visible; never fill a long table with empty rows.
    actual_fields = [(label, key) for label, key in fields
                     if a.get(key) not in (None, "", []) or b.get(key) not in (None, "", [])]
    if actual_fields:
        main.append(section("Experimentelle / kuratierte Stoffdaten"))
        for label, key in actual_fields:
            main.append(row(label, a.get(key), b.get(key)))
    else:
        main.append(section("Experimenteller Stoffdatenbezug"))
        main.append("<tr><td colspan='3' style='padding:10px 12px;"
                    "border-bottom:1px solid #e3eaf1;color:#4e6878;'>"
                    "Für dieses Paar liegen derzeit keine hinreichend "
                    "kuratierten experimentellen Einzelstoffdaten vor. "
                    "Die Struktur- und Stereochemiefrage ist dennoch bearbeitbar."
                    "</td></tr>")
    reactive_fields = [
        ("Einordnung", "hinweis"), ("Strukturelle Begründung", "warum"),
        ("Beispielhafte Reaktionsidee", "reaktionsidee")
    ]
    nonempty = [(label, key) for label, key in reactive_fields
                if ra.get(key) not in (None, "", []) or rb.get(key) not in (None, "", [])]
    if nonempty:
        main.append(section("Qualitative Reaktivität (keine Reaktionsprognose)"))
        for label, key in nonempty:
            main.append(row(label, ra.get(key), rb.get(key)))
    else:
        main.append(section("Reaktivität – strukturelle Einordnung"))
        main.append("<tr><td colspan='3' style='padding:10px 12px;color:#4e6878;'>"
                    "Keine einzelnen Reaktionsangaben kuratiert. "
                    "Die MC-Frage und die strukturelle Interpretation bleiben relevant."
                    "</td></tr>")
    header = (
        "<table style='border-collapse:collapse;width:100%;min-width:490px;"
        "font-size:0.91rem;color:#213747;line-height:1.45;'>"
        "<thead><tr style='background:#113d59;color:#fff;'>"
        "<th style='text-align:left;padding:12px 11px;'>Eigenschaft</th>"
        f"<th style='text-align:left;padding:12px 11px;'>{esc(card_a['name'])}</th>"
        f"<th style='text-align:left;padding:12px 11px;'>{esc(card_b['name'])}</th></tr></thead>"
    )
    source_lines = [
        f"{esc(card['name'])}: {esc(sd.get('quelle') or 'Quellenangabe noch offen')}; "
        f"{esc(sd.get('bedingungen') or 'Bedingungen nicht hinterlegt')}"
        for card, sd in [(card_a, a), (card_b, b)]
        if sd.get("quelle") or sd.get("bedingungen")
    ]
    sources = (
        "<p style='font-size:0.79rem;color:#526d7e;line-height:1.5;'>"
        "<b>Quellen / Bedingungen:</b><br>" + "<br>".join(source_lines) +
        "<br>Fehlende Werte werden nicht geschätzt.</p>"
    ) if source_lines else (
        "<p style='font-size:0.79rem;color:#526d7e;'>"
        "Keine geprüfte Stoffdatenquelle hinterlegt; "
        "keine Schätzung aus Strukturdeskriptoren.</p>"
    )
    model_rows = [
        row("TPSA / Å² (topologisch)", props_a["TPSA / Å²"], props_b["TPSA / Å²"]),
        row("logP (Wildman–Crippen, Modell)", props_a["logP (Modellwert)"], props_b["logP (Modellwert)"]),
        row("Rotierbare Bindungen (RDKit-Definition)",
            props_a["rotierbare Bindungen"], props_b["rotierbare Bindungen"]),
        row("Ringanzahl (RDKit)", props_a["Ringanzahl"], props_b["Ringanzahl"])
    ]
    extra = (
        "<details style='margin:14px 0;border:1px solid #cbdfe8;border-radius:11px;"
        "padding:10px 13px;background:#f8fbfd;'>"
        "<summary style='cursor:pointer;font-weight:700;color:#175374;'>"
        "Weitere RDKit-Modellwerte und ihre Grenzen</summary>"
        "<div style='overflow-x:auto;margin-top:9px;'>"
        + header + "<tbody>" + "".join(model_rows) + "</tbody></table></div>"
        "<p style='font-size:0.83rem;line-height:1.48;color:#415d70;'>"
        "<b>Wildman–Crippen-logP</b> ist eine atombeitragsbasierte Vorhersage, "
        "kein gemessener Octanol/Wasser-Verteilungskoeffizient. "
        "<b>TPSA</b> ist eine topologische Fragmentgröße und kein Dipolmoment. "
        "<b>Rotierbare Bindungen</b> folgen der RDKit-Zählkonvention; "
        "Bewegungen endständiger Methylgruppen zählen dabei nicht automatisch mit. "
        "Ähnliche Modellwerte belegen keine gleichen "
        "experimentellen Eigenschaften.</p></details>"
    )
    pair_notes = {
        "D-Alanin": "Enantiomere besitzen in achiralen Umgebungen normalerweise gleiche "
                    "makroskopische Stoffwerte. D/L beschreibt nicht die Drehrichtung "
                    "des polarisierten Lichts. Der Unterschied ist die "
                    "entgegengesetzte Konfiguration, nicht eine freie Rotation.",
        "Glucose offen": "Die offenen Aldose-/Ketose-Formeln sind Strukturmodelle. "
                          "In wässriger Lösung überwiegen Ringformen; "
                          "die gezeigten SMILES kodieren keine Stereozentren. "
                          "Stoffdaten der reinen Glucose/Fructose dürfen nicht "
                          "als Daten der isolierten offenen Form eingesetzt werden.",
        "Maleinsäure": "Die cis/trans-Anordnung beeinflusst Kristallpackung, "
                       "Schmelzverhalten und Wasserlöslichkeit. "
                       "Die Stereomarker des SMILES sind zu erhalten.",
        "Butan": "Beide Alkane sind unpolar. Die andere Verzweigung beeinflusst "
                 "die Molekülgestalt und London-Wechselwirkungen, "
                 "nicht die Bildung von Wasserstoffbrücken.",
        "Propansäure": "Eine freie Carboxyl-OH-Gruppe erlaubt Protonenabgabe "
                       "und intermolekulare Wasserstoffbrücken. "
                       "Ein Ester besitzt keine freie Carboxyl-OH-Gruppe."
    }
    note = pair_notes.get(card_a["name"], "")
    note_html = (
        "<p style='margin:9px 0;color:#274d65;background:#eef7fa;"
        "border-left:3px solid #73bbca;padding:10px 12px;'>"
        "<b>Zum Vergleich:</b> " + esc(note) + "</p>"
    ) if note else ""
    return (
        "<section style='max-width:1040px;margin:12px 0;font-family:Arial,sans-serif;'>"
        "<h3 style='margin-bottom:10px;'>Direkter Struktur–Eigenschafts-Vergleich</h3>"
        "<div style='overflow-x:auto;border:1px solid #cbdfe8;border-radius:12px;'>"
        + header + "<tbody>" + "".join(main) + "</tbody></table></div>"
        + sources + note_html + extra +
        "<p style='color:#526d7e;font-size:0.81rem;'>"
        "Die frühere didaktische Polaritäts-/Löslichkeits-Score-Heuristik ist "
        "bewusst nicht Teil dieser Eigenschaftstabelle: Sie ist kein "
        "experimentell bestätigter Stoffwert.</p></section>"
    )


def _isomer_structure_widget(card, mol, highlight_atoms=None, highlight_bonds=None):
    """An in-place 2D / 3D / elemental-surface toggle for each comparison card.

    Uses a live widgets.HTML value, avoiding nested Output widgets which may not
    update in Colab. 3D is generated only when selected, then cached per card.
    """
    name = str(card["name"])
    smiles = str(card["smiles"])
    drawing = mol_png_data_uri(mol, 390, 265, highlight_atoms=highlight_atoms,
                               highlight_bonds=highlight_bonds)
    # Both modes occupy the same fixed-size display area; the molecule remains proportional.
    two_d = (
        "<div style='width:410px;max-width:100%;height:295px;"
        "display:flex;align-items:center;justify-content:center;background:white;'>"
        f"<img alt='2D-Struktur von {html.escape(name, quote=True)}'"
        f" src='{drawing}' style='max-width:100%;max-height:100%;"
        "width:auto;height:auto;object-fit:contain;display:block;' /></div>"
    )
    view = widgets.HTML(value=two_d, layout=widgets.Layout(
        width="410px", max_width="100%", min_height="295px"
    ))
    toggle = widgets.ToggleButtons(
        options=[("2D", "2d"), ("3D", "3d"),
                 ("Oberfläche (qualitativ)", "surface")],
        value="2d",
        description="", layout=widgets.Layout(width="410px", max_width="100%")
    )
    model_cache = {"mol": None}
    def refresh(change=None):
        selected = toggle.value
        if selected == "2d":
            view.value = two_d
            return
        if model_cache["mol"] is None:
            try:
                model_cache["mol"] = make_3d(mol)
            except Exception as exc:
                view.value = ("<p style='padding:12px;color:#a33;'>3D-Modell "
                              "nicht verfügbar: " + html.escape(str(exc)) + "</p>")
                return
        surface = selected == "surface"
        chosen_style = "Didaktische Polaritätsoberfläche" if surface else style_dd.value
        if chosen_style == "Didaktische Polaritätsoberfläche" and not surface:
            # For this explicit 3D tab, use a standard molecular representation.
            chosen_style = "Kugel-Stab"
        frame = mol3d_iframe_html(
            model_cache["mol"], width=390, height=265, style=chosen_style,
            mode=render_mode_dd.value
        )
        legend = (
            "<p style='font-size:0.78rem;line-height:1.4;color:#405a6b;"
            "margin:5px 1px 0;'>"
            "<b>Qualitative Elementoberfläche:</b> farbige Van-der-Waals-Oberfläche "
            "nach Atomsorten; zeigt heteroatomreiche und C/H-Bereiche. "
            "<b>Keine</b> berechnete elektrostatische Potentialfläche, "
            "<b>keine</b> quantitative Anzeige von positiv/negativem Potential."
            "</p>"
        ) if surface else (
            "<p style='font-size:0.76rem;color:#526d79;margin:4px 0;'>"
            "RDKit-Modellgeometrie, keine experimentelle Struktur.</p>"
        )
        view.value = frame + legend
    toggle.observe(refresh, names="value")
    return widgets.VBox([
        widgets.HTML(
            f"<div style='font-size:1.06rem;font-weight:700;color:#174d65'>"
            f"{html.escape(name)}</div>"
            "<div style='font-size:0.79rem;color:#607789;overflow-wrap:anywhere'>"
            f"SMILES: <code>{html.escape(smiles)}</code></div>"
        ),
        toggle, view
    ], layout=widgets.Layout(
        width="475px", max_width="100%", padding="10px",
        border="1px solid #d5e4eb"
    ))


def _isomer_pair_kind(name_a, name_b):
    names = {name_a, name_b}
    if names == {"D-Alanin", "L-Alanin"}:
        return "enantiomers"
    if names == {"Maleinsäure", "Fumarsäure"}:
        return "ez"
    return "constitutional"


def _stereo_pair_markings(mol, kind):
    """Relevant stereogenic atom or geometrically defined alkene, not all atoms."""
    if kind == "enantiomers":
        Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
        return [a.GetIdx() for a in mol.GetAtoms()
                if a.HasProp("_CIPCode")], []
    if kind == "ez":
        for bond in mol.GetBonds():
            if bond.GetStereo() in (Chem.BondStereo.STEREOE,
                                    Chem.BondStereo.STEREOZ):
                focus = [bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()]
                focus.extend(int(i) for i in bond.GetStereoAtoms())
                return sorted(set(focus)), [bond.GetIdx()]
    return [], []


def _stereo_pair_explanation(card_a, card_b, mol_a, mol_b):
    """Stereo category with D/L vs CIP and E/Z chemical qualifications."""
    kind = _isomer_pair_kind(card_a["name"], card_b["name"])
    if kind == "enantiomers":
        def cip(mol):
            Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
            return ", ".join(a.GetProp("_CIPCode") for a in mol.GetAtoms()
                             if a.HasProp("_CIPCode")) or "nicht definiert"
        return (
            "<h4>Vertiefung · Stereoisomerie: Enantiomere</h4>"
            "<p><b>D-Alanin (" + html.escape(cip(mol_a)) +
            ") ↔ L-Alanin (" + html.escape(cip(mol_b)) +
            ")</b>: Das markierte α-C ist stereogen. Beide Moleküle haben "
            "dieselbe Bindungsverknüpfung und sind nicht deckungsgleiche "
            "Spiegelbilder. D/L ist eine relative Konfigurationsangabe, "
            "R/S folgt den CIP-Prioritäten. D bedeutet nicht generell R "
            "und legt auch nicht das Vorzeichen der optischen Drehung fest.</p>"
            "<p>In achiraler Umgebung sind makroskopische Stoffwerte der "
            "Enantiomere gewöhnlich gleich; Unterschiede treten insbesondere "
            "bei Wechselwirkungen mit chiralen Partnern auf.</p>"
        )
    if kind == "ez":
        def ez(mol):
            return next((("Z" if b.GetStereo() == Chem.BondStereo.STEREOZ
                          else "E") for b in mol.GetBonds()
                         if b.GetStereo() in (Chem.BondStereo.STEREOZ,
                                              Chem.BondStereo.STEREOE)), "?")
        return (
            "<h4>Vertiefung · Stereoisomerie: E/Z-Geometrie</h4>"
            "<p><b>Maleinsäure (" + ez(mol_a) +
            ") ↔ Fumarsäure (" + ez(mol_b) +
            ")</b>: An der markierten C=C-Bindung ist die relative "
            "Stellung der Carboxylgruppen fixiert. Rotation an der "
            "Doppelbindung ist unter normalen Bedingungen nicht frei. "
            "Hier stimmen cis/trans und Z/E überein; generell ist "
            "E/Z die systematischere Bezeichnung.</p>"
        )
    return (
        "<p style='font-size:0.85rem;color:#506578;'>"
        "<b>Konstitutionsisomerie:</b> gleiche Summenformel, andere "
        "Bindungsverknüpfung. Gleiche RDKit-Deskriptoren bedeuten "
        "nicht automatisch identische Stoffeigenschaften.</p>"
    )


def display_isomer_comparison(card_a, card_b, style="Kugel-Stab"):
    """Place each 2D/3D toggle *above* one unified, compact comparison table."""
    mol_a = mol_from_smiles(card_a["smiles"])
    mol_b = mol_from_smiles(card_b["smiles"])
    props_a, props_b = calc_properties(mol_a), calc_properties(mol_b)
    groups_a, groups_b = detect_functional_groups(mol_a), detect_functional_groups(mol_b)
    display(HTML(
        "<h3 style='margin-bottom:6px'>Isomerenpaar: " +
        html.escape(card_a["name"]) + " ↔ " + html.escape(card_b["name"]) + "</h3>"
        "<p style='color:#405c6e;margin:3px 0 11px;'>Vergleiche die "
        "Bindungsverknüpfung und räumliche Anordnung. Schalte direkt am "
        "jeweiligen Molekül zwischen 2D, 3D und einer qualitativen "
        "Elementoberfläche um.</p>"
    ))
    kind = _isomer_pair_kind(card_a["name"], card_b["name"])
    display(HTML(_stereo_pair_explanation(card_a, card_b, mol_a, mol_b)))
    atoms_a, bonds_a = _stereo_pair_markings(mol_a, kind)
    atoms_b, bonds_b = _stereo_pair_markings(mol_b, kind)
    display(widgets.HBox([
        _isomer_structure_widget(card_a, mol_a, highlight_atoms=atoms_a,
                                 highlight_bonds=bonds_a),
        _isomer_structure_widget(card_b, mol_b, highlight_atoms=atoms_b,
                                 highlight_bonds=bonds_b)
    ], layout=widgets.Layout(
        display="flex", flex_flow="row wrap", width="100%",
        max_width="1040px", align_items="flex-start", gap="10px"
    )))
    display(HTML(_isomer_comparison_html(
        card_a, card_b, mol_a, mol_b, props_a, props_b, groups_a, groups_b
    )))
    display(HTML(
        "<p style='margin:11px 0;'><b>Arbeitsauftrag:</b> "
        "Formuliere die wichtigste strukturelle Gemeinsamkeit und den entscheidenden "
        "Unterschied. Beziehe experimentelle Stoffdaten ein und prüfe, "
        "welche berechneten Modellgrößen dabei helfen oder auch täuschen können.</p>"
    ))


def _display_py3dmol(viewer, width=520, height=400):
    display(HTML(_py3dmol_iframe_html(viewer, width=width, height=height)))

def show_conf(mol, conf_id, title="Konformer", style="Kugel-Stab", width=360, height=300):
    mb = Chem.MolToMolBlock(mol, confId=conf_id)
    display(HTML(f"<b>{title}</b><br>" + molblock_3d_html(mb, width=width, height=height, style=style, mode=render_mode_dd.value)))

def conf_iframe_html(mol, conf_id, title="Konformer", style="Kugel-Stab", width=360, height=300):
    mb = Chem.MolToMolBlock(mol, confId=conf_id)
    frame = molblock_3d_html(mb, width=width, height=height, style=style, mode=render_mode_dd.value)
    return f"<div style='flex:1 1 380px; min-width:330px; max-width:520px;'><b>{html.escape(title)}</b><br>" + frame + "</div>"

def display_conformers_side_by_side(mol, conformer_list, style="Kugel-Stab", max_n=4):
    """Konformere horizontal nebeneinander als getrennte, unabhängig drehbare 3D-Fenster."""
    boxes=[]
    for i,(cid,e) in enumerate(conformer_list[:max_n]):
        boxes.append(conf_iframe_html(mol, cid, f"Konformer {i+1}, ΔE = {e} kcal/mol" +
            (f" · {_conformer_geometry_label(mol, current_report.get('smiles', ''), cid)}"
             if _conformer_geometry_label(mol, current_report.get('smiles', ''), cid) else ""),
            style=style, width=340, height=280))
    display(HTML("<div style='display:flex; flex-wrap:wrap; gap:14px; align-items:flex-start;'>" + "".join(boxes) + "</div>"))

def interpret_properties(props):
    parts=[]
    logp=props.get("logP (Modellwert)",0)
    tpsa=props.get("TPSA / Å²",0)
    hbd=props.get("H-Brücken-Donoren",0)
    hba=props.get("H-Brücken-Akzeptoren",0)
    rot=props.get("rotierbare Bindungen",0)
    if hbd>0: parts.append("Das Molekül kann Wasserstoffbrücken spenden.")
    else: parts.append("Das Molekül kann keine Wasserstoffbrücken spenden.")
    if hba>0: parts.append("Es kann Wasserstoffbrücken aufnehmen.")
    if tpsa < 20: parts.append("Die berechnete polare Oberfläche ist klein; das spricht für eher unpolare Eigenschaften.")
    elif tpsa < 80: parts.append("Die berechnete polare Oberfläche ist mittelgroß; polare und unpolare Bereiche können nebeneinander vorkommen.")
    else: parts.append("Die berechnete polare Oberfläche ist groß; deutliche polare Bereiche sind zu erwarten.")
    if logp < 0: parts.append("Der logP-Wert spricht eher für hydrophile Eigenschaften.")
    elif logp < 3: parts.append("Der logP-Wert spricht für ein gemischtes Verhältnis zwischen hydrophilen und lipophilen Anteilen.")
    else: parts.append("Der logP-Wert spricht für ein eher lipophiles Molekül.")
    if rot >= 5: parts.append("Mehrere rotierbare Bindungen machen das Molekül vergleichsweise flexibel.")
    elif rot >= 1: parts.append("Das Molekül besitzt einige bewegliche Einfachbindungen.")
    else: parts.append("Das Molekül ist relativ starr.")
    return " ".join(parts)

def _conformer_dihedral(mol, smiles, cid):
    """Defined central torsion, meaningful for the two teaching examples only."""
    pattern = {
        "CCCC": "[C]-[C]-[C]-[C]",          # terminal C-C-C-C, n-butane
        "ClCCCl": "[Cl]-[C]-[C]-[Cl]",     # Cl-C-C-Cl
    }.get(smiles)
    if not pattern:
        return None
    query = Chem.MolFromSmarts(pattern)
    match = mol.GetSubstructMatch(query)
    if len(match) != 4:
        return None
    return float(rdMolTransforms.GetDihedralDeg(mol.GetConformer(cid), *match))


def _conformer_geometry_label(mol, smiles, cid):
    angle = _conformer_dihedral(mol, smiles, cid)
    if angle is None:
        return None
    magnitude = abs(angle)
    if magnitude >= 135:
        return f"anti ({angle:+.0f}°)"
    if 30 <= magnitude <= 110:
        return f"gauche ({angle:+.0f}°)"
    return f"andere Lage ({angle:+.0f}°)"


def generate_conformers(smiles, n=60):
    """Distinct optimized conformers based on geometry, not only energy gaps.

    ETKDG/MMFF or UFF explores local minima; no transition barriers/populations
    are calculated. Never mistake equal energies for identical spatial structures.
    """
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.pruneRmsThresh = -1.0
    try:
        params.useRandomCoords = True
    except Exception:
        pass
    ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n, params=params))
    results = []
    props = AllChem.MMFFGetMoleculeProperties(mol)
    for cid in ids:
        try:
            if props is not None:
                ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
            else:
                ff = AllChem.UFFGetMoleculeForceField(mol, confId=cid)
            ff.Minimize(maxIts=500)
            energy = ff.CalcEnergy()
        except Exception:
            try:
                ff = AllChem.UFFGetMoleculeForceField(mol, confId=cid)
                ff.Minimize(maxIts=500)
                energy = ff.CalcEnergy()
            except Exception:
                continue
        results.append((cid, energy))
    if not results:
        return mol, [], []
    minimum = min(e for _, e in results)
    rel = sorted(((cid, e-minimum) for cid,e in results), key=lambda x:x[1])

    # Keep geometrically different minima even when calculated energies match.
    # Compare heavy-atom geometries without mirror reflection; keep at most six.
    atom_map = [[(i, i) for i in range(mol.GetNumHeavyAtoms())]]
    unique = []
    for cid, e in rel:
        differs = True
        for uid, _ in unique:
            try:
                rms = rdMolAlign.GetBestRMS(
                    mol, mol, prbId=cid, refId=uid, map=atom_map, maxMatches=1
                )
                if rms < 0.32:
                    differs = False
                    break
            except Exception:
                # Energy separation remains a fallback if alignment fails.
                if abs(e - next(v for k,v in unique if k == uid)) < 0.08:
                    differs = False
                    break
        if differs:
            unique.append((cid,e))
        if len(unique) >= 6:
            break
    return mol, [(cid, round(e,2)) for cid,e in unique], [
        (cid,round(e,2)) for cid,e in rel
    ]


def card_for(module, name):
    for c in MOLECULES[module]:
        if c["name"] == name:
            return c
    return None

def _stoffdaten_lines(card, prefix=""):
    sd = card.get("stoffdaten") or {}
    lines=[]
    fields=[("Aggregatzustand", "aggregatzustand_raumtemperatur"),("Schmelzpunkt / °C", "schmelzpunkt_C"),("Siedepunkt / °C", "siedepunkt_C"),("Dichte / g cm⁻³", "dichte_g_cm3"),("Wasserlöslichkeit", "wasserloeslichkeit")]
    for label,key in fields:
        val=sd.get(key)
        if val not in (None,"",[]):
            lines.append(f"{prefix}- {label}: {val}")
    return lines

def report_text(rep):
    """Erzeugt einen einfachen kopierbaren SchülerInnen-Protokolltext."""
    lines=[]
    lines.append("MOL_LAB DIDACTIC – SchülerInnen-Protokoll")
    lines.append("="*44)
    lines.append("")
    rtype=rep.get("type", "molecule")
    if rtype == "molecule":
        card=rep.get("card", {})
        lines.append(f"Molekül: {rep.get('name','')}")
        lines.append(f"SMILES: {rep.get('smiles','')}")
        lines.append("")
        lines.append("Leitfrage:")
        lines.append(rep.get("leitfrage", ""))
        lines.append("")
        lines.append("Wichtige Modellwerte:")
        for k,v in rep.get("props",{}).items():
            lines.append(f"- {k}: {v}")
        lines.append("- Funktionelle Gruppen: " + (", ".join(rep.get('groups',[])) or "keine aus der einfachen Liste erkannt"))
        if rep.get("cyclohexane_geometries"):
            lines.append("")
            lines.append("Cyclohexan – gezielt erzeugte Ring-Referenzgeometrien:")
            for shape in rep["cyclohexane_geometries"]:
                lines.append(f"- {shape['name']}: ΔE(MMFF) = {shape['delta_e']:.2f} kcal/mol; {shape['classification']}")
            lines.append("Wanne = Sattel-/Übergangsgeometrie, kein stabiles Minimum.")
            lines.append("Die MMFF-Modellpotentiale sind keine experimentellen freien Energien oder Ringinversionsbarrieren.")
        if rep.get('assessment'):
            lines.append("- Didaktischer Polaritätseindruck: " + rep['assessment'].get('polarity_label',''))
            lines.append("- Wasserlöslichkeits-Tendenz: " + rep['assessment'].get('solubility_label',''))
            lines.append("- Kurzbegründung: " + rep['assessment'].get('reason',''))
        sd_lines=_stoffdaten_lines(card)
        if sd_lines:
            lines.append("")
            lines.append("Stoffdaten / experimenteller Bezug:")
            lines.extend(sd_lines)
        if _has_reactivity(card):
            r=card.get("reaktivitaet") or {}
            lines.append("")
            lines.append("Reaktivitäts-Hinweis:")
            lines.append("- Hinweis: " + _safe_text(r.get("hinweis")))
            lines.append("- Warum? " + _safe_text(r.get("warum")))
            lines.append("- Reaktionsidee: " + _safe_text(r.get("reaktionsidee")))
        tasks=card.get("mini_aufgaben") or (card.get("didaktik") or {}).get("mini_aufgaben") or []
        if tasks:
            lines.append("")
            lines.append("Mini-Aufgaben:")
            for i,t in enumerate(tasks,1): lines.append(f"{i}. {t}")
        lines.append("")
        lines.append("Erklärung / Kernaussage:")
        lines.append(rep.get("erklaerung", ""))
        lines.append("")
        lines.append("Modellgrenze:")
        lines.append(rep.get("modellgrenze", ""))
    else:
        pair=rep.get("pair", {})
        cards=rep.get("cards", [])
        lines.append("Vergleich: " + rep.get("title", ""))
        lines.append("")
        if pair.get("fehlvorstellung"):
            lines.append("Typische Fehlvorstellung:")
            lines.append(pair.get("fehlvorstellung", ""))
            lines.append("")
        lines.append("Leitfrage:")
        lines.append(pair.get("leitfrage") or rep.get("leitfrage", ""))
        lines.append("")
        for card, props, groups, assessment in zip(cards, rep.get("props", []), rep.get("groups", []), rep.get("assessments", [])):
            lines.append(f"Molekül: {card.get('name','')}")
            lines.append(f"SMILES: {card.get('smiles','')}")
            for key in ["Summenformel","Molmasse / g mol⁻¹","H-Brücken-Donoren","H-Brücken-Akzeptoren","logP (Modellwert)","TPSA / Å²"]:
                if key in props: lines.append(f"- {key}: {props[key]}")
            lines.append("- Funktionelle Gruppen: " + (", ".join(groups) or "keine aus der einfachen Liste erkannt"))
            if assessment:
                lines.append("- Polaritätseindruck: " + assessment.get("polarity_label", ""))
                lines.append("- Wasserlöslichkeits-Tendenz: " + assessment.get("solubility_label", ""))
            sd_lines=_stoffdaten_lines(card)
            if sd_lines:
                lines.extend(sd_lines)
            lines.append("")
        tasks=pair.get("mini_aufgaben") or []
        if tasks:
            lines.append("Mini-Aufgaben:")
            for i,t in enumerate(tasks,1): lines.append(f"{i}. {t}")
            lines.append("")
        if pair.get("experiment") or pair.get("experimenteller_bezug"):
            lines.append("Experimenteller Bezug:")
            lines.append(pair.get("experiment") or pair.get("experimenteller_bezug"))
            lines.append("")
        if pair.get("kernaussage"):
            lines.append("Kernaussage:")
            lines.append(pair.get("kernaussage"))
            lines.append("")
        if pair.get("modellgrenze"):
            lines.append("Modellgrenze:")
            lines.append(pair.get("modellgrenze"))
    return "\n".join(lines)


# 3D-Rendering-Aufräumen: bewusst deaktiviert.
# Aggressives Entfernen von 3D-iframes kann in Colab leere 3D-Anzeigen verursachen.
def cleanup_3d_frames():
    return

# Oberfläche
module_dd = widgets.Dropdown(options=list(MOLECULES.keys()), description="Modul:", layout=widgets.Layout(width="520px"))
example_dd = widgets.Dropdown(description="Beispiel:", layout=widgets.Layout(width="520px"))
custom_smiles = widgets.Text(value="", placeholder="optional eigener SMILES-Code, z.B. CCO", description="SMILES:", layout=widgets.Layout(width="700px"))
use_custom = widgets.Checkbox(value=False, description="eigenen SMILES verwenden")
style_dd = widgets.Dropdown(
    options=["Kugel-Stab", "Stabmodell", "Kalottenmodell", "Didaktische Polaritätsoberfläche"],
    value="Kugel-Stab",
    description="3D:",
    layout=widgets.Layout(width="420px")
)

render_mode_dd = widgets.Dropdown(
    options=[
        ("3Dmol.js leicht (empfohlen)", "custom"),
        ("py3Dmol iFrame (Fallback)", "py3dmol")
    ],
    value="custom",
    description="3D-Engine:",
    layout=widgets.Layout(width="350px")
)
run_btn = widgets.Button(description="Molekül analysieren", button_style="success")
answer_radio = widgets.RadioButtons(options=[], description="Antwort:")
# Hidden answer state; Colab native RadioButtons cannot reliably wrap long choices.
check_btn = widgets.Button(description="Antwort prüfen", button_style="info")
explain_btn = widgets.Button(description="Erklärung anzeigen")
report_btn = widgets.Button(description="SchülerInnen-Protokolltext erzeugen", layout=widgets.Layout(width="300px"))
conf_btn = widgets.Button(description="Konformere berechnen", button_style="warning")
_pair_seen = set()
ISOMER_PAIR_OPTIONS = [("— bitte Isomerenpaar auswählen —", None)]
for _name, _other in ISOMER_PAIRS.items():
    _pair = frozenset((_name, _other))
    if _pair not in _pair_seen:
        _pair_seen.add(_pair)
        _category = (" · Enantiomerie" if _name in ("D-Alanin", "L-Alanin") else
                     " · E/Z-Stereoisomerie" if _name in ("Maleinsäure", "Fumarsäure")
                     else " · Konstitutionsisomerie")
        ISOMER_PAIR_OPTIONS.append((f"{_name} ↔ {_other}{_category}", _name))
isomer_pair_dd = widgets.Dropdown(
    options=ISOMER_PAIR_OPTIONS,
    value=None,
    description="Isomerenpaar:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="min(100%, 590px)")
)
single_isomer_toggle = widgets.Checkbox(
    value=False, description="Optional: Einzelmolekül oder eigene SMILES untersuchen",
    indent=False, layout=widgets.Layout(width="auto")
)
polar_pair_dd = widgets.Dropdown(
    options=[("— bitte Vergleichspaar wählen —", None)] + [(k, k) for k in POLARITY_PAIRS.keys()],
    value=None,
    description="Vergleichspaar:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="min(100%, 620px)")
)
glucose_btn = widgets.Button(description="Glucose: Ring–Ketten-Gleichgewicht", button_style="info")
reset3d_btn = widgets.Button(description="3D-Anzeige neu aufbauen", layout=widgets.Layout(width="230px"))
out = widgets.Output()
feedback_out = widgets.Output()
# Inline feedback is visible beside the MC buttons, not above the comparison.
feedback_html = widgets.HTML(value="", layout=widgets.Layout(width="100%"))
report_out = widgets.Output()
protocol_out = widgets.Output()


def _quiz_answer_rows(answers):
    """Four independent, wrapping answer rows; exclusive selection in Colab.

    Native ipywidgets.RadioButtons collapse multi-line labels in some Colab
    themes, overlapping answers and radio circles. Each answer is therefore
    shown next to its own checkbox, while answer_radio stores the selection
    for the unchanged checking/explanation functions.
    """
    answer_radio.options = [("— bitte auswählen —", None)] + [(a, a) for a in answers]
    answer_radio.value = None
    selectors, rows = [], []
    for answer in answers:
        selector = widgets.Checkbox(
            value=False, description="", indent=False,
            layout=widgets.Layout(width="32px", min_width="32px", margin="0")
        )
        label = widgets.HTML(
            value=("<div style='white-space:normal;overflow-wrap:anywhere;"
                   "line-height:1.45;font-size:0.95rem;'>" +
                   html.escape(str(answer)) + "</div>"),
            layout=widgets.Layout(width="auto", min_width="0", flex="1 1 auto")
        )
        selectors.append(selector)
        rows.append(widgets.HBox(
            [selector, label],
            layout=widgets.Layout(
                width="100%", align_items="center", gap="8px",
                padding="7px 10px", border="1px solid #dbe5ec"
            )
        ))
    for i, selector in enumerate(selectors):
        def on_select(change, index=i):
            if change["name"] != "value":
                return
            if change["new"]:
                for other_index, other in enumerate(selectors):
                    if other_index != index and other.value:
                        other.value = False
                answer_radio.value = answers[index]
            elif answer_radio.value == answers[index]:
                answer_radio.value = None
        selector.observe(on_select, names="value")
    return widgets.VBox(rows, layout=widgets.Layout(width="100%", max_width="900px", gap="5px"))


def refresh_examples(*args):
    """Keep single-molecule controls in other modules and pair-first in Isomerie."""
    global current_report
    current_report = {}
    module = module_dd.value
    names = [c["name"] for c in MOLECULES[module]]
    example_dd.options = []
    example_dd.options = tuple(names)
    if names:
        example_dd.value = names[0]

    isomer_mode = module == "Isomerie"
    isomer_section.layout.display = "" if isomer_mode else "none"
    single_isomer_toggle.layout.display = "" if isomer_mode else "none"
    single_section.layout.display = (
        "" if (not isomer_mode or single_isomer_toggle.value) else "none"
    )
    polar_section.layout.display = "" if module == "Polarität und Löslichkeit" else "none"
    polar_pair_dd.value = None
    out.clear_output(); feedback_out.clear_output()
    report_out.clear_output(); protocol_out.clear_output()
    if isomer_mode:
        # A module choice must NOT silently launch the first curated pair.
        if isomer_pair_dd.value is not None:
            isomer_pair_dd.value = None
        with report_out:
            clear_output(wait=True)
            display(HTML("<p style='padding:12px;border-left:3px solid #71b7c4;"
                         "background:#f1f8fb;'>Wähle oben bewusst ein Isomerenpaar. "
                         "Der Vergleich startet direkt nach der Auswahl.</p>"))

def update_single_isomer_controls(change=None):
    if module_dd.value == "Isomerie":
        single_section.layout.display = "" if single_isomer_toggle.value else "none"
        if single_isomer_toggle.value and isomer_pair_dd.value in example_dd.options:
            example_dd.value = isomer_pair_dd.value

def refresh_isomer_pair(change):
    """Launch a selected curated pair immediately; keep module entry empty."""
    global current_report
    if change["name"] != "value" or module_dd.value != "Isomerie":
        return
    current_report = {}
    feedback_html.value = ""
    feedback_out.clear_output()
    if change["new"] is None:
        cleanup_3d_frames()
        with report_out:
            clear_output(wait=True)
            display(HTML("<p style='padding:10px;background:#f1f8fb;"
                         "border-left:3px solid #71b7c4;'>"
                         "Bitte ein Isomerenpaar auswählen.</p>"))
        return
    do_isomer_comparison()

# Observers are registered AFTER the control containers have been created below.
def refresh_polar_pair(change):
    """Paarwechsel startet direkt; bloße Modulauswahl bleibt ohne Paar."""
    global current_report
    if change["name"] != "value" or module_dd.value != "Polarität und Löslichkeit":
        return
    current_report = {}
    feedback_html.value = ""
    feedback_out.clear_output()
    if change["new"] is not None:
        do_polarity_comparison()
    else:
        with report_out:
            clear_output(wait=True)


def reset_pair_on_example_change(*args):
    if module_dd.value == "Polarität und Löslichkeit":
        polar_pair_dd.value = None
    # A single-molecule switch should not delete the guided pair comparison.
    if module_dd.value != "Isomerie":
        report_out.clear_output()

def analyze(_=None):
    global current_report
    if module_dd.value == "Polarität und Löslichkeit" and polar_pair_dd.value is not None:
        polar_pair_dd.value = None
    cleanup_3d_frames()
    if module_dd.value == "Isomerie" and single_isomer_toggle.value:
        # An optional individual analysis replaces the former pair result.
        report_out.clear_output()
        feedback_out.clear_output()
    with out:
        clear_output()
        gc.collect()
        module=module_dd.value
        if use_custom.value:
            smiles=custom_smiles.value.strip()
            if not smiles:
                print("Bitte einen SMILES-Code eingeben oder das Häkchen entfernen."); return
            card={"name":"Eigenes Molekül", "smiles":smiles, "leitfrage":"Welche Strukturmerkmale könnten die Stoffeigenschaften beeinflussen?", "beobachten":["Suche polare Gruppen.","Achte auf die Molekülgröße.","Vergleiche logP, TPSA und H-Brücken."], "frage":"Welche Aussage ist bei eigenen Molekülen besonders wichtig?", "antworten":["Die berechneten Werte sind Modellwerte und müssen fachlich geprüft werden.","SMILES-Eingaben sind immer experimentelle Messwerte.","Die Summenformel allein reicht immer aus.","Große Moleküle sind in dieser App immer zuverlässig berechnet."], "richtig":0, "feedback":"Richtig: Die App gibt Modellhinweise, keine endgültigen Messwerte.", "erklaerung":"Eigene SMILES-Eingaben sind für kleine bis mittelkleine Moleküle gedacht. Prüfe immer, ob die Struktur chemisch sinnvoll ist.", "modellgrenze":"Für große, geladene oder ungewöhnliche Moleküle ist diese einfache RDKit-Analyse nur eingeschränkt geeignet."}
        else:
            card=card_for(module, example_dd.value)
            smiles=card["smiles"]
        try:
            mol=mol_from_smiles(smiles)
        except Exception as e:
            print("Fehler:", e); return
        props=calc_properties(mol)
        groups=detect_functional_groups(mol)
        assessment=(qualitative_polarity_assessment(mol, props)
                    if module == "Polarität und Löslichkeit" else None)
        interp=interpret_properties(props)
        display(HTML(f"<h2>{card['name']}</h2><p><b>SMILES:</b> <code>{smiles}</code></p>"))
        display(HTML(f"<h3>Leitfrage</h3><p>{card.get('leitfrage','')}</p>"))
        display(HTML("<h3>Beobachte</h3><ul>" + "".join(f"<li>{x}</li>" for x in card.get('beobachten',[])) + "</ul>"))
        mol3d = make_3d(mol)
        display_structure_panel(mol, mol3d, props=props, groups=groups, style=style_dd.value)
        dipole_id = dipole_case_id(module, card, mol, use_custom.value)
        if dipole_id:
            display(dipole_learning_widget(mol3d, dipole_id))
        if module == "Wirkstoffe und Biomoleküle" and card.get("id") == "atorvastatin":
            Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
            cip_codes = [a.GetProp("_CIPCode") for a in mol.GetAtoms()
                         if a.HasProp("_CIPCode")]
            display(HTML(
                "<div style='max-width:1040px;border-left:4px solid #6d8295;"
                "padding:11px 15px;background:#f3f7f9;line-height:1.5;'>"
                "<b>Komplexität auf einen Blick:</b> "
                f"{mol.GetNumHeavyAtoms()} Schweratome · "
                f"{mol.GetRingInfo().NumRings()} Ringe · "
                f"{rdMolDescriptors.CalcNumRotatableBonds(mol)} "
                f"RDKit-rotierbare Bindungen · {len(cip_codes)} "
                f"definierte Stereozentren ({html.escape(', '.join(cip_codes))}). "
                "<br><small>Eine einzige RDKit-Kraftfeldgeometrie ist "
                "keine experimentelle Bindungsstruktur und keine "
                "Konformeren- oder Wirkungsprognose.</small></div>"
            ))
        display(HTML("<h3>Erste Modell-Deutung</h3><p>" + interp + "</p>"))
        display(HTML(_reactivity_box_html(card)))
        answers = list(card.get("antworten", []))
        random.shuffle(answers)  # correct answer must not always be first
        quiz_rows = _quiz_answer_rows(answers)
        display(HTML("<h3>Vermute</h3><p>" + card.get('frage','') + "</p>"))
        feedback_html.value = ""
        display(widgets.VBox([quiz_rows, widgets.HBox([check_btn, explain_btn]),
                             feedback_html]))
        current_report={"type":"molecule", "name":card["name"], "smiles":smiles, "card":card, "props":props, "groups":groups, "assessment":assessment, "leitfrage":card.get("leitfrage",""), "interpretation":interp, "erklaerung":card.get("erklaerung",""), "modellgrenze":card.get("modellgrenze","")}
    feedback_out.clear_output(); report_out.clear_output(); protocol_out.clear_output()

def check_answer(_=None):
    """Feedback next to the active multiple-choice question (Colab-compatible)."""
    quiz = (current_report.get("quiz") or current_report.get("card")) if current_report else None
    if not quiz:
        feedback_html.value = "<p>Bitte zuerst ein Molekül oder ein Paar analysieren.</p>"
        return
    answers = list(quiz.get("antworten") or [])
    selected = answer_radio.value
    if selected in (None, "— bitte auswählen —"):
        feedback_html.value = (
            "<p role='status' style='border-left:3px solid #c49439;padding:8px 11px;'>"
            "Bitte zuerst eine Antwort auswählen.</p>"
        )
        return
    if selected not in answers:
        feedback_html.value = (
            "<p role='status' style='color:#a1212b;'>Die gewählte Antwort gehört "
            "nicht mehr zum aktuellen Vergleich. Bitte erneut wählen.</p>"
        )
        return
    if answers.index(selected) == quiz.get("richtig", 0):
        feedback_html.value = (
            "<p role='status' style='padding:9px 12px;background:#eff8f0;"
            "border-left:3px solid #3c945f;'><b>Richtig.</b> " +
            html.escape(str(quiz.get("feedback", ""))) + "</p>"
        )
    else:
        right = answers[quiz["richtig"]]
        feedback_html.value = (
            "<p role='status' style='padding:9px 12px;background:#fff6f5;"
            "border-left:3px solid #bb4c4c;'><b>Noch nicht ganz.</b> "
            "Die beste Antwort ist: <b>" + html.escape(str(right)) +
            "</b></p><p>" + html.escape(str(quiz.get("feedback", ""))) + "</p>"
        )

def explain(_=None):
    """Explanation is placed directly below the quiz's active buttons."""
    if not current_report:
        feedback_html.value = "<p>Bitte zuerst ein Molekül oder ein Paar analysieren.</p>"
        return
    card = current_report.get("card") or {}
    pair = current_report.get("pair") or {}
    quiz = current_report.get("quiz") or {}
    text = (current_report.get("erklaerung") or quiz.get("erklaerung") or
            quiz.get("feedback") or pair.get("kernaussage") or "")
    limit = current_report.get("modellgrenze") or pair.get("modellgrenze") or ""
    feedback_html.value = (
        "<section role='status' style='padding:11px 13px;border-radius:9px;"
        "background:#f0f7fb;border:1px solid #d4e5ef;'>"
        "<h4 style='margin:0 0 7px'>Fachliche Erklärung</h4><p>" +
        html.escape(str(text)) + "</p>" +
        _reactivity_box_html(card, open_by_default=True) +
        ("<p><b>Grenze des Modells:</b> "+html.escape(str(limit))+"</p>"
         if limit else "") + "</section>"
    )

def show_report(_=None):
    with protocol_out:
        clear_output()
        if not current_report:
            print("Bitte zuerst ein Molekül analysieren oder ein Vergleichspaar anzeigen.")
            return
        txt=report_text(current_report)
        display(HTML("<h3>SchülerInnen-Protokolltext</h3><p>Der Text kann kopiert und in ein Heft-/Lernplattform-Protokoll übernommen werden.</p>"))
        display(widgets.Textarea(value=txt, layout=widgets.Layout(width="100%", height="300px")))

def do_conformers(_=None):
    cleanup_3d_frames()
    with report_out:
        clear_output()
        if not current_report:
            print("Bitte zuerst ein Molekül analysieren."); return
        smiles=current_report["smiles"]
        if (current_report.get("type") == "molecule" and
                current_report.get("name") == "Cyclohexan" and
                smiles == "C1CCCCC1"):
            try:
                cyclo_models = display_cyclohexane_reference_comparison()
            except Exception as exc:
                print("Cyclohexan-Referenzvergleich konnte nicht erstellt werden:", exc)
                return
            current_report["cyclohexane_geometries"] = [
                {k: m[k] for k in ("name", "delta_e", "classification")}
                for m in cyclo_models
            ]
            return
        mol0 = Chem.MolFromSmiles(smiles)
        if mol0 is None or mol0.GetNumHeavyAtoms() > 35:
            print("Konformer-Suche nur für kleine bis mittelgroße Moleküle empfohlen."); return
        display(HTML("<h3>Konformere: einfache RDKit/MMFF-Suche</h3><p>Relative MMFF-/UFF-Energien sind Modellwerte in kcal/mol. Geometrisch unterschiedliche optimierte Konformere werden verglichen, auch wenn ihre Energien ähnlich sind. Bei Butan und 1,2-Dichlorethan wird der zentrale Diederwinkel angegeben. Ekliptische Lagen sind bei einer gewöhnlichen Minimierung keine stabilen Minima. Eine Barriere oder Boltzmann-Verteilung wird hier nicht berechnet.</p>"))
        mol, unique_rel, all_rel = generate_conformers(smiles, n=120)
        if not unique_rel:
            print("Keine Konformere erzeugt."); return
        display(pd.DataFrame([{
            "Konformer": i+1,
            "ΔE / kcal mol⁻¹ (Modell)": e,
            "zentraler Diederwinkel": _conformer_geometry_label(mol, smiles, cid) or
                                   "für dieses Molekül nicht kuratiert"
        } for i,(cid,e) in enumerate(unique_rel[:6])]))
        if len(unique_rel) <= 2:
            display(HTML("<p><b>Hinweis:</b> Eine Konformersuche zeigt hier nur wenige geometrisch verschiedene optimierte Minima. Die Ergebnisse hängen von Stichprobe und Kraftfeld ab; rotierende Moleküle können weitere Zustände besitzen.</p>"))
        display(HTML("<h4>Vergleichsansicht</h4><p>Die Konformere werden nebeneinander in getrennten 3D-Fenstern angezeigt und können unabhängig voneinander gedreht werden.</p>"))
        display_conformers_side_by_side(mol, unique_rel, style=style_dd.value, max_n=4)



def _build_cyclohexane_reference(shape):
    """Deterministic MMFF94 ring geometry from a named puckering template.

    The boat is a symmetry-preserving stationary/saddle geometry, not
    an energy minimum. The half-chair is a position-restrained approximate
    transition-like reference, NOT an independently validated saddle point.
    """
    if shape not in ("Sessel", "Wanne", "Twist-Boat", "Halbsessel"):
        raise ValueError("Unbekannte Cyclohexan-Ringform.")
    base = Chem.MolFromSmiles("C1CCCCC1")
    conf = Chem.Conformer(6)
    radius, amplitude = 1.44, 0.35
    for index in range(6):
        azimuth = 2.0 * math.pi * index / 6.0
        if shape == "Sessel":
            z = amplitude * math.cos(3.0 * azimuth)
        elif shape == "Halbsessel":
            # Four near-coplanar ring carbons and two opposite flaps.
            # This is a geometry-constrained proxy, NOT a located transition state.
            z = (0.0, 0.0, 0.0, 0.0, amplitude, -amplitude)[index]
        else:
            phase = 0.0 if shape == "Wanne" else math.pi / 6.0
            z = amplitude * math.cos(2.0 * azimuth + phase)
        conf.SetAtomPosition(
            index, (radius * math.cos(azimuth),
                    radius * math.sin(azimuth), z)
        )
    base.AddConformer(conf, assignId=True)
    mol = Chem.AddHs(base, addCoords=True)
    mmff = AllChem.MMFFGetMoleculeProperties(mol)
    if mmff is None:
        raise RuntimeError("Für Cyclohexan sind keine MMFF-Parameter verfügbar.")
    # Relax hydrogen placement while preserving the chosen ring-pucker template.
    constrained = AllChem.MMFFGetMoleculeForceField(mol, mmff)
    for atom_id in range(6):
        constrained.MMFFAddPositionConstraint(atom_id, 0.03, 2000.0)
    if constrained.Minimize(maxIts=900) != 0:
        raise RuntimeError("Konformer-Vorbereitung nicht konvergiert.")
    # A full minimization collapses the half-chair towards a true minimum.
    # Keep ring-C positional restraints ONLY for this teaching reference.
    # The energy is the *unrestrained* MMFF potential evaluated at the
    # restrained geometry (NOT an optimized saddle-point barrier).
    ff = AllChem.MMFFGetMoleculeForceField(mol, mmff)
    if shape != "Halbsessel" and ff.Minimize(maxIts=900) != 0:
        raise RuntimeError("MMFF-Referenzgeometrie nicht konvergiert.")
    energy = float(ff.CalcEnergy())
    ring = mol.GetConformer()
    torsions = [
        float(rdMolTransforms.GetDihedralDeg(
            ring, *((i + j) % 6 for j in range(4))
        )) for i in range(6)
    ]
    absolute = sorted(abs(v) for v in torsions)
    if shape == "Sessel":
        plausible = absolute[0] > 40.0
    elif shape == "Wanne":
        plausible = absolute[0] < 10.0 and absolute[1] < 10.0 and absolute[-1] > 35.0
    elif shape == "Halbsessel":
        plausible = absolute[0] < 12.0 and absolute[-1] > 50.0
    else:
        plausible = 15.0 < absolute[0] < 40.0 and absolute[-1] > 45.0
    if not plausible:
        raise RuntimeError(
            "Die berechnete Ringform entspricht nicht der gewünschten "
            "Referenzgeometrie: " + shape
        )
    return mol, energy, torsions


def cyclohexane_reference_models():
    """Exactly four curated educational ring geometries in one MMFF model."""
    geometries = {
        shape: _build_cyclohexane_reference(shape)
        for shape in ("Sessel", "Twist-Boat", "Wanne", "Halbsessel")
    }
    chair_energy = geometries["Sessel"][1]
    labels = {
        "Sessel": ("globales Energieminimum (Modell)",
                   "Nahezu gestaffelte C–H-Bindungen; geringe Winkel- und Torsionsspannung."),
        "Twist-Boat": ("höheres lokales Minimum (Modell)",
                       "Gegenüber der Wanne verdrillt; weniger ekliptische Wechselwirkungen."),
        "Wanne": ("höherenergetische Sattel-/Übergangsgeometrie",
                  "Zwei ungünstige Flaggenmast-H-Nähen und ekliptische Bindungen."),
        "Halbsessel": ("fixierte Annäherung an eine Übergangslage (kein Minimum)",
                       "Vier annähernd koplanare Ring-C-Atome; zwei benachbarte Ring-C "
                       "liegen auf verschiedenen Seiten. Hohe Torsions- und Winkelspannung.")
    }
    models = []
    for shape in ("Sessel", "Twist-Boat", "Wanne", "Halbsessel"):
        mol, energy, torsions = geometries[shape]
        models.append({
            "name": shape, "mol": mol, "energy": energy,
            "delta_e": round(energy - chair_energy, 2),
            "torsions": torsions,
            "geometry_restrained": shape == "Halbsessel",
            "classification": labels[shape][0],
            "description": labels[shape][1]
        })
    if not (models[0]["delta_e"] == 0.0 and
            0.0 < models[1]["delta_e"] < models[2]["delta_e"] <
            models[3]["delta_e"]):
        raise RuntimeError("Die erwartete MMFF-Energiereihenfolge wurde nicht bestätigt.")
    return models


def display_cyclohexane_reference_comparison():
    """Curated cyclohexane teaching case, independent from random minimum search."""
    models = cyclohexane_reference_models()
    display(HTML(
        "<h3>Cyclohexan: Sessel · Twist-Boat · Wanne · Halbsessel</h3>"
        "<p><b>Warum vier Formen?</b> Die vier 3D-Referenzen werden "
        "aus gezielt vorgegebenen Ringverformungen erzeugt. Sessel und "
        "Twist-Boat sind Minima; die Wanne ist eine Sattelgeometrie. "
        "Der Halbsessel wird geometrisch eingeschränkt dargestellt, "
        "weil eine gewöhnliche Minimierung ihn nicht erhält.</p>"
    ))
    blocks = []
    for model in models:
        caption = (
            model["name"] + " · ΔE(MMFF94) = " +
            f'{model["delta_e"]:.2f}' + " kcal/mol"
        )
        blocks.append(
            conf_iframe_html(
                model["mol"], model["mol"].GetConformer().GetId(),
                title=caption, style=style_dd.value,
                width=325, height=260
            )
        )
    display(HTML(
        "<div style='display:flex;flex-wrap:wrap;gap:12px;"
        "align-items:flex-start;max-width:1100px;'>"
        + "".join(blocks) + "</div>"
    ))
    table = pd.DataFrame([{
        "Ringform": model["name"],
        "relatives MMFF-Potential / kcal mol⁻¹": model["delta_e"],
        "Einordnung": model["classification"]
    } for model in models])
    with pd.option_context("display.max_colwidth", None):
        table_html = table.to_html(index=False, escape=True)
    display(HTML(
        "<style>.cyclo-energy-table td, .cyclo-energy-table th "
        "{white-space:normal;overflow-wrap:anywhere;text-align:left;max-width:none;}</style>"
        "<div class='cyclo-energy-table' style='max-width:100%;overflow-x:auto;'>"
        + table_html + "</div>"
    ))
    display(HTML(
        "<details style='margin:10px 0;max-width:1050px;'>"
        "<summary><b>Was zeigen die vier Ringformen?</b></summary>"
        "<p><b>Sessel:</b> " + models[0]["description"] + "</p>"
        "<p><b>Twist-Boat:</b> " + models[1]["description"] + "</p>"
        "<p><b>Wanne:</b> " + models[2]["description"] + "</p>"
        "<p><b>Halbsessel:</b> " + models[3]["description"] + "</p>"
        "<p><b>Wichtige Modellgrenze:</b> Die Wanne ist im "
        "Ringinversionsprofil eine stationäre Sattel-/Übergangsgeometrie, "
        "<b>kein drittes stabiles Minimum</b>. Der Halbsessel ist "
        "eine gezielt <b>geometrisch eingeschränkte Annäherung</b> "
        "an die hochenergetische Übergangslage; es wurde kein exakter "
        "Sattelpunkt durch Frequenz-/Hesse-Analyse nachgewiesen. "
        "Für den Halbsessel wird das MMFF-Potential an dieser "
        "eingeschränkten Geometrie ausgewertet. Die Zahlen sind "
        "keine freien Energien, keine Gleichgewichtsanteile und "
        "<b>keine berechnete Ringinversionsbarriere</b>.</p>"
        "<p><b>Ringinversion:</b> Sessel → Halbsessel → Twist-Boat "
        "→ Wanne → Twist-Boat → Halbsessel → Sessel.</p>"
        "<p>Literaturorientierung (andere Methoden/Bedingungen): "
        "Sessel 0, Twist-Boat ungefähr +5,5, Wanne ungefähr "
        "+6,9 und Halbsessel ungefähr +10,8 kcal/mol. "
        "Die hier ausgewiesenen Zahlen stammen "
        "ausschließlich aus dem konkreten MMFF-Modell.</p>"
        "</details>"
    ))
    return models


def _comparison_assessment_label(mol, props):
    a = qualitative_polarity_assessment(mol, props)
    pol_labels = ["sehr schwach", "schwach", "mittel", "stark", "sehr stark"]
    sol_labels = ["kaum wasserlöslich", "begrenzt", "mäßig", "gut", "sehr gut"]
    return pol_labels[a["polarity_index"]], sol_labels[a["solubility_index"]]


def display_polarity_comparison(pair_key, style="Kugel-Stab"):
    """Kuratierter Paarvergleich mit zwei kompakten, umschaltbaren Strukturkarten."""
    pair = POLARITY_PAIRS[pair_key]
    card_a = card_for("Polarität und Löslichkeit", pair["a"])
    card_b = card_for("Polarität und Löslichkeit", pair["b"])
    if not card_a or not card_b:
        raise ValueError("Das Vergleichspaar ist in der Moleküldatenbank unvollständig.")
    mol_a, mol_b = mol_from_smiles(card_a["smiles"]), mol_from_smiles(card_b["smiles"])
    props_a, props_b = calc_properties(mol_a), calc_properties(mol_b)
    groups_a = detect_functional_groups(mol_a)
    groups_b = detect_functional_groups(mol_b)
    display(HTML(
        "<h3>Vergleichspaar: " + html.escape(pair_key) + "</h3>"
        "<p><b>Leitfrage:</b> " + html.escape(str(pair.get("leitfrage", ""))) + "</p>"
        "<details style='margin:5px 0 11px;'><summary>Typische Fehlvorstellung ansehen</summary>"
        "<p>" + html.escape(str(pair.get("fehlvorstellung", ""))) + "</p></details>"
    ))
    display(widgets.HBox([
        _isomer_structure_widget(card_a, mol_a),
        _isomer_structure_widget(card_b, mol_b)
    ], layout=widgets.Layout(
        display="flex", flex_flow="row wrap", width="100%",
        max_width="1040px", align_items="flex-start", gap="10px"
    )))
    display(HTML(_isomer_comparison_html(
        card_a, card_b, mol_a, mol_b, props_a, props_b, groups_a, groups_b
    )))
    display(HTML(
        "<p style='font-size:0.84rem;color:#435c70;max-width:1040px;'>"
        "<b>Zur Oberfläche:</b> Elementfarben auf einer Van-der-Waals-Fläche "
        "sind weder Dipolvektoren noch elektrostatisches Potential und "
        "belegen für sich keine Wasserlöslichkeit.</p>"
    ))
    tasks = pair.get("mini_aufgaben") or []
    if tasks:
        display(HTML("<h4>Mini-Aufgaben</h4><ol>" +
            "".join("<li>" + html.escape(str(x)) + "</li>" for x in tasks) + "</ol>"))
    if pair.get("experiment"):
        display(HTML(
            "<details style='margin:8px 0'><summary><b>Experimenteller Bezug anzeigen</b></summary>"
            "<p>" + html.escape(str(pair["experiment"])) + "</p></details>"
        ))
    display(HTML(
        "<details style='margin:8px 0'><summary><b>Lösungsvorschlag / Kernaussage</b></summary>"
        "<p>" + html.escape(str(pair.get("kernaussage", ""))) + "</p>"
        "<p><b>Modellgrenze:</b> " +
        html.escape(str(pair.get("modellgrenze", ""))) + "</p></details>"
    ))


def show_comparison_quiz(quiz):
    """Zeigt eine MC-Frage zum aktiven Vergleich, ohne eine Antwort vorzugeben."""
    answers=list(quiz.get("antworten") or [])
    if not quiz.get("frage") or len(answers)<2:
        return
    random.shuffle(answers)
    quiz_rows = _quiz_answer_rows(answers)
    display(HTML("<h4>Überprüfe deine Deutung</h4><p>" + html.escape(str(quiz["frage"])) + "</p>"))
    feedback_html.value = ""
    display(widgets.VBox([quiz_rows, widgets.HBox([check_btn, explain_btn]),
                         feedback_html]))

def do_polarity_comparison(_=None):
    """Explizite Paarauswahl startet sofort, Einzelanalyse bleibt unabhängig."""
    global current_report
    current_report = {}
    cleanup_3d_frames()
    with report_out:
        clear_output(wait=True)
        answer_radio.options = []
        feedback_html.value = ""
        feedback_out.clear_output()
        if module_dd.value != "Polarität und Löslichkeit":
            print("Der Paarvergleich gehört zum Modul Polarität und Löslichkeit.")
            return
        key = polar_pair_dd.value
        if not key:
            return
        pair = POLARITY_PAIRS[key]
        card_a = card_for("Polarität und Löslichkeit", pair["a"])
        card_b = card_for("Polarität und Löslichkeit", pair["b"])
        if not card_a or not card_b:
            print("Die Einträge für dieses Vergleichspaar sind unvollständig.")
            return
        try:
            display_polarity_comparison(key, style=style_dd.value)
        except Exception as exc:
            print("Polaritätsvergleich konnte nicht dargestellt werden:", exc)
            return
        out.clear_output()
        mol_a, mol_b = mol_from_smiles(card_a["smiles"]), mol_from_smiles(card_b["smiles"])
        props_a, props_b = calc_properties(mol_a), calc_properties(mol_b)
        current_report = {
            "type": "polarity_pair", "title": key, "pair": pair,
            "cards": [card_a, card_b], "props": [props_a, props_b],
            "groups": [detect_functional_groups(mol_a), detect_functional_groups(mol_b)],
            "assessments": [None, None]
        }
        if pair.get("quiz"):
            current_report["quiz"] = pair["quiz"]
            show_comparison_quiz(pair["quiz"])
        else:
            display(HTML("<p>Für dieses Paar fehlt noch eine kuratierte MC-Frage.</p>"))


def do_isomer_comparison(_=None):
    """Pair-first isomer display. No hidden dependence on single-molecule selection."""
    global current_report
    current_report = {}
    cleanup_3d_frames()
    with report_out:
        clear_output(wait=True)
        answer_radio.options = []
        feedback_html.value = ""
        feedback_out.clear_output()
        if module_dd.value != "Isomerie":
            print("Der Paarvergleich ist nur im Modul Isomerie aktiv.")
            return
        name_a = isomer_pair_dd.value
        name_b = ISOMER_PAIRS.get(name_a)
        if not name_b:
            print("Bitte ein kuratiertes Isomerenpaar auswählen.")
            return
        card_a = card_for("Isomerie", name_a)
        card_b = card_for("Isomerie", name_b)
        if not card_a or not card_b:
            print("Das Isomerenpaar ist in den DIDACTIC-Daten nicht vollständig hinterlegt.")
            return
        out.clear_output()  # Remove stale single-molecule analysis from a previous module.
        try:
            display_isomer_comparison(card_a, card_b, style=style_dd.value)
        except Exception as exc:
            print("Isomerenvergleich konnte nicht dargestellt werden:", exc)
            return
        mol_a = mol_from_smiles(card_a["smiles"])
        mol_b = mol_from_smiles(card_b["smiles"])
        props_a, props_b = calc_properties(mol_a), calc_properties(mol_b)
        current_report = {
            "type": "isomer_pair",
            "title": card_a["name"] + " vs. " + card_b["name"],
            "pair": {
                "leitfrage": "Wie beeinflusst die andere Anordnung der Atome die Eigenschaften?",
                "mini_aufgaben": [
                    "Vergleiche Summenformel und 2D-Struktur.",
                    "Finde die relevanten Unterschiede in Stoffdaten und Reaktivität.",
                    "Erläutere, welche Modellwerte keine Unterscheidung erlauben."
                ],
                "kernaussage": "Isomere können trotz gleicher Summenformel unterschiedliche Eigenschaften und Reaktivitäten aufweisen.",
                "modellgrenze": "RDKit-logP, TPSA und Rotationszahlen sind definitionsabhängige Modellgrößen; die einfache Polaritätsheuristik ist nicht experimentell validiert."
            },
            "cards": [card_a, card_b],
            "props": [props_a, props_b],
            "groups": [detect_functional_groups(mol_a), detect_functional_groups(mol_b)],
            # Keep the report's zip() aligned without publishing unvalidated heuristic labels.
            "assessments": [None, None]
        }
        pair_quiz = ISOMER_QUIZZES.get(name_a)
        if pair_quiz:
            current_report["quiz"] = pair_quiz
            current_report["pair"]["kernaussage"] = pair_quiz["feedback"]
            show_comparison_quiz(pair_quiz)
        else:
            # Explicit fallback for unforeseen JSON pairs; never silently assume a quiz.
            display(HTML("<p>Für dieses Paar ist noch keine kuratierte Vergleichsfrage hinterlegt.</p>"))

def _glucose_form_card(key):
    """A curated structure card, with explicit C1/O5 focus and lazy 3D."""
    f = GLUCOSE_FORMS[key]
    mol = mol_from_smiles(f["smiles"])
    return _isomer_structure_widget(
        f, mol, highlight_atoms=f["focus"]
    )


def do_glucose_explorer(_=None):
    """Guided open-chain/anomer workflow, independent of the seven isomer pairs."""
    global current_report
    if module_dd.value != "Isomerie":
        return
    start_stage = (current_report.get("glucose_stage", "open")
                   if current_report.get("type") == "glucose_equilibrium"
                   else "open")
    if isomer_pair_dd.value is not None:
        isomer_pair_dd.value = None  # explicitly leave the former curated pair
    cleanup_3d_frames()
    out.clear_output()
    with report_out:
        clear_output(wait=True)
        answer_radio.options = []
        feedback_html.value = ""
        feedback_out.clear_output()
        display(HTML(
            "<h3>Glucose: Ring–Ketten-Gleichgewicht und Mutarotation</h3>"
            "<p><b>Leitfrage:</b> Wie kann aus einer Aldose ein cyclisches "
            "Halbacetal entstehen, und warum gibt es α- und β-Anomere?</p>"
            "<p style='font-size:0.88rem;color:#496275;'>Die drei Formeln "
            "haben dieselbe Summenformel C₆H₁₂O₆; hervorgehoben sind "
            "das Carbonyl-C1 und C5–O bzw. das anomere C1 mit Ring-O. "
            "Die gezeigten stereochemisch definierten Strukturen "
            "sind keine simulierte Reaktionsdynamik.</p>"
        ))
        stage = widgets.ToggleButtons(
            options=[
                ("1 · Offene Kette", "open"),
                ("2 · Kette ↔ α", "alpha"),
                ("3 · Kette ↔ β", "beta"),
                ("4 · α ↔ β", "compare")
            ],
            value=start_stage,
            layout=widgets.Layout(width="100%")
        )
        info = widgets.HTML(layout=widgets.Layout(width="100%", max_width="1040px"))
        panel = widgets.VBox(layout=widgets.Layout(width="100%", max_width="1040px"))
        forms = {k: _glucose_form_card(k) for k in GLUCOSE_FORMS}
        stage_keys = {
            "open": ("open",),
            "alpha": ("open", "alpha"),
            "beta": ("open", "beta"),
            "compare": ("alpha", "beta")
        }
        def update_stage(change=None):
            selected = stage.value
            keys = stage_keys[selected]
            panel.children = (tuple(forms[k] for k in keys)
                              if selected == "open"
                              else (widgets.HBox(
                                  [forms[k] for k in keys],
                                  layout=widgets.Layout(
                                      display="flex", flex_flow="row wrap",
                                      max_width="1040px", gap="10px",
                                      align_items="flex-start")
                              ),))
            if selected == "compare":
                message = (
                    "<b>Anomerie:</b> Die beiden Ringformen haben identische "
                    "Bindungsverknüpfung und dieselbe Konfiguration an C2–C5; "
                    "nur C1 ist verschieden. α und β sind <b>Diastereomere "
                    "(Anomere)</b>, keine Spiegelbildisomere. Der unmittelbare "
                    "Wechsel geschieht in wässriger Lösung über Ringöffnung."
                )
            else:
                message = " ".join(GLUCOSE_FORMS[k]["beschreibung"] for k in keys)
            info.value = (
                "<div style='border-left:3px solid #71b7c4;background:#f1f8fb;"
                "padding:11px 13px;margin:10px 0;line-height:1.5;'>"
                + message + "</div>"
            )
            if current_report.get("type") == "glucose_equilibrium":
                current_report["glucose_stage"] = selected
        stage.observe(update_stage, names="value")
        display(widgets.VBox([stage, info, panel], layout=widgets.Layout(
            width="100%", max_width="1040px", gap="8px"
        )))
        display(HTML(
            "<details style='max-width:1040px;margin-top:10px;'>"
            "<summary><b>Gleichgewicht und Modellgrenzen</b></summary>"
            "<p>Die Ringbildung ist eine intramolekulare Halbacetalbildung "
            "ohne Wasserabspaltung. Über die offene Aldehydform können "
            "α und β ineinander übergehen (Mutarotation). "
            "In Wasser überwiegen die beiden Pyranoseformen; kleine "
            "Mengen Furanosen und sehr wenig offene Kette kommen ebenfalls "
            "vor. Die relativen Anteile hängen unter anderem von den "
            "Bedingungen ab; diese Ansicht berechnet keine Gleichgewichtslage, "
            "Geschwindigkeiten oder optische Drehung.</p>"
            "<p>Strukturquellen: PubChem CID 107526, 79025 und 64689. "
            "Eine Übersicht der Ring-Ketten-Gleichgewichte findet sich "
            "in der NCBI-Publikation zu Glycan Diversity (NBK1955).</p>"
            "</details>"
        ))
        cards = [GLUCOSE_FORMS[k] for k in ("open", "alpha", "beta")]
        current_report = {
            "type": "glucose_equilibrium",
            "title": "D-Glucose: offene Kette, α- und β-D-Glucopyranose",
            "glucose_stage": start_stage,
            "pair": {
                "leitfrage": "Wie entstehen Anomere durch intramolekulare Halbacetalbildung?",
                "mini_aufgaben": [
                    "Identifiziere Carbonyl-C1 und C5–OH in der offenen Kette.",
                    "Finde Ring-O5, C1–OH und das neu entstandene Stereozentrum in der Pyranose.",
                    "Vergleiche α und β: Was unterscheidet sich, was bleibt gleich?",
                    "Erkläre Mutarotation über Ringöffnung und erneuten Ringschluss."
                ],
                "kernaussage": GLUCOSE_QUIZ["feedback"],
                "modellgrenze": (
                    "Die stereochemisch definierten 2D-/3D-Formeln "
                    "zeigen diskrete Strukturmodelle, keine Reaktionsmechanik "
                    "und keine quantitativen Gleichgewichtsanteile. "
                    "Furanoseformen werden hier nicht gesondert visualisiert."
                )
            },
            "cards": cards,
            "props": [calc_properties(mol_from_smiles(c["smiles"])) for c in cards],
            "groups": [detect_functional_groups(mol_from_smiles(c["smiles"]))
                       for c in cards],
            "assessments": [None, None, None],
            "quiz": GLUCOSE_QUIZ
        }
        update_stage()
        show_comparison_quiz(GLUCOSE_QUIZ)


def reset_3d(_=None):
    """Sicherheitsknopf: 3D-iframes entfernen und letzte Analyse neu aufbauen."""
    cleanup_3d_frames()
    feedback_out.clear_output(); report_out.clear_output(); protocol_out.clear_output()
    gc.collect()
    if (module_dd.value == "Isomerie" and
            current_report.get("type") == "glucose_equilibrium"):
        do_glucose_explorer()
    elif module_dd.value == "Isomerie" and not single_isomer_toggle.value:
        if isomer_pair_dd.value:
            do_isomer_comparison()
        else:
            refresh_examples()
    elif module_dd.value == "Polarität und Löslichkeit" and polar_pair_dd.value:
        do_polarity_comparison()
    elif (module_dd.value == "Konformere" and
          current_report.get("cyclohexane_geometries") and
          example_dd.value == "Cyclohexan"):
        analyze()
        do_conformers()
    else:
        analyze()

run_btn.on_click(analyze)
check_btn.on_click(check_answer)
explain_btn.on_click(explain)
report_btn.on_click(show_report)
conf_btn.on_click(do_conformers)
reset3d_btn.on_click(reset_3d)
glucose_btn.on_click(do_glucose_explorer)

# Learner-facing controls: source work in DIDACTIC stays available in every module.
isomer_section = widgets.VBox([
    widgets.HTML("<strong>Isomerenpaar auswählen</strong><div style='font-size:0.86rem;'>"
                 "Die beiden 2D-Strukturen und alle kuratierten Eigenschaften werden "
                 "direkt verglichen; 3D ist auf Wunsch zuschaltbar.</div>"),
    isomer_pair_dd,
    widgets.HTML("<b>Spezialfall: Glucose-Ring–Ketten-Gleichgewicht</b>"
                 "<div style='font-size:0.84rem;'>Offene D-Glucose, α-/β-D-Glucopyranose "
                 "und Mutarotation schrittweise erkunden.</div>"),
    glucose_btn
])
single_section = widgets.VBox([
    widgets.HTML("<b>Einzelmolekül untersuchen</b>"),
    example_dd,
    use_custom,
    custom_smiles,
    widgets.HBox([run_btn, conf_btn])
])
polar_section = widgets.VBox([
    widgets.HTML("<b>Kuratierte Polaritäts-/Löslichkeitsvergleiche</b>"
                 "<div style='font-size:0.85rem;'>Paar auswählen und sofort vergleichen. "
                 "Die Einzelmolekül-Analyse bleibt darunter verfügbar.</div>"),
    polar_pair_dd
])
# Keep individual controls visible for all non-isomer modules.
refresh_examples()
module_dd.observe(refresh_examples, names="value")
single_isomer_toggle.observe(update_single_isomer_controls, names="value")
isomer_pair_dd.observe(refresh_isomer_pair, names="value")
polar_pair_dd.observe(refresh_polar_pair, names="value")
example_dd.observe(reset_pair_on_example_change, names="value")

display(HTML(
    '<h1>MOL_LAB DIDACTIC <span style="font-size:0.55em;color:#666">v0.9d · Entwurf</span></h1>'
    '<p>Wähle ein Lernmodul. Im Modul <b>Isomerie</b> beginnt die Untersuchung direkt '
    'mit einem Molekülpaar. Alle anderen Module behalten die Einzelmolekül-Analyse.</p>'
))
controls = widgets.VBox([
    widgets.HTML("<b>1. Lernmodul wählen</b>"),
    module_dd,
    isomer_section,
    single_isomer_toggle,
    polar_section,
    single_section,
    widgets.HTML("<b>2. 3D-Darstellung (optional)</b>"),
    style_dd,
    widgets.HBox([render_mode_dd, reset3d_btn]),
    widgets.HTML("<b>3. Lernkontrolle und Ergebnissicherung</b>"),
    report_btn,
    widgets.HTML("<small>Modelldeskriptoren sind keine experimentellen Messwerte. "
                 "Eigene SMILES sind auch weiterhin über die Einzelanalyse möglich.</small>")
])
display(controls, out, feedback_out, report_out, protocol_out)
